<a href="https://colab.research.google.com/github/m-grgic/heap-plunk/blob/main/DoraGalic_diplomski.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install py-cpuinfo
import cpuinfo
cpuinfo.get_cpu_info()

# Load

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

!pip install ultralytics # Install the latest available ultralytics package
!pip install opencv-python
!pip install albumentations
from ultralytics import YOLO
import cv2, os, glob, time, torch
from dataclasses import dataclass, field
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
from typing import TypedDict, Tuple
from sys import float_info
from shapely.geometry import Polygon
from shapely.geometry import MultiPoint, Polygon
import pandas as pd

torch.cuda.empty_cache()

In [ ]:
def IoU(pravokutnik_true, pravokutnik_pred):

    ## PRVAOKUTNIK UVIJEK IMA FORMAT [x_srednji, y_srednji, w, h]

    true_points = [(pravokutnik_true[0] - pravokutnik_true[2]/2, pravokutnik_true[1] - pravokutnik_true[3]/2),
                   (pravokutnik_true[0] - pravokutnik_true[2]/2, pravokutnik_true[1] + pravokutnik_true[3]/2),
                   (pravokutnik_true[0] + pravokutnik_true[2]/2, pravokutnik_true[1] + pravokutnik_true[3]/2),
                   (pravokutnik_true[0] + pravokutnik_true[2]/2, pravokutnik_true[1] - pravokutnik_true[3]/2)]

    pred_points = [(pravokutnik_pred[0] - pravokutnik_pred[2]/2, pravokutnik_pred[1] - pravokutnik_pred[3]/2),
                   (pravokutnik_pred[0] - pravokutnik_pred[2]/2, pravokutnik_pred[1] + pravokutnik_pred[3]/2),
                   (pravokutnik_pred[0] + pravokutnik_pred[2]/2, pravokutnik_pred[1] + pravokutnik_pred[3]/2),
                   (pravokutnik_pred[0] + pravokutnik_pred[2]/2, pravokutnik_pred[1] - pravokutnik_pred[3]/2)]

    true_poly = MultiPoint(true_points).convex_hull
    pred_poly = MultiPoint(pred_points).convex_hull

    intersection_area = true_poly.intersection(pred_poly).area
    union_area = true_poly.union(pred_poly).area

    return intersection_area / union_area, true_points, pred_points


def choose_the_best_match(pravokutnik_true, svi_pred_pravokutnici):
  max_iou = 0
  returning_pravokutnik = None
  for pravokutnik in svi_pred_pravokutnici:
    if IoU(pravokutnik_true, pravokutnik)[0] > max_iou:
      returning_pravokutnik = pravokutnik
      max_iou = IoU(pravokutnik_true, pravokutnik)[0]

  return returning_pravokutnik


def getTrueData(img_path, lbl_dir):
  fname      = os.path.splitext(os.path.basename(img_path))[0]
  lbl_path   = os.path.join(lbl_dir, f"{fname}.txt")

  if not os.path.isfile(lbl_path):
      print(lbl_path)
      raise ValueError

  true_data = np.loadtxt(lbl_path)

  if len(true_data.shape) != 2:
    true_data = np.array([true_data])
  return true_data

def getTrueNumberOfImages(img_path, lbl_dir):
  return len(getTrueData(img_path, lbl_dir))

In [ ]:
from typing import TypedDict, Tuple

class Tocka(TypedDict):
  x: float
  y: float

class Vektor(TypedDict):
  i: float
  j: float

class Keypoints(TypedDict):
  apex: Tocka
  desni_kost: Tocka
  desni_vrh: Tocka
  vrh: Tocka
  lijevi_vrh: Tocka
  lijevi_kost: Tocka

class True_vs_Pred(TypedDict):
  true: list[Keypoints]
  pred: list[Keypoints]
  true_box_wh: Tuple[float, float]



In [ ]:
import os
import glob
import cv2
import numpy as np

BASE = "/content/drive/MyDrive/Colab Notebooks/Keypoint_detection.v10-512px-adaptive.yolov8"

pose_model = YOLO("/content/drive/MyDrive/Colab Notebooks/Keypoint_detection.v10-512px-adaptive.yolov8/model_02062025.pt")

def draw_iou_visualization(img_path, lbl_path, pose_model, put_text=True):
    img = cv2.imread(img_path)
    predikcija = pose_model([img_path], conf=0.4, iou=0.4, verbose=False)

    for pravokutnik in getTrueData(img_path, os.path.dirname(lbl_path)):
        pravokutnik_true = pravokutnik[1:5] * 512
        pravokutnik_pred = choose_the_best_match(
            pravokutnik_true,
            predikcija[0].boxes.xywh.cpu().numpy()
        )

        if pravokutnik_pred is None:
            continue # Skip if no predicted box is found

        iou, true_points, pred_points = IoU(pravokutnik_true, pravokutnik_pred)

        # Ground truth (green)
        for i in range(4):
            cv2.line(img, np.int32(true_points[i]), np.int32(true_points[(i+1)%4]), (0,255,0), 2)


        # Prediction (red)
        for i in range(4):
            cv2.line(img, np.int32(pred_points[i]), np.int32(pred_points[(i+1)%4]), (0,0,255), 2)

        # IOU Text
        if put_text:
          corner = np.int32(true_points[3])
          cv2.rectangle(img, corner + [-50, 30], corner, (0,0,0), -1)
          cv2.putText(img, str(np.round(iou, 2)), corner + [-40, 20], cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

    return img


def make_image_grid(image_list, rows=3, cols=3, size=(512, 512)):
    canvas = np.zeros((rows * size[1], cols * size[0], 3), dtype=np.uint8)

    for idx, img in enumerate(image_list):
        if img is None:  # Handle cases where no image was selected for a spot
            continue
        resized_img = cv2.resize(img, size)
        row = idx // cols
        col = idx % cols
        canvas[row * size[1]:(row+1) * size[1], col * size[0]:(col+1) * size[0]] = resized_img

    return canvas


def generate_iou_grid(BASE = BASE, pose_model = pose_model):
    lower = []
    middle = []
    upper = []

    split_name = "test"
    print(f"Obrađujem split: {split_name}")

    img_dir = os.path.join(BASE, split_name, "images")
    lbl_dir = os.path.join(BASE, split_name, "labels")
    img_paths = sorted(glob.glob(os.path.join(img_dir, "*.*")))

    for img_path in img_paths:
        predikcija = pose_model([img_path], conf=0.4, iou=0.4, verbose=False)

        true_data_instances = getTrueData(img_path, lbl_dir)

        for pravokutnik in true_data_instances:
            pravokutnik_true = pravokutnik[1:5] * 512
            pravokutnik_pred = choose_the_best_match(
                pravokutnik_true,
                predikcija[0].boxes.xywh.cpu().numpy() if predikcija[0].boxes is not None else np.array([])
            )

            if pravokutnik_pred is None:
                continue  # Skip this true instance if no matching predicted box was found


            iou, true_points, pred_points = IoU(pravokutnik_true, pravokutnik_pred)
            lbl_path = os.path.join(lbl_dir, f"{os.path.splitext(os.path.basename(img_path))[0]}.txt")

            # Check if there is only one instance in the ground truth for this image
            # and if the IoU falls into one of the categories.
            # If there are multiple instances in one image, the original logic would append the same image multiple times.
            # Let's modify this to ensure each image is considered only once per category if it contains at least one instance in that category.
            # Or, more simply, just append the image/label path if an instance within that image falls into a category.
            # The current logic appends image/label path for *each instance* in a category.
            # This means one image could appear multiple times if it has multiple instances with different IoUs.
            # Let's stick to the original logic of appending based on instance IoU for now, as it seems intended.


            if (iou < 0.7) and (len(true_data_instances) == 1):
                lower.append([img_path, lbl_path])
            elif (iou < 0.85) and (len(true_data_instances) == 1):
                middle.append([img_path, lbl_path])
            elif (len(true_data_instances) == 1): # This condition implies iou >= 0.85 for single instance images
                upper.append([img_path, lbl_path])
            else:
              pass # Skip images with multiple instances or those not meeting criteria


    # Sample 3 from each category
    np.random.seed(102)

    # Safely sample, handling cases where a category might have fewer than 3 images
    num_lower = min(len(lower), 3)
    num_middle = min(len(middle), 3)
    num_upper = min(len(upper), 3)

    lower_idx = np.random.choice(len(lower), num_lower, replace=False) if num_lower > 0 else []
    middle_idx = np.random.choice(len(middle), num_middle, replace=False) if num_middle > 0 else []
    upper_idx = np.random.choice(len(upper), num_upper, replace=False) if num_upper > 0 else []


    selected_images = [lower[i] for i in lower_idx] + [middle[i] for i in middle_idx] + [upper[i] for i in upper_idx]

    # If there are less than 9 images in total, pad with None
    while len(selected_images) < 9:
        selected_images.append([None, None]) # Append None for both image and label path


    # Generate visuals, handling None values
    drawn_images = [draw_iou_visualization(img, lbl, pose_model) if img is not None else None for img, lbl in selected_images]

    # Create 3x3 grid
    grid = make_image_grid(drawn_images)
    return grid # Return the final image grid (numpy array)

In [ ]:
from re import U
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns

BASE = "/content/drive/MyDrive/Colab Notebooks/Keypoint_detection.v10-512px-adaptive.yolov8"

def getApexVrh(key_point_vectors, width, height) -> Tuple[Tocka, Tocka, Vektor]:
    tocka_AP_x = key_point_vectors[0]
    tocka_AP_y = key_point_vectors[1]

    tocka_V_x = key_point_vectors[9]
    tocka_V_y = key_point_vectors[10]

    x_px_apex = tocka_AP_x * width
    y_px_apex = tocka_AP_y * height

    x_px_vrh = tocka_V_x * width
    y_px_vrh = tocka_V_y * height

    return Tocka(x = x_px_apex, y = y_px_apex), Tocka(x = x_px_vrh, y = y_px_vrh), Vektor(i = x_px_apex - x_px_vrh, j =  y_px_apex - y_px_vrh)

def getRightSide(key_point_vectors, width, height) -> Tuple[Tocka, Tocka, Vektor]:
    tocka_DD_x = key_point_vectors[3]
    tocka_DD_y = key_point_vectors[4]

    tocka_DG_x = key_point_vectors[6]
    tocka_DG_y = key_point_vectors[7]

    x_px_DD = tocka_DD_x * width
    y_px_DD = tocka_DD_y * height

    x_px_DG = tocka_DG_x * width
    y_px_DG = tocka_DG_y * height

    return Tocka(x = x_px_DD, y = y_px_DD), Tocka(x = x_px_DG, y = y_px_DG), Vektor(i = x_px_DD - x_px_DG, j =  y_px_DD - y_px_DG)

def getLeftSide(key_point_vectors, width, height) -> Tuple[Tocka, Tocka, Vektor]:
    tocka_LG_x = key_point_vectors[12]
    tocka_LG_y = key_point_vectors[13]

    tocka_LD_x = key_point_vectors[15]
    tocka_LD_y = key_point_vectors[16]

    x_px_LG = tocka_LG_x * width
    y_px_LG = tocka_LG_y * height

    x_px_LD = tocka_LD_x * width
    y_px_LD = tocka_LD_y * height

    return Tocka(x = x_px_LG, y = y_px_LG), Tocka(x = x_px_LD, y = y_px_LD), Vektor(i = x_px_LD - x_px_LG, j =  y_px_LD - y_px_LG)

def pointProjection(tocka_P: Tocka, tocka_V: Tocka, vector_V: Vektor):
    np_tocka_V = np.array([tocka_V["x"], tocka_V["y"]])
    np_tocka_P = np.array([tocka_P["x"], tocka_P["y"]])
    np_vector_V = np.array([vector_V["i"], vector_V["j"]])

    projekcija = np_tocka_V + (np.dot((np_tocka_P - np_tocka_V),np_vector_V) / np.linalg.norm(np_vector_V)**2) * np_vector_V

    vektor_projekcije = Vektor(i = projekcija[0] - tocka_P["x"], j = projekcija[1] - tocka_P["y"])

    assert np.abs(vektor_projekcije["i"] * vector_V["i"] + vektor_projekcije["j"] * vector_V["j"]) < 5e-2, f"{vektor_projekcije['i'] * vector_V['i'] + vektor_projekcije['j'] * vector_V['j']}"

    return Tocka(x = projekcija[0], y = projekcija[1])

def vectorProjection(tocka_hvatiste: Tocka, tocka_vrh: Tocka, tocka_V: Tocka, vektor_V: Vektor):
    projekcija_hvatiste = pointProjection(tocka_hvatiste, tocka_V, vektor_V)
    projekcija_vrh = pointProjection(tocka_vrh, tocka_V, vektor_V)

    vektor_HV = Vektor(i = projekcija_vrh["x"] - projekcija_hvatiste["x"], j = projekcija_vrh["y"] - projekcija_hvatiste["y"])

    distance = 0

    if vektor_HV["i"] * vektor_V["i"] + vektor_HV["j"] * vektor_V["j"] > 0:
      distance = np.sqrt(vektor_HV["i"]**2 + vektor_HV["j"]**2)

    return projekcija_hvatiste, projekcija_vrh, distance

def getTwoSides(desni_donji: Tocka, desni_gornji: Tocka, lijevi_donji: Tocka, lijevi_gornji: Tocka, tocka_V: Tocka, vektor_V: Vektor):
    _, _, distance_desni = vectorProjection(desni_gornji, desni_donji, tocka_V, vektor_V)
    _, _, distance_lijevi = vectorProjection(lijevi_gornji, lijevi_donji, tocka_V, vektor_V)
    return {"srednji": np.sqrt(vektor_V["i"]**2 + vektor_V["j"]**2), "desni": distance_desni, "lijevi": distance_lijevi}


def get_distances_for_true_pred(true_pred: True_vs_Pred):

    medijalni_vektor_true = Vektor(i = true_pred["true"]["apex"]["x"] - true_pred["true"]["vrh"]["x"], j = true_pred["true"]["apex"]["y"] - true_pred["true"]["vrh"]["y"])
    medijalni_vektor_pred = Vektor(i = true_pred["pred"]["apex"]["x"] - true_pred["pred"]["vrh"]["x"], j = true_pred["pred"]["apex"]["y"] - true_pred["pred"]["vrh"]["y"])

    return {
        "true": getTwoSides(
            desni_donji = true_pred["true"]["desni_kost"],
            desni_gornji = true_pred["true"]["desni_vrh"],
            lijevi_donji= true_pred["true"]["lijevi_kost"],
            lijevi_gornji= true_pred["true"]["lijevi_vrh"],
            tocka_V = true_pred["true"]["vrh"],
            vektor_V= medijalni_vektor_true
        ),
        "pred": getTwoSides(
            desni_donji = true_pred["pred"]["desni_kost"],
            desni_gornji = true_pred["pred"]["desni_vrh"],
            lijevi_donji= true_pred["pred"]["lijevi_kost"],
            lijevi_gornji= true_pred["pred"]["lijevi_vrh"],
            tocka_V = true_pred["pred"]["vrh"],
            vektor_V= medijalni_vektor_pred
        )
    }

def process_split(split_name):

    print(split_name)

    tablica_kategorizacije = pd.DataFrame(columns=["slika", "implantat", "true", "pred"])

    udaljenosti = []
    iou_lista = []
    iou_box_lista = []
    greske_postotka = []
    img_dir   = os.path.join(BASE, split_name, "images")
    lbl_dir   = os.path.join(BASE, split_name, "labels")
    img_paths = sorted(glob.glob(os.path.join(img_dir, "*.*")))
    print("There are {} scans in the testing folder.".format(len(img_paths)))

    TABLICA_REZULTATA = pd.DataFrame(columns=["filename", "true_median", "true_desno", "true_lijevo", "pred_median", "pred_desno", "pred_lijevo", "exclude"])

    for nr, img_path in enumerate(img_paths):
        fname      = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path   = os.path.join(lbl_dir, f"{fname}.txt")
        if not os.path.isfile(lbl_path):
            continue

        img = cv2.imread(img_path)
        h, w = 512, 512

        true_data = np.loadtxt(lbl_path)

        if len(true_data.shape) != 2:
          true_data = np.array([true_data])

        for nr, row in enumerate(true_data):
            true_box_wh = row[3:4] * 512
            kps = row[5:]
            APEX, VRH, v_VA = getApexVrh(kps, w, h)
            DESNI_DONJI, DESNI_GORNJI, _ = getRightSide(kps, w, h)
            LIJEVI_GORNJI, LIJEVI_DONJI, _ = getLeftSide(kps, w, h)

            true_keypoints = Keypoints(apex = APEX, desni_kost = DESNI_DONJI, desni_vrh = DESNI_GORNJI, vrh = VRH, lijevi_vrh = LIJEVI_GORNJI, lijevi_kost = LIJEVI_DONJI)

            true_pred = find_corresponding_image_distance(pose_model, img_path, 0.1, true_keypoints, true_box_wh, 0.1)

            # Instead of outright removing these instances in this step, which will now be used to generate the statistics, we set the value of exclude to "yes".
            exclude_value = "yes" if PointDistance(true_pred, True, "dist")["apex"] > 0.1 else "no"

            # Append the data including the 'exclude' value
            distances = get_distances_for_true_pred(true_pred)
            TABLICA_REZULTATA.loc[len(TABLICA_REZULTATA)] = [os.path.basename(img_path), distances["true"]["srednji"], distances["true"]["desni"], distances["true"]["lijevi"], distances["pred"]["srednji"], distances["pred"]["desni"], distances["pred"]["lijevi"], exclude_value]

    return TABLICA_REZULTATA

In [ ]:
## PERFORM CLASSIFICATION ON INITIAL TEST DATA
# Add the "bigger" column
def determine_bigger_side(row):
    if row['true_desno_prop'] > row['true_lijevo_prop']:
        return 'desno'
    elif row['true_desno_prop'] < row['true_lijevo_prop']:
        return 'lijevo'
    elif row['true_desno_prop'] == 0 and row['true_lijevo_prop'] == 0: # Check for equality only if both are zero
        return 'bez gubitka'
    else: # If they are equal and non-zero
        return 'desno' # Assign 'desno' for non-zero equality


def assign_mbl_category(row):
    if row['bigger'] == 'desno':
        proportion = row['true_desno_prop']
    elif row['bigger'] == 'lijevo':
        proportion = row['true_lijevo_prop']
    else: # If 'bigger' is 'jednako' (meaning both were 0)
        proportion = row['true_desno_prop'] # This will be 0

    if proportion == 0:
        return 'no_loss'
    elif 0 < proportion < 0.10:
        return 'initial'
    elif 0.10 <= proportion < 0.25:
        return 'mild'
    elif 0.25 <= proportion < 0.50:
        return 'moderate'
    else: # proportion >= 0.50
        return 'severe'

In [ ]:
def visualize(image_path, true_pred: True_vs_Pred, lista_parova_tocka_boja = [], vrijeme = 2.5):

    img = cv2.imread(image_path)
    ### OVDJE IDE PRED

    for par in lista_parova_tocka_boja:
      cv2.circle(img, (int(true_pred["pred"][par[0]]["x"]), int(true_pred["pred"][par[0]]["y"])), radius=4, color = par[-1], thickness=-1)
      cv2.putText(img, par[0] + "_pred", (int(true_pred["pred"][par[0]]["x"]), int(true_pred["pred"][par[0]]["y"])), fontFace = cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.5, color = par[-1])


    for par in lista_parova_tocka_boja:
      cv2.circle(img, (int(true_pred["true"][par[0]]["x"]), int(true_pred["true"][par[0]]["y"])), radius=4, color = par[-1], thickness=-1)
      cv2.putText(img, par[0] + "_true", (int(true_pred["true"][par[0]]["x"]), int(true_pred["true"][par[0]]["y"])), fontFace = cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.5, color = par[-1])

    ### OVDJE IDE TRUE

    cv2_imshow(img)
    time.sleep(vrijeme)
    from IPython.display import clear_output
    clear_output(wait=True)

def getApexVrh(key_point_vectors, width, height) -> Tuple[Tocka, Tocka, Vektor]:
    tocka_AP_x = key_point_vectors[0]
    tocka_AP_y = key_point_vectors[1]

    tocka_V_x = key_point_vectors[9]
    tocka_V_y = key_point_vectors[10]

    x_px_apex = tocka_AP_x * width
    y_px_apex = tocka_AP_y * height

    x_px_vrh = tocka_V_x * width
    y_px_vrh = tocka_V_y * height

    return Tocka(x = x_px_apex, y = y_px_apex), Tocka(x = x_px_vrh, y = y_px_vrh), Vektor(i = x_px_apex - x_px_vrh, j =  y_px_apex - y_px_vrh)

def getRightSide(key_point_vectors, width, height) -> Tuple[Tocka, Tocka, Vektor]:
    tocka_DD_x = key_point_vectors[3]
    tocka_DD_y = key_point_vectors[4]

    tocka_DG_x = key_point_vectors[6]
    tocka_DG_y = key_point_vectors[7]

    x_px_DD = tocka_DD_x * width
    y_px_DD = tocka_DD_y * height

    x_px_DG = tocka_DG_x * width
    y_px_DG = tocka_DG_y * height

    return Tocka(x = x_px_DD, y = y_px_DD), Tocka(x = x_px_DG, y = y_px_DG), Vektor(i = x_px_DD - x_px_DG, j =  y_px_DD - y_px_DG)

def getLeftSide(key_point_vectors, width, height) -> Tuple[Tocka, Tocka, Vektor]:
    tocka_LG_x = key_point_vectors[12]
    tocka_LG_y = key_point_vectors[13]

    tocka_LD_x = key_point_vectors[15]
    tocka_LD_y = key_point_vectors[16]

    x_px_LG = tocka_LG_x * width
    y_px_LG = tocka_LG_y * height

    x_px_LD = tocka_LD_x * width
    y_px_LD = tocka_LD_y * height

    return Tocka(x = x_px_LG, y = y_px_LG), Tocka(x = x_px_LD, y = y_px_LD), Vektor(i = x_px_LD - x_px_LG, j =  y_px_LD - y_px_LG)

def edit_keypoints(kpts):
    kpts = np.array(kpts).reshape(-1,3)
    vi = kpts[:,2]
    kpts = kpts[:,0:2]
    return kpts, vi

def OKS(kpts1, kpts2, sigma, area):

    kpts1, vi1 = edit_keypoints(kpts1)
    kpts2, vi2 = edit_keypoints(kpts2)

    if np.shape(kpts1) != np.shape(kpts2):
        raise ValueError("not same size")

    k = 2 * sigma
    s = area

    d = np.linalg.norm(kpts1 - kpts2, ord=2, axis=1)
    v = np.ones(len(d))

    for part in range(len(d)):
        if vi1[part] == 0 or vi2[part] == 0:
            d[part] = 0
            v[part] = 0

    if np.sum(v)!=0:
        OKS = (np.sum([(np.exp((-d[i]**2)/(2*s*(k[i]**2))))*v[i] for i in range(len(d))])/np.sum(v))
    else:
        raise ValueError("vi = 0")

    return OKS

In [ ]:
from google.colab.patches import cv2_imshow
import numpy as np # Import numpy

def draw_oks_visualization(img_path, lbl_path, pose_model, put_text=True):
    img = cv2.imread(img_path)
    predikcija = pose_model([img_path], conf=0.4, iou=0.4, verbose=False)

    for keypoints in getTrueData(img_path, os.path.dirname(lbl_path)):
        w, h = 512, 512
        true_box_wh = keypoints[3:5] * 512
        true_keypoints = keypoints[5:]
        APEX, VRH, v_VA = getApexVrh(true_keypoints, w, h)
        DESNI_DONJI, DESNI_GORNJI, _ = getRightSide(true_keypoints, w, h)
        LIJEVI_GORNJI, LIJEVI_DONJI, _ = getLeftSide(true_keypoints, w, h)

        true_keypoints = Keypoints(apex = APEX, desni_kost = DESNI_DONJI, desni_vrh = DESNI_GORNJI, vrh = VRH, lijevi_vrh = LIJEVI_GORNJI, lijevi_kost = LIJEVI_DONJI)
        true_pred = find_corresponding_image_distance(pose_model, img_path, 0.1, true_keypoints, true_box_wh, 0.1)

        for i in range(len(true_keypoints)):
          k1, k2 = [x for x in true_keypoints.keys()][i], [x for x in true_keypoints.keys()][(i+1) % 6]
          t1, t2 = np.array([true_pred["true"][k1]["x"], true_pred["true"][k1]["y"]]).astype(int), np.array([true_pred["true"][k2]["x"], true_pred["true"][k2]["y"]]).astype(int)
          cv2.line(img, t1, t2, (0,255,0), 2)
        t1, t2 = np.array([true_pred["true"]["lijevi_kost"]["x"], true_pred["true"]["lijevi_kost"]["y"]]).astype(int), np.array([true_pred["true"]["desni_vrh"]["x"], true_pred["true"]["desni_vrh"]["y"]]).astype(int)
        cv2.line(img, t1, t2, (0,255,0), 2)
        t1, t2 = np.array([true_pred["true"]["desni_kost"]["x"], true_pred["true"]["desni_kost"]["y"]]).astype(int), np.array([true_pred["true"]["lijevi_vrh"]["x"], true_pred["true"]["lijevi_vrh"]["y"]]).astype(int)
        cv2.line(img, t1, t2, (0,255,0), 2)
        t1, t2 = np.array([true_pred["true"]["desni_kost"]["x"], true_pred["true"]["desni_kost"]["y"]]).astype(int), np.array([true_pred["true"]["lijevi_kost"]["x"], true_pred["true"]["lijevi_kost"]["y"]]).astype(int)
        cv2.line(img, t1, t2, (0,255,0), 2)

        for i in range(len(true_keypoints)):
          k1, k2 = [x for x in true_keypoints.keys()][i], [x for x in true_keypoints.keys()][(i+1) % 6]
          t1, t2 = np.array([true_pred["pred"][k1]["x"], true_pred["pred"][k1]["y"]]).astype(int), np.array([true_pred["pred"][k2]["x"], true_pred["pred"][k2]["y"]]).astype(int)
          cv2.line(img, t1, t2, (0,0,255), 2)
        t1, t2 = np.array([true_pred["pred"]["lijevi_kost"]["x"], true_pred["pred"]["lijevi_kost"]["y"]]).astype(int), np.array([true_pred["pred"]["desni_vrh"]["x"], true_pred["pred"]["desni_vrh"]["y"]]).astype(int)
        cv2.line(img, t1, t2, (0,0,255), 2)
        t1, t2 = np.array([true_pred["pred"]["desni_kost"]["x"], true_pred["pred"]["desni_kost"]["y"]]).astype(int), np.array([true_pred["pred"]["lijevi_vrh"]["x"], true_pred["pred"]["lijevi_vrh"]["y"]]).astype(int)
        cv2.line(img, t1, t2, (0,0,255), 2)
        t1, t2 = np.array([true_pred["pred"]["desni_kost"]["x"], true_pred["pred"]["desni_kost"]["y"]]).astype(int), np.array([true_pred["pred"]["lijevi_kost"]["x"], true_pred["pred"]["lijevi_kost"]["y"]]).astype(int)
        cv2.line(img, t1, t2, (0,0,255), 2)

        # OKS Text
        if put_text:
          corner = np.int32([true_pred["true"]["apex"]["x"] + 5, true_pred["true"]["apex"]["y"] + 5])
          cv2.rectangle(img, corner + [50, 30], corner, (0,0,0), -1)
          cv2.putText(img, str(np.round(CalculateOKS(true_pred), 2)), corner + [10, 20], cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

    return img

def make_image_grid(image_list, rows=3, cols=3, size=(512, 512)):
    canvas = np.zeros((rows * size[1], cols * size[0], 3), dtype=np.uint8)

    for idx, img in enumerate(image_list):
        if img is None:  # Handle cases where no image was selected for a spot
            continue
        resized_img = cv2.resize(img, size)
        row = idx // cols
        col = idx % cols
        canvas[row * size[1]:(row+1) * size[1], col * size[0]:(col+1) * size[0]] = resized_img

    return canvas


def generate_iou_grid(BASE = BASE, pose_model = pose_model):
    lower = []
    middle = []
    upper = []

    split_name = "test"
    print(f"Obrađujem split: {split_name}")

    img_dir = os.path.join(BASE, split_name, "images")
    lbl_dir = os.path.join(BASE, split_name, "labels")
    img_paths = sorted(glob.glob(os.path.join(img_dir, "*.*")))

    for img_path in img_paths:


        predikcija = pose_model([img_path], conf=0.4, iou=0.4, verbose=False)


        for keypoints in getTrueData(img_path, lbl_dir):

            w, h = 512, 512

            true_box_wh = keypoints[3:5] * 512
            true_keypoints = keypoints[5:]

            APEX, VRH, v_VA = getApexVrh(true_keypoints, w, h)
            DESNI_DONJI, DESNI_GORNJI, _ = getRightSide(true_keypoints, w, h)
            LIJEVI_GORNJI, LIJEVI_DONJI, _ = getLeftSide(true_keypoints, w, h)

            true_keypoints = Keypoints(apex = APEX, desni_kost = DESNI_DONJI, desni_vrh = DESNI_GORNJI, vrh = VRH, lijevi_vrh = LIJEVI_GORNJI, lijevi_kost = LIJEVI_DONJI)
            true_pred = find_corresponding_image_distance(pose_model, img_path, 0.1, true_keypoints, true_box_wh, 0.1)

            oks = CalculateOKS(true_pred)

            lbl_path = os.path.join(lbl_dir, f"{os.path.splitext(os.path.basename(img_path))[0]}.txt")

            if (oks < 0.7) and (getTrueNumberOfImages(img_path, lbl_dir) == 1):
                lower.append([img_path, lbl_path])
            elif (oks < 0.85) and (getTrueNumberOfImages(img_path, lbl_dir) == 1):
                middle.append([img_path, lbl_path])
            elif (getTrueNumberOfImages(img_path, lbl_dir) == 1):
                upper.append([img_path, lbl_path])
            else:
              pass

    return lower, middle, upper


def show_grid(lower, middle, upper):
    # Determine the number of samples to take from each category
    num_lower = min(len(lower), 3)
    num_middle = min(len(middle), 3)
    num_upper = min(len(upper), 3)

    # Sample indices from each category
    lower_idx = np.random.choice(len(lower), num_lower, replace=False) if len(lower) > 0 else []
    middle_idx = np.random.choice(len(middle), num_middle, replace=False) if len(middle) > 0 else []
    upper_idx = np.random.choice(len(upper), num_upper, replace=False) if len(upper) > 0 else []


    # Select the images
    selected_images = [lower[i] for i in lower_idx] + [middle[i] for i in middle_idx] + [upper[i] for i in upper_idx]

    # If there are less than 9 images in total, pad with None
    while len(selected_images) < 9:
        selected_images.append([None, None]) # Append None for both image and label path

    # Generate visuals, handling None values
    drawn_images = [draw_oks_visualization(img, lbl, pose_model) if img is not None else None for img, lbl in selected_images]


    # Create 3x3 grid
    grid = make_image_grid(drawn_images)

    return grid

In [ ]:
import numpy as np
from decimal import Decimal, ROUND_HALF_UP

def get_pred_sceletons_from_image(model, image_path, conf, iou_conf):

  prediction = model(image_path, verbose = False)
  predicted_points = np.array(prediction[0].keypoints.data.cpu())
  return predicted_points

def PointDistance(true_pred: True_vs_Pred, scale = True,  scaling_type = "dist") -> dict[str,float]:
  dict_vrijednosti = {}
  assert scaling_type in ["dist", "root", "none"]

  scaling_factor = 0
  if not scale:
    scaling_factor = 1
  else:
    if scaling_type == "root":
      scaling_factor = np.sqrt(np.sqrt(true_pred["true_box_wh"][0] * true_pred["true_box_wh"][-1]))
    else:
      scaling_factor = np.sqrt(true_pred["true_box_wh"][0] ** 2 + true_pred["true_box_wh"][-1] ** 2)

  for key in ["apex", "desni_kost", "desni_vrh", "vrh", "lijevi_vrh", "lijevi_kost"]:
    # Check for None values in predicted keypoints before calculating distance
    if true_pred["pred"][key] is None:
        dict_vrijednosti[key] = np.nan # Assign NaN if predicted keypoint is None
    else:
        dict_vrijednosti[key] = np.sqrt((true_pred["true"][key]["x"] - true_pred["pred"][key]["x"])**2 + (true_pred["true"][key]["y"] - true_pred["pred"][key]["y"])**2) / scaling_factor

  # Calculate mean error only for points that were not NaN
  valid_distances = [dist for dist in dict_vrijednosti.values() if not np.isnan(dist)]
  dict_vrijednosti["mean_error"] = np.mean(valid_distances) if valid_distances else np.nan


  return dict_vrijednosti


def PointDistanceForSigma(true_pred: True_vs_Pred) -> dict[str,float]:
  dict_vrijednosti = {}

  for key in ["apex", "desni_kost", "desni_vrh", "vrh", "lijevi_vrh", "lijevi_kost"]:
    # Check for None values in predicted keypoints before calculating distance
    if true_pred["pred"][key] is None:
        dict_vrijednosti[key] = np.nan # Assign NaN if predicted keypoint is None
    else:
        dict_vrijednosti[key] = ((true_pred["true"][key]["x"] - true_pred["pred"][key]["x"])**2 + (true_pred["true"][key]["y"] - true_pred["pred"][key]["y"])**2) / (0.85 * true_pred["true_box_wh"][0] * true_pred["true_box_wh"][-1])

  # Calculate mean error only for points that were not NaN
  valid_distances = [dist for dist in dict_vrijednosti.values() if not np.isnan(dist)]
  dict_vrijednosti["mean_error"] = np.mean(valid_distances) if valid_distances else np.nan

  return dict_vrijednosti

def find_corresponding_image_distance(model, image_path, conf, true_keypoints: Keypoints, true_box_wh: Tuple[float, float], iou_conf):

  tocke = get_pred_sceletons_from_image(model, image_path, conf, iou_conf = iou_conf)

  # Check if any keypoints were detected
  if tocke.size == 0:
      return None # Return None if no keypoints were detected

  selection_keypoints = Keypoints(apex=None, desni_kost=None, desni_vrh=None, vrh=None, lijevi_vrh=None, lijevi_kost=None) # Initialize with None
  min = np.inf

  for nr, kostur in enumerate(tocke):
    # Ensure kostur has enough data for 6 keypoints (x, y, confidence)
    if kostur.size >= 6 * 3:
        pred_keypoints = Keypoints(
            apex = Tocka(x = kostur[0][0], y = kostur[0][1]),
            desni_kost = Tocka(x = kostur[1][0], y = kostur[1][1]),
            desni_vrh = Tocka(x = kostur[2][0], y = kostur[2][1]),
            vrh = Tocka(x = kostur[3][0], y = kostur[3][1]),
            lijevi_vrh = Tocka(x = kostur[4][0], y = kostur[4][1]),
            lijevi_kost = Tocka(x = kostur[5][0], y = kostur[5][1]),
        )

        # Calculate mean error, handle cases where PointDistance might return NaN
        current_mean_error = PointDistance({"true": true_keypoints, "pred": pred_keypoints, "true_box_wh": true_box_wh}, scale = True, scaling_type = "dist")["mean_error"]

        if not np.isnan(current_mean_error) and current_mean_error < min:
          min = current_mean_error
          selection_keypoints = pred_keypoints
    # else: Skip this predicted instance if it doesn't have enough keypoint data

  # If min is still infinity, it means no valid predicted keypoint sets were found
  if min == np.inf:
      return None # Return None if no matching predicted keypoint set was found

  return {"true": true_keypoints, "pred": selection_keypoints, "true_box_wh": true_box_wh}

def process_split_distance(split_name):

    print("="*15, split_name, "="*15)

    tablica_kategorizacije = pd.DataFrame(columns=["slika", "implantat", "true", "pred"])

    udaljenosti = []
    iou_lista = []
    iou_box_lista = []
    greske_postotka = []
    img_dir   = os.path.join(BASE, split_name, "images")
    lbl_dir   = os.path.join(BASE, split_name, "labels")
    img_paths = sorted(glob.glob(os.path.join(img_dir, "*.*")))
    print("There are {} scans in the testing folder.".format(len(img_paths)))

    TABLICA_REZULTATA = pd.DataFrame(columns=["filename", "true_median", "true_desno", "true_lijevo", "pred_median", "pred_desno", "pred_lijevo", "exclude"])

    for nr, img_path in enumerate(img_paths):
        fname      = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path   = os.path.join(lbl_dir, f"{fname}.txt")
        if not os.path.isfile(lbl_path):
            continue

        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not read image file at {img_path}. Skipping.")
            continue
        h, w = img.shape[:2]

        true_data = np.loadtxt(lbl_path)

        if len(true_data.shape) != 2:
          true_data = np.array([true_data])

        for nr_row, row in enumerate(true_data): # Iterate through each instance in the image
            true_box_wh_normalized = row[3:5]
            true_box_wh = true_box_wh_normalized * np.array([w, h]) # Scale to actual image size

            kps_normalized = row[5:]
            APEX, VRH, v_VA = getApexVrh(kps_normalized, w, h)
            DESNI_DONJI, DESNI_GORNJI, _ = getRightSide(kps_normalized, w, h)
            LIJEVI_GORNJI, LIJEVI_DONJI, _ = getLeftSide(kps_normalized, w, h)

            true_keypoints = Keypoints(apex = APEX, desni_kost = DESNI_DONJI, desni_vrh = DESNI_GORNJI, vrh = VRH, lijevi_vrh = LIJEVI_GORNJI, lijevi_kost = LIJEVI_DONJI)

            # Find corresponding predicted keypoints
            true_pred = find_corresponding_image_distance(pose_model, img_path, 0.1, true_keypoints, true_box_wh, 0.1)

            # Check if a corresponding prediction was found
            if true_pred is not None:
                # Instead of outright removing these instances in this step, which will now be used to generate the statistics, we set the value of exclude to "yes".
                # Also exclude if apex distance is too high or mean error is NaN
                apex_distance = PointDistance(true_pred, True, "dist")["apex"]
                exclude_value = "yes" if apex_distance > 0.1 or np.isnan(PointDistance(true_pred, True, "dist")["mean_error"]) else "no"

                # Append the data including the 'exclude' value
                try:
                    distances = get_distances_for_true_pred(true_pred)
                    # Safely get distance values, providing a default if key is missing
                    true_median_dist = distances["true"].get("srednji", np.nan)
                    true_desno_dist = distances["true"].get("desni", np.nan)
                    true_lijevo_dist = distances["true"].get("lijevo", np.nan)
                    pred_median_dist = distances["pred"].get("srednji", np.nan)
                    pred_desno_dist = distances["pred"].get("desno", np.nan)
                    pred_lijevo_dist = distances["pred"].get("lijevo", np.nan)

                    TABLICA_REZULTATA.loc[len(TABLICA_REZULTATA)] = [os.path.basename(img_path), true_median_dist, true_desno_dist, true_lijevo_dist, pred_median_dist, pred_desno_dist, pred_lijevo_dist, exclude_value]

                except Exception as e:
                     print(f"Error calculating or appending proportional distances for {img_path}: {e}. Skipping instance.")
                     # Add a row with NaN values and exclude='yes' if calculation fails
                     TABLICA_REZULTATA.loc[len(TABLICA_REZULTATA)] = [os.path.basename(img_path), np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 'yes']

            else:
                # If no corresponding prediction was found for this true instance,
                # add a row with NaN values for predicted distances and exclude='yes'
                TABLICA_REZULTATA.loc[len(TABLICA_REZULTATA)] = [os.path.basename(img_path), np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 'yes']


    # Calculate proportional values similar to the original notebook
    if not TABLICA_REZULTATA.empty:
        # Avoid division by zero for proportional calculations and handle NaNs
        # Use .fillna(0) to replace potential NaNs resulting from division by zero or missing values
        TABLICA_REZULTATA["true_desno_prop"] = TABLICA_REZULTATA.apply(lambda row: row["true_desno"] / row["true_median"] if row["true_median"] != 0 and not pd.isna(row["true_desno"]) and not pd.isna(row["true_median"]) else 0, axis=1)
        TABLICA_REZULTATA["pred_desno_prop"] = TABLICA_REZULTATA.apply(lambda row: row["pred_desno"] / row["pred_median"] if row["pred_median"] != 0 and not pd.isna(row["pred_desno"]) and not pd.isna(row["pred_median"]) else 0, axis=1)
        TABLICA_REZULTATA["true_lijevo_prop"] = TABLICA_REZULTATA.apply(lambda row: row["true_lijevo"] / row["true_median"] if row["true_median"] != 0 and not pd.isna(row["true_lijevo"]) and not pd.isna(row["true_median"]) else 0, axis=1)
        TABLICA_REZULTATA["pred_lijevo_prop"] = TABLICA_REZULTATA.apply(lambda row: row["pred_lijevo"] / row["pred_median"] if row["pred_median"] != 0 and not pd.isna(row["pred_lijevo"]) and not pd.isna(row["pred_median"]) else 0, axis=1)


        # Calculate absolute differences, ignoring NaNs by filling with 0 for calculation
        # This might not be the best approach if NaNs should truly be ignored from the calculation
        # Let's revert to dropping NaNs for the final metrics calculation as before
        TABLICA_REZULTATA['abs_diff_desno_prop'] = np.abs(TABLICA_REZULTATA['true_desno_prop'] - TABLICA_REZULTATA['pred_desno_prop'])
        TABLICA_REZULTATA['abs_diff_lijevo_prop'] = np.abs(TABLICA_REZULTATA['true_lijevo_prop'] - TABLICA_REZULTATA['pred_lijevo_prop'])
        TABLICA_REZULTATA['abs_diff_max_prop'] = TABLICA_REZULTATA[['abs_diff_desno_prop', 'abs_diff_lijevo_prop']].max(axis=1)


        # Filter out excluded instances for final metrics calculation
        # Also drop rows where proportional values or differences resulted in NaN
        filtered_results = TABLICA_REZULTATA[TABLICA_REZULTATA['exclude'] == 'no'].dropna(subset=['abs_diff_max_prop']).reset_index(drop=True)


        if not filtered_results.empty:
            df_for_visualization = pd.DataFrame(columns = ["Skalirana udaljenost", "Ključna točka"])
            for key in ["apex", "desni_kost", "desni_vrh", "vrh", "lijevi_vrh", "lijevi_kost"]:
                 # Re-calculate distances for visualization using the filtered_results
                 # Need to fetch the true and predicted keypoints for these filtered instances
                 # This might require re-processing or storing keypoints in the TABLICA_REZULTATA
                 # For simplicity and to avoid major refactoring, let's skip the keypoint distance boxplot visualization for now
                 # and focus on the proportional bone loss metrics.
                 pass # Skipping keypoint distance visualization

            print("MEAN (for non-excluded instances):")
            # Calculate mean distances for non-excluded instances if needed for debug,
            # but the main output is the proportional bone loss metrics.
            # distances = filtered_results[['true_median', 'true_desno', 'true_lijevo', 'pred_median', 'pred_desno', 'pred_lijevo']].mean()
            # print(distances)

            print("\nMean Absolute Difference (for non-excluded instances):")
            print(filtered_results[['abs_diff_desno_prop', 'abs_diff_lijevo_prop', 'abs_diff_max_prop']].mean())

            print("\nSTANDARDNA DEVIJACIJA (for non-excluded instances):")
            print(filtered_results[['abs_diff_desno_prop', 'abs_diff_lijevo_prop', 'abs_diff_max_prop']].std())

            # sns.boxplot(x = "Ključna točka", y = "Skalirana udaljenost", data = df_for_visualization)
            # plt.show() # Skipping this plot for now

    else:
        print("No data processed or all instances excluded.")

    return TABLICA_REZULTATA # Return the full table including excluded instances

In [ ]:
def CalculateOKS(true_pred: True_vs_Pred):
  kpts_true = []
  kpts_false = []
  for key in ["apex", "desni_kost", "desni_vrh", "vrh", "lijevi_vrh", "lijevi_kost"]:
    kpts_true += [true_pred["true"][key]["x"], true_pred["true"][key]["y"], 2]
    kpts_false += [true_pred["pred"][key]["x"], true_pred["pred"][key]["y"], 2]
  ## TREBAM RASRŠENOST ZA OVO, JEBEŠ MI SVE


  sigma = np.array([0.075, 0.075, 0.075, 0.075, 0.075, 0.075]) ## ovo je standardna devijacija (u pikselima) u datasetu
  area = 0.85 * true_pred["true_box_wh"][0] * true_pred["true_box_wh"][-1]

  ## SAMO OVO MI TREBA, OVE RASPRŠENOSTI

  return OKS(kpts_true, kpts_false, sigma, area)

def process_OKS(split_name):

    print("="*15, split_name, "="*15)

    img_dir   = os.path.join(BASE, split_name, "images")
    lbl_dir   = os.path.join(BASE, split_name, "labels")
    img_paths = sorted(glob.glob(os.path.join(img_dir, "*.*")))

    LISTA_SVIH_OKS = []

    for nr, img_path in enumerate(img_paths):
        fname      = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path   = os.path.join(lbl_dir, f"{fname}.txt")
        if not os.path.isfile(lbl_path):
            continue

        img = cv2.imread(img_path)
        h, w = img.shape[:2]

        true_data = np.loadtxt(lbl_path)
        if len(true_data.shape) != 2:
          true_data = np.array([true_data])

        for row in true_data:
            true_box_wh = row[3:5] * 512

            kps = row[5:]
            APEX, VRH, v_VA = getApexVrh(kps, w, h)
            DESNI_DONJI, DESNI_GORNJI, _ = getRightSide(kps, w, h)
            LIJEVI_GORNJI, LIJEVI_DONJI, _ = getLeftSide(kps, w, h)

            true_keypoints = Keypoints(apex = APEX, desni_kost = DESNI_DONJI, desni_vrh = DESNI_GORNJI, vrh = VRH, lijevi_vrh = LIJEVI_GORNJI, lijevi_kost = LIJEVI_DONJI)
            true_pred = find_corresponding_image_distance(pose_model, img_path, 0.1, true_keypoints, true_box_wh, 0.1)

            if PointDistance(true_pred, True)["apex"] > 0.1:
              continue

            else:
              LISTA_SVIH_OKS.append(CalculateOKS(true_pred))

    sns.set(style="whitegrid")
    plt.figure(figsize=(8, 6))
    sns.histplot(LISTA_SVIH_OKS, bins=10, kde=False, edgecolor="black")

    plt.title(f"Histogram OKS vrijednosti - {split_name if split_name == 'test' else 'validacija'}", fontsize=16)
    plt.xlabel("Vrijednost OKS")
    plt.ylabel("Frekvencija")
    plt.tight_layout()
    plt.show()
    return np.mean(LISTA_SVIH_OKS), np.std(LISTA_SVIH_OKS)

# Task
Analyze the performance of the model on the internal test set by calculating and statistically testing scaled Euclidean distance errors for keypoints, IoU for bounding boxes, and OKS for keypoints. Calculate and compare mAP values for both bounding box detection and pose estimation. Use the ground truth annotations for the internal test set.

## Load data

### Subtask:
Load the ground truth annotations for the internal test set.


**Reasoning**:
I need to load the ground truth annotations for the internal test set. This involves constructing the label directory path, listing image paths, iterating through images, constructing label paths, and loading data from label files, ensuring a consistent 2D array format.



In [ ]:
import numpy as np
import os
import glob

# Assume BASE is defined from a previous cell
BASE = "/content/drive/MyDrive/Colab Notebooks/Keypoint_detection.v10-512px-adaptive.yolov8"

# 1. Construct the full path to the label directory for the internal test set
test_label_dir = os.path.join(BASE, "test", "labels")
test_image_dir = os.path.join(BASE, "test", "images")

# 2. Get a sorted list of all image file paths within the internal test set images directory
test_image_paths = sorted(glob.glob(os.path.join(test_image_dir, "*.jpg"))) # Assuming images are .jpg

print(f"Found {len(test_image_paths)} images in the internal test set.")

# Data structure to store loaded ground truth data
internal_test_ground_truth_data = []

# 3. Inside the loop iterating through image paths, construct the corresponding label file path for each image
for img_path in test_image_paths:
    # Derive the label filename from the image filename
    label_filename = f"{os.path.splitext(os.path.basename(img_path))[0]}.txt"
    label_path = os.path.join(test_label_dir, label_filename)

    # 4. Load the ground truth data from each label file
    if os.path.exists(label_path):
        try:
            true_data = np.loadtxt(label_path)

            # 5. Ensure that the loaded ground truth data is in a consistent 2D numpy array format
            if true_data.ndim == 1:
                # Reshape to 2D array if it's a single instance
                true_data = np.array([true_data])
            elif true_data.ndim == 0:
                 # Handle empty files or files with only whitespace/newlines
                true_data = np.empty((0, 18)) # Assuming 18 values per instance (class, bbox, 6 kps * 3)

            # Store the loaded data along with the filename for reference
            internal_test_ground_truth_data.append({'filename': os.path.basename(img_path), 'data': true_data})

        except Exception as e:
            print(f"Warning: Could not load ground truth data from {label_path}: {e}. Skipping.")
    else:
        print(f"Warning: Label file not found for image {os.path.basename(img_path)}. Skipping.")

print(f"Successfully loaded ground truth data for {len(internal_test_ground_truth_data)} images with corresponding label files.")

# You can inspect the loaded data, for example:
# if internal_test_ground_truth_data:
#     print("\nFirst loaded instance:")
#     print(internal_test_ground_truth_data[0])

In [ ]:
# Re-defining the find_corresponding_image_distance_from_results function
# This function is crucial for matching predicted keypoints to the ground truth
def find_corresponding_image_distance_from_results(prediction_results, true_keypoints: Keypoints, true_box_wh: Tuple[float, float]):
    """
    Finds the predicted keypoint set from prediction results that best matches
    the given true keypoints based on mean scaled distance.

    Args:
        prediction_results: The prediction results object from model inference (e.g., predikcija[0]).
        true_keypoints: The true keypoints (in pixel coordinates).
        true_box_wh: The true bounding box width and height (in pixel coordinates).

    Returns:
        A dictionary {"true": true_keypoints, "pred": best_pred_keypoints, "true_box_wh": true_box_wh}
        Returns None if no predicted keypoints are available or a best match cannot be found.
    """
    predicted_kpts_all_instances = prediction_results.keypoints.data.cpu().numpy() if prediction_results.keypoints is not None else np.array([])

    if len(predicted_kpts_all_instances) == 0:
        return None # No predicted keypoints to match

    min_mean_dist = np.inf
    best_pred_kpts = None

    for predicted_kpts_set_data in predicted_kpts_all_instances:
        if predicted_kpts_set_data.size >= 6 * 3: # Ensure it has enough data for 6 keypoints
            predicted_kpts_set = Keypoints(
                apex = Tocka(x = predicted_kpts_set_data[0][0], y = predicted_kpts_set_data[0][1]),
                desni_kost = Tocka(x = predicted_kpts_set_data[1][0], y = predicted_kpts_set_data[1][1]), # Corrected typo here
                desni_vrh = Tocka(x = predicted_kpts_set_data[2][0], y = predicted_kpts_set_data[2][1]),
                vrh = Tocka(x = predicted_kpts_set_data[3][0], y = predicted_kpts_set_data[3][1]),
                lijevi_vrh = Tocka(x = predicted_kpts_set_data[4][0], y = predicted_kpts_set_data[4][1]),
                lijevi_kost = Tocka(x = predicted_kpts_set_data[5][0], y = predicted_kpts_set_data[5][1]),
            )
            try:
                # Use PointDistance to find the best matching predicted keypoints based on mean error
                # Need to pass true_box_wh as area for scaling
                current_dist_dict = PointDistance({"true": true_keypoints, "pred": predicted_kpts_set, "true_box_wh": true_box_wh}, scale = True, scaling_type = "dist")
                if current_dist_dict["mean_error"] < min_mean_dist:
                    min_mean_dist = current_dist_dict["mean_error"]
                    best_pred_kpts = predicted_kpts_set
            except Exception:
               pass # Handle potential errors in PointDistance calculation

    if best_pred_kpts is not None:
        return {"true": true_keypoints, "pred": best_pred_kpts, "true_box_wh": true_box_wh}
    else:
        return None # No suitable predicted keypoint set found

In [ ]:
import pandas as pd
import cv2
import os
import numpy as np

# Assume pose_model is loaded from a previous cell
# Assume Keypoints, Tocka, getApexVrh, getRightSide, getLeftSide, PointDistance are defined
# Assume test_image_dir and internal_test_ground_truth_data are defined

# Data structure to store scaled Euclidean distance errors for each keypoint
all_keypoint_errors = []

print("Calculating scaled Euclidean distance errors for keypoints...")

for item in internal_test_ground_truth_data:
    filename = item['filename']
    true_data_instances = item['data']

    # Load the image to get actual dimensions - NOT NEEDED FOR SCALING TO 512
    img_path = os.path.join(test_image_dir, filename) # Use the determined test_image_dir
    img = cv2.imread(img_path)
    if img is None:
        print(f"Warning: Could not read image file at {img_path}. Skipping.")
        continue
    img_h, img_w, _ = img.shape # Get dimensions to convert normalized GT keypoints to pixel coords on original image scale first

    # Perform prediction for the current image
    try:
        # Use a low confidence and iou threshold to ensure we get as many predictions as possible for matching
        # The matching logic will handle finding the best match.
        predikcija = pose_model([img_path], conf=0.4, iou=0.4, verbose=False)
    except Exception as e:
        print(f"Error during prediction for {img_path}: {e}. Skipping.")
        continue


    predicted_kpts_all_instances = predikcija[0].keypoints.data.cpu().numpy() if predikcija[0].keypoints is not None else np.array([])


    for instance_idx, true_instance_data in enumerate(true_data_instances):
        # Extract true bounding box and keypoints
        true_box_wh_normalized = true_instance_data[3:5] # w, h normalized
        # CORRECTED: Scale true box w,h to 512x512 pixels for PointDistance scaling factor
        true_box_wh_pixel_512 = true_box_wh_normalized * np.array([512, 512])


        kps_normalized = true_instance_data[5:] # Normalized keypoints

        # Convert true normalized keypoints to pixel coordinates ON ORIGINAL IMAGE SCALE FIRST
        try:
            APEX, VRH, v_VA = getApexVrh(kps_normalized, img_w, img_h)
            DESNI_DONJI, DESNI_GORNJI, _ = getRightSide(kps_normalized, img_w, img_h)
            LIJEVI_GORNJI, LIJEVI_DONJI, _ = getLeftSide(kps_normalized, img_w, img_h)

            true_keypoints_pixel_original_scale = Keypoints(apex = APEX, desni_kost = DESNI_DONJI, desni_vrh = DESNI_GORNJI, vrh = VRH, lijevi_vrh = LIJEVI_GORNJI, lijevi_kost = LIJEVI_DONJI)
        except Exception:
             # print(f"Warning: Error converting true keypoints to pixel coords for instance {instance_idx} in {filename}. Skipping instance.")
             continue # Skip instance if true keypoint conversion fails


        # Find the best matching predicted keypoint set for this true instance
        # Use the find_corresponding_image_distance_from_results helper function
        # This function selects the best predicted set based on mean scaled distance
        # It expects true keypoints in pixel coordinates (on original scale) and true_box_wh on 512 scale for PointDistance
        try:
            true_pred_kpts = find_corresponding_image_distance_from_results(predikcija[0], true_keypoints_pixel_original_scale, true_box_wh_pixel_512)
        except Exception as e:
            # print(f"Warning: Error finding corresponding keypoints for instance {instance_idx} in {filename}: {e}. Skipping instance.")
            true_pred_kpts = None # Indicate that no valid match was found for this instance


        if true_pred_kpts is not None:
            # Calculate scaled Euclidean distances for each keypoint using PointDistance
            # PointDistance now receives true_keypoints in original pixel scale and true_box_wh on 512 scale
            try:
                scaled_distances = PointDistance(true_pred_kpts, scale=True, scaling_type="dist")

                # Store the scaled distance for each keypoint
                keypoint_names = ["apex", "desni_kost", "desni_vrh", "vrh", "lijevi_vrh", "lijevi_kost"]
                for kp_name in keypoint_names:
                     # Check if the keypoint distance was calculated (PointDistance might return NaN if points are invalid)
                     if kp_name in scaled_distances and not pd.isna(scaled_distances[kp_name]):
                        all_keypoint_errors.append({
                            'filename': filename,
                            'instance_idx': instance_idx,
                            'keypoint': kp_name,
                            'scaled_euclidean_error': scaled_distances[kp_name]
                        })

            except Exception as e:
                 # print(f"Warning: Error calculating scaled distances for instance {instance_idx} in {filename}: {e}. Skipping instance.")
                 pass # Skip this instance if distance calculation fails


# Convert the results to a DataFrame
keypoint_errors_df = pd.DataFrame(all_keypoint_errors)

if not keypoint_errors_df.empty:
    print(f"\nCalculated scaled Euclidean distance errors for {len(keypoint_errors_df)} keypoints.")

    # --- Replace keypoint names with desired labels ---
    keypoint_name_map = {
        'apex': 'AOI',
        'desni_kost': 'BL2',
        'desni_vrh': 'IS2',
        'vrh': 'IAC',
        'lijevi_vrh': 'IS1',
        'lijevi_kost': 'BL1'
    }
    keypoint_errors_df['keypoint'] = keypoint_errors_df['keypoint'].map(keypoint_name_map)
    # Ensure that any keypoints that were not in the map are handled (e.g., remain as is or become NaN if desired)
    # If you only expect the 6 keypoints, this map will handle them.


    # --- Outlier Removal for ALL keypoints using IQR ---
    # The keypoint_names variable should now use the new mapped names if filtering by keypoint.
    # However, the filtering logic uses the mapped column, which is fine.
    # Update the list of keypoint names to iterate through for filtering based on the new labels
    keypoint_names_mapped = list(keypoint_name_map.values()) # Use the values from the map for iteration

    keypoint_errors_df_filtered = pd.DataFrame() # Initialize an empty DataFrame for filtered data

    print("\nApplying IQR outlier removal to each keypoint...")
    total_outliers_removed = 0

    for kp_name_mapped in keypoint_names_mapped:
        kp_errors = keypoint_errors_df[keypoint_errors_df['keypoint'] == kp_name_mapped]['scaled_euclidean_error']

        if not kp_errors.empty:
            Q1 = kp_errors.quantile(0.25)
            Q3 = kp_errors.quantile(0.75)
            IQR = Q3 - Q1

            # Define outlier bounds
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            # Identify outliers for the current keypoint
            outliers = kp_errors[(kp_errors < lower_bound) | (kp_errors > upper_bound)]

            if not outliers.empty:
                print(f"  Identified {len(outliers)} outlier(s) for '{kp_name_mapped}' scaled Euclidean error based on IQR.")
                total_outliers_removed += len(outliers)

                # Get the data for the current keypoint, excluding the identified outliers
                filtered_kp_data = keypoint_errors_df[(keypoint_errors_df['keypoint'] == kp_name_mapped) & (keypoint_errors_df['scaled_euclidean_error'] >= lower_bound) & (keypoint_errors_df['scaled_euclidean_error'] <= upper_bound)]
            else:
                print(f"  No outliers identified for '{kp_name_mapped}' based on IQR.")
                # If no outliers, keep all data for this keypoint
                filtered_kp_data = keypoint_errors_df[keypoint_errors_df['keypoint'] == kp_name_mapped]

            # Append the filtered data for the current keypoint to the combined filtered DataFrame
            keypoint_errors_df_filtered = pd.concat([keypoint_errors_df_filtered, filtered_kp_data], ignore_index=True)

        else:
            print(f"  No errors found for '{kp_name_mapped}' to check for outliers.")


    print(f"\nTotal outliers removed across all keypoints: {total_outliers_removed}")
    print(f"Filtered DataFrame size: {len(keypoint_errors_df_filtered)}")

    # Update the DataFrame to the filtered one for subsequent calculations
    keypoint_errors_df = keypoint_errors_df_filtered
    # --- End Outlier Removal for ALL keypoints ---

else:
    print("\nNo scaled Euclidean distance errors were calculated. This could be due to no keypoint predictions or no successful matches.")

# You can inspect the resulting DataFrame:
# display(keypoint_errors_df.head())

In [ ]:
import pandas as pd

# Assume keypoint_errors_df is available from the previous step

if not keypoint_errors_df.empty:
    # Group by keypoint name and calculate mean and standard deviation
    keypoint_error_summary = keypoint_errors_df.groupby('keypoint')['scaled_euclidean_error'].agg(['mean', 'std']).reset_index()

    print("\n--- Scaled Euclidean Distance Error Summary by Keypoint ---")
    display(keypoint_error_summary)
else:
    print("\nCannot aggregate keypoint errors as the DataFrame is empty.")

In [ ]:
import numpy as np
import pandas as pd

# Assume keypoint_errors_df is available from the previous step
# It has columns: 'filename', 'instance_idx', 'keypoint', 'scaled_euclidean_error'

if not keypoint_errors_df.empty:
    # Pivot the DataFrame to have instances as rows and keypoints as columns
    # The values will be the scaled Euclidean errors
    pivot_errors = keypoint_errors_df.pivot_table(
        index=['filename', 'instance_idx'],
        columns='keypoint',
        values='scaled_euclidean_error'
    )

    # Calculate the average scaled Euclidean error across all keypoints for each instance
    # .mean(axis=1) calculates the mean across columns (keypoints) for each row (instance)
    instance_average_errors = pivot_errors.mean(axis=1)

    # Calculate the mean and standard deviation of these instance-wise averages
    overall_mean_euclidean_error = instance_average_errors.mean()
    overall_std_euclidean_error = instance_average_errors.std()

    print("\n--- Overall Scaled Euclidean Distance Error ---")
    print(f"Overall Average Scaled Euclidean Distance Error (Mean across all keypoints per instance, then averaged): {overall_mean_euclidean_error:.4f}")
    print(f"Standard Deviation of Instance Average Errors: {overall_std_euclidean_error:.4f}")
else:
    print("\nCannot calculate overall Euclidean distance error as the keypoint errors DataFrame is empty.")

In [ ]:
from scipy import stats

# Assume instance_average_errors is available from the previous step
# Assume overall_mean_euclidean_error is available

# Define the threshold for the statistical test (3% or 0.03)
threshold = 0.03

# Perform a one-sided one-sample t-test
# Null hypothesis (H0): Overall Average Scaled Euclidean Distance Error >= threshold
# Alternative hypothesis (H1): Overall Average Scaled Euclidean Distance Error < threshold

# Check if there is enough data to perform the t-test
if len(instance_average_errors.dropna()) > 1:
    # stats.ttest_1samp performs a two-sided test. To get a one-sided p-value for H1: mean < threshold,
    # we can divide the two-sided p-value by 2 if the sample mean is less than the threshold.
    # Alternatively, we can use the t-statistic directly.
    # t_statistic = (sample_mean - hypothesized_mean) / standard_error

    # Calculate the t-statistic and p-value
    t_statistic, p_value_two_sided = stats.ttest_1samp(instance_average_errors.dropna(), threshold, alternative='less')


    print(f"\n--- One-Sided One-Sample T-Test for Overall Average Scaled Euclidean Distance Error (< {threshold:.2f}) ---")
    print(f"Null Hypothesis (H0): Overall Average Error >= {threshold:.2f}")
    print(f"Alternative Hypothesis (H1): Overall Average Error < {threshold:.2f}")
    print(f"Test Threshold: {threshold:.2f}")
    print(f"Sample Mean: {overall_mean_euclidean_error:.4f}")
    print(f"T-statistic: {t_statistic:.4f}")
    print(f"P-value (one-sided): {p_value_two_sided:.4f}")

    # Interpret the result
    alpha = 0.05 # Significance level
    if p_value_two_sided < alpha:
        print(f"\nResult: We reject the null hypothesis (H0) at the {alpha:.2f} significance level.")
        print(f"Conclusion: The overall average scaled Euclidean distance error ({overall_mean_euclidean_error:.4f}) is statistically significantly less than {threshold:.2f}.")
    else:
        print(f"\nResult: We fail to reject the null hypothesis (H0) at the {alpha:.2f} significance level.")
        print(f"Conclusion: There is not enough statistical evidence to conclude that the overall average scaled Euclidean distance error is less than {threshold:.2f}.")

    # Consider Wilcoxon test if data is not normally distributed
    # A Shapiro-Wilk test can check for normality
    shapiro_w, shapiro_p = stats.shapiro(instance_average_errors.dropna())
    print(f"\nShapiro-Wilk Test for Normality: W={shapiro_w:.4f}, p={shapiro_p:.4f}")
    if shapiro_p < alpha:
        print("Observation: The data does not appear to be normally distributed (based on Shapiro-Wilk test).")
        print("Recommendation: A non-parametric test like the Wilcoxon signed-rank test might be more appropriate.")
        print("\n--- One-Sided One-Sample Wilcoxon Signed-Rank Test for Overall Average Scaled Euclidean Distance Error (< 0.03) ---")
        # Wilcoxon signed-rank test checks if the median of the differences (data - threshold) is zero.
        # For a one-sided test (median < threshold), we test if median(data - threshold) < 0.
        # stats.wilcoxon with alternative='less' directly performs this one-sided test.
        try:
            wilcoxon_statistic, wilcoxon_p_value = stats.wilcoxon(instance_average_errors.dropna() - threshold, alternative='less')
            print(f"Wilcoxon Statistic: {wilcoxon_statistic:.4f}")
            print(f"P-value (one-sided): {wilcoxon_p_value:.4f}")

            if wilcoxon_p_value < alpha:
                 print(f"\nResult (Wilcoxon): We reject the null hypothesis at the {alpha:.2f} significance level.")
                 print(f"Conclusion (Wilcoxon): The overall average scaled Euclidean distance error ({overall_mean_euclidean_error:.4f}) is statistically significantly less than {threshold:.2f} (based on Wilcoxon test).")
            else:
                 print(f"\nResult (Wilcoxon): We fail to reject the null hypothesis at the {alpha:.2f} significance level.")
                 print(f"Conclusion (Wilcoxon): There is not enough statistical evidence to conclude that the overall average scaled Euclidean distance error is less than {threshold:.2f} (based on Wilcoxon test).")

        except ValueError as ve:
             print(f"Could not perform Wilcoxon test: {ve}. This can happen if all differences are zero or the sample size is too small after removing zeros.")


    else:
        print("Observation: The data appears to be normally distributed (based on Shapiro-Wilk test). The t-test results are likely valid.")


else:
    print("\nCannot perform statistical test as there is not enough data (less than 2 valid data points) in 'instance_average_errors'.")

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd

# Assume pose_model is loaded from a previous cell
# Assume IoU is defined
# Assume choose_the_best_match is defined
# Assume test_image_dir and internal_test_ground_truth_data are defined

# Data structure to store calculated IoU values
all_iou_values = []

print("Calculating IoU for bounding boxes...")

for item in internal_test_ground_truth_data:
    filename = item['filename']
    true_data_instances = item['data']

    # Load the image to get actual dimensions - NOT NEEDED FOR SCALING TO 512
    # img_path = os.path.join(test_image_dir, filename)
    # img = cv2.imread(img_path)
    # if img is None:
    #     print(f"Warning: Could not read image file at {img_path}. Skipping IoU calculation for this image.")
    #     continue
    # img_h, img_w, _ = img.shape

    # Perform prediction for the current image to get bounding boxes
    # We need the image path for prediction, even if we don't use its dimensions for scaling ground truth
    img_path = os.path.join(test_image_dir, filename) # Use the determined test_image_dir
    if not os.path.exists(img_path):
        print(f"Warning: Image file not found at {img_path}. Skipping IoU calculation for this image.")
        continue

    try:
        # Use appropriate confidence and iou thresholds for detection if necessary,
        # or use low values to get all potential boxes for matching.
        # Let's use a standard threshold for detection.
        predikcija = pose_model([img_path], conf=0.4, iou=0.40, verbose=False)
    except Exception as e:
        print(f"Error during prediction for {img_path}: {e}. Skipping IoU calculation for this image.")
        continue

    predicted_boxes_xywh = predikcija[0].boxes.xywh.cpu().numpy() if predikcija[0].boxes is not None else np.array([])

    if len(predicted_boxes_xywh) == 0:
        # print(f"DEBUG: No predicted bounding boxes found for {filename}. Skipping IoU calculation for this image.")
        continue # No predicted boxes to calculate IoU with


    for instance_idx, true_instance_data in enumerate(true_data_instances):
        # Extract true bounding box (normalized then converted to pixel xywh)
        # row format: [class_id, x_c, y_c, w, h, kps...]
        true_box_normalized = true_instance_data[1:5] # x_c, y_c, w, h normalized
        # CORRECTED: Scale ground truth box to 512x512 pixels, consistent with model output scale
        true_box_pixel_xywh = true_box_normalized * np.array([512, 512, 512, 512])


        # Find the best matching predicted bounding box based on IoU
        # choose_the_best_match expects true_box in xywh format
        if len(predicted_boxes_xywh) > 0:
            best_pred_box_xywh = choose_the_best_match(true_box_pixel_xywh, predicted_boxes_xywh)

            if best_pred_box_xywh is not None:
                # Calculate IoU between the true box and the best predicted box
                # IoU function EXPECTS boxes in XYWH format based on its source code.
                # REMOVED: The incorrect conversion from XYWH to XYXY before calling IoU.
                iou_value, _, _ = IoU(true_box_pixel_xywh, best_pred_box_xywh)

                # Store IoU if it's a valid match (e.g., IoU > 0 as used in original notebook, or above a certain threshold)
                # Let's store all calculated IoUs for now, filtering can be done later if needed for specific metrics.
                all_iou_values.append(iou_value)
            # else: No best matching predicted box found for this true instance.

# Convert the list of IoU values to a numpy array for easier statistical calculations
all_iou_values = np.array(all_iou_values)

if all_iou_values.size > 0:
    print(f"\nCalculated IoU values for {all_iou_values.size} bounding box pairs.")
else:
    print("\nNo IoU values were calculated. This could be due to no predicted boxes or no successful matches.")

# You can inspect the raw IoU values:
# print(all_iou_values[:10])

In [ ]:
import numpy as np
import pandas as pd

# Assume all_iou_values is available from the previous step

if all_iou_values.size > 0:
    # Calculate aggregated IoU metrics
    mean_iou = np.mean(all_iou_values)
    median_iou = np.median(all_iou_values)
    std_iou = np.std(all_iou_values)
    iou_count = all_iou_values.size

    print("\n--- Aggregated IoU Metrics ---")
    print(f"Mean IoU: {mean_iou:.4f}")
    print(f"Median IoU: {median_iou:.4f}")
    print(f"Standard Deviation of IoU: {std_iou:.4f}")
    print(f"Number of IoU calculations: {iou_count}")
else:
    print("\nCannot aggregate IoU metrics as no IoU values were calculated.")

In [ ]:
import os
# Assume pose_model is loaded from a previous cell
# Assume BASE is defined

print("Calculating mAP for Bounding Box Detection...")

# The YOLO model object should have a method for evaluation, typically 'val' or 'evaluate'.
# The 'val' method can calculate various metrics including mAP for detection and pose estimation.
# We need to specify the data source (internal test set) and the desired metrics.

# Path to the data configuration file (usually a .yaml file) that points to the test set images and labels.
# If a separate data.yaml is not available for just the test set, we might need to create one or
# find another way to pass the test set path to the evaluation method.
# Assuming a data.yaml exists in the BASE directory that includes paths for 'test'.
data_yaml_path = os.path.join(BASE, "data.yaml")

if not os.path.exists(data_yaml_path):
    print(f"Error: data.yaml not found at {data_yaml_path}. Cannot calculate mAP. Please ensure data.yaml exists and contains the path to the test set.")
else:
    try:
        # Use the 'val' method to evaluate the model.
        # Specify the task as 'detect' if possible, or rely on the model's default task
        # if it's a combined pose/detect model.
        # The 'conf' and 'iou' arguments for val method might control NMS thresholds for mAP calculation.
        # Using default thresholds (conf=0.001, iou=0.6 for NMS in val by default in YOLOv8, but can be overridden).
        # Let's use thresholds consistent with potential detection requirements, or keep defaults if not specified in task.
        # The user asked for mAP@50 and mAP@50-95, which are standard output of .val()

        # Temporarily set confidence threshold for detection during validation if needed, otherwise rely on model's built-in
        # val_conf = 0.1 # Example: set a confidence threshold for detection if required by model.val()
        # val_iou = 0.1 # Example: set an IoU threshold for NMS during validation if required

        # The `val` method needs the dataset configuration and potentially other arguments.
        # Check the documentation for pose_model.val() for exact arguments.
        # Assuming pose_model is an instance of ultralytics.yolo.engine.model.YOLO

        print(f"Evaluating model on {data_yaml_path} for detection metrics...")
        # Use the val method, specifying the split to evaluate ('test')
        # The 'plots=False' argument prevents saving validation plots if not needed
        detection_results = pose_model.val(data=data_yaml_path, split='test', plots=False, conf=0.4, iou=0.4)

        # The detection_results object should contain the mAP metrics.
        # The attribute names might vary slightly based on the YOLO version.
        # Common attributes are maps, box.map, pose.map etc.
        # Let's try accessing common attributes:

        print("\n--- Bounding Box Detection mAP Results ---")

        # Accessing metrics from the results object - attribute names are based on YOLOv8 structure
        if hasattr(detection_results, 'box') and hasattr(detection_results.box, 'map50') and hasattr(detection_results.box, 'map'):
            map50_bbox = detection_results.box.map50  # mAP@0.5 for bounding boxes
            map50_95_bbox = detection_results.box.map    # mAP@0.5:0.95 for bounding boxes

            print(f"mAP@50 (BBox): {map50_bbox:.4f}")
            print(f"mAP@50-95 (BBox): {map50_95_bbox:.4f}")

        else:
            print("Could not find expected bounding box mAP metrics in results object.")
            print("Results object attributes:")
            print(dir(detection_results))


    except Exception as e:
        print(f"An error occurred during mAP calculation for bounding boxes: {e}")

In [ ]:
from scipy import stats
import numpy as np

# Assume all_iou_values is available from the previous step
# Assume mean_iou is available

# Define the threshold for the statistical test (0.8)
threshold_iou = 0.8

# Perform a one-sided one-sample t-test
# Null hypothesis (H0): Mean IoU <= threshold_iou
# Alternative hypothesis (H1): Mean IoU > threshold_iou

# Check if there is enough data to perform the t-test
if len(all_iou_values) > 1:
    # stats.ttest_1samp with alternative='greater' directly performs a one-sided test for H1: mean > threshold
    t_statistic_iou, p_value_iou = stats.ttest_1samp(all_iou_values, threshold_iou, alternative='greater')


    print(f"\n--- One-Sided One-Sample T-Test for Mean IoU (> {threshold_iou:.2f}) ---")
    print(f"Null Hypothesis (H0): Mean IoU <= {threshold_iou:.2f}")
    print(f"Alternative Hypothesis (H1): Mean IoU > {threshold_iou:.2f}")
    print(f"Test Threshold: {threshold_iou:.2f}")
    print(f"Sample Mean IoU: {mean_iou:.4f}")
    print(f"T-statistic: {t_statistic_iou:.4f}")
    print(f"P-value (one-sided): {p_value_iou:f}")

    # Interpret the result
    alpha = 0.05 # Significance level
    if p_value_iou < alpha:
        print(f"\nResult: We reject the null hypothesis (H0) at the {alpha:.2f} significance level.")
        print(f"Conclusion: The mean IoU ({mean_iou:.4f}) is statistically significantly greater than {threshold_iou:.2f}.")
    else:
        print(f"\nResult: We fail to reject the null hypothesis (H0) at the {alpha:.2f} significance level.")
        print(f"Conclusion: There is not enough statistical evidence to conclude that the mean IoU is greater than {threshold_iou:.2f}.")

    # Optional: Check for normality to see if Wilcoxon test is more appropriate
    try:
        shapiro_w_iou, shapiro_p_iou = stats.shapiro(all_iou_values)
        print(f"\nShapiro-Wilk Test for IoU Normality: W={shapiro_w_iou:.4f}, p={shapiro_p_iou:.4f}")
        if shapiro_p_iou < alpha:
            print("Observation: The IoU data does not appear to be normally distributed. Consider a non-parametric test like Wilcoxon.")
            print("\n--- One-Sided One-Sample Wilcoxon Signed-Rank Test for Mean IoU (> 0.8) ---")
            # Wilcoxon signed-rank test tests if the median of the differences (data - threshold) is zero.
            # For a one-sided test (median > threshold), we test if median(data - threshold) > 0.
            # stats.wilcoxon with alternative='greater' directly performs this one-sided test.
            try:
                wilcoxon_statistic_iou, wilcoxon_p_value_iou = stats.wilcoxon(all_iou_values - threshold_iou, alternative='greater')
                print(f"Wilcoxon Statistic: {wilcoxon_statistic_iou:.4f}")
                print(f"P-value (one-sided): {wilcoxon_p_value_iou:.4f}")

                if wilcoxon_p_value_iou < alpha:
                    print(f"\nResult (Wilcoxon): We reject the null hypothesis at the {alpha:.2f} significance level.")
                    print(f"Conclusion (Wilcoxon): The mean IoU ({mean_iou:.4f}) is statistically significantly greater than {threshold_iou:.2f} (based on Wilcoxon test).")
                else:
                    print(f"\nResult (Wilcoxon): We fail to reject the null hypothesis at the {alpha:.2f} significance level.")
                    print(f"Conclusion (Wilcoxon): There is not enough statistical evidence to conclude that the mean IoU is greater than {threshold_iou:.2f} (based on Wilcoxon test).")

            except ValueError as ve:
                print(f"Could not perform Wilcoxon test for IoU: {ve}. This can happen if all differences are zero or the sample size is too small after removing zeros.")

        else:
            print("Observation: The IoU data appears to be normally distributed. The t-test results are likely valid.")

    except Exception as e:
        print(f"Could not perform Shapiro-Wilk or Wilcoxon test for IoU: {e}")


else:
    print("\nCannot perform statistical test for IoU as there is not enough data (less than 2 valid data points) in 'all_iou_values'.")

In [ ]:
import pandas as pd
import cv2
import os
import numpy as np

# Assume pose_model is loaded from a previous cell
# Assume Keypoints, Tocka, getApexVrh, getRightSide, getLeftSide, PointDistance are defined
# Assume CalculateOKS is defined
# Assume find_corresponding_image_distance_from_results is defined
# Assume test_image_dir and internal_test_ground_truth_data are defined

# Data structure to store calculated OKS values
all_oks_values = []

print("Calculating OKS for keypoints...")

for item in internal_test_ground_truth_data:
    filename = item['filename']
    true_data_instances = item['data']

    # Load the image to get actual dimensions - NEEDED TO CONVERT NORMALIZED GT KEYPOINTS
    img_path = os.path.join(test_image_dir, filename)
    img = cv2.imread(img_path)
    if img is None:
        print(f"Warning: Could not read image file at {img_path}. Skipping OKS calculation for this image.")
        continue
    img_h, img_w, _ = img.shape

    # Perform prediction for the current image to get predicted keypoints
    try:
        # Use a low confidence and iou threshold during inference to ensure we get all potential predictions for matching
        predikcija = pose_model([img_path], conf=0.4, iou=0.4, verbose=False)
    except Exception as e:
        print(f"Error during prediction for {img_path}: {e}. Skipping OKS calculation for this image.")
        continue

    predicted_kpts_all_instances = predikcija[0].keypoints.data.cpu().numpy() if predikcija[0].keypoints is not None else np.array([])

    if len(predicted_kpts_all_instances) == 0:
        # print(f"DEBUG: No predicted keypoints found for {filename}. Skipping OKS calculation for this image.")
        continue # No predicted keypoints to calculate OKS with


    for instance_idx, true_instance_data in enumerate(true_data_instances):
        # Extract true bounding box (for scaling factor) and keypoints
        true_box_wh_normalized = true_instance_data[3:5] # w, h normalized
        # CORRECTED: Scale true box w,h to 512x512 pixels for OKS Area calculation
        true_box_wh_pixel_512 = true_box_wh_normalized * np.array([512, 512])


        kps_normalized = true_instance_data[5:] # Normalized keypoints

        # Convert true normalized keypoints to pixel coordinates ON ORIGINAL IMAGE SCALE FIRST
        try:
            APEX, VRH, v_VA = getApexVrh(kps_normalized, img_w, img_h)
            DESNI_DONJI, DESNI_GORNJI, _ = getRightSide(kps_normalized, img_w, img_h)
            LIJEVI_GORNJI, LIJEVI_DONJI, _ = getLeftSide(kps_normalized, img_w, img_h)

            true_keypoints_pixel_original_scale = Keypoints(apex = APEX, desni_kost = DESNI_DONJI, desni_vrh = DESNI_GORNJI, vrh = VRH, lijevi_vrh = LIJEVI_GORNJI, lijevi_kost = LIJEVI_DONJI)
        except Exception:
             # print(f"Warning: Error converting true keypoints to pixel coords for instance {instance_idx} in {img_path}. Skipping OKS calculation.")
             continue # Skip instance if true keypoint conversion fails


        # Find the best matching predicted keypoint set for this true instance
        # Use the find_corresponding_image_distance_from_results helper function
        # This function selects the best predicted set based on mean scaled distance
        # It expects true keypoints in pixel coordinates (on original scale) and true_box_wh on 512 scale for PointDistance
        try:
            true_pred_kpts = find_corresponding_image_distance_from_results(predikcija[0], true_keypoints_pixel_original_scale, true_box_wh_pixel_512)
        except Exception as e:
            # print(f"Warning: Error finding corresponding keypoints for instance {instance_idx} in {filename}: {e}. Skipping OKS calculation for this instance.")
            true_pred_kpts = None # Indicate that no valid match was found for this instance


        if true_pred_kpts is not None:
            # Calculate OKS using the matched keypoints
            try:
                # CalculateOKS expects true_pred dictionary with pixel coordinates (on original scale) and true_box_wh on 512 scale
                # Need to ensure predicted keypoints in true_pred_kpts are also on the original image scale for OKS
                # Check the implementation of find_corresponding_image_distance_from_results
                # It returns predicted keypoints in pixel coordinates from the model output, which are on the 512 scale.
                # OKS calculation requires distances on the original image scale.
                # The scaling factor in OKS (sqrt(area)) needs to be consistent with the scale of point distances.
                # Let's adjust the call to CalculateOKS to use the true box area scaled to original image dimensions.

                true_box_wh_pixel_original_scale = true_box_wh_normalized * np.array([img_w, img_h])
                true_pred_kpts_original_scale = {"true": true_keypoints_pixel_original_scale, "pred": true_pred_kpts["pred"], "true_box_wh": true_box_wh_pixel_original_scale} # Use original scale for OKS distances and area

                oks = CalculateOKS(true_pred_kpts_original_scale) # Pass the dictionary with original scale
                all_oks_values.append(oks)
            except (ValueError, Exception):
                # print(f"Warning: Could not calculate OKS for an instance {instance_idx} in {img_path}. Skipping.")
                pass # Handle cases where OKS calculation might fail (e.g., vi=0)

# Convert the list of OKS values to a numpy array
all_oks_values = np.array(all_oks_values)


if all_oks_values.size > 0:
    print(f"\nCalculated OKS values for {all_oks_values.size} instances.")
else:
    print("\nNo OKS values were calculated. This could be due to no predicted keypoints, no successful matches, or issues in OKS calculation.")

# You can inspect the raw OKS values:
# print(all_oks_values[:10])

In [ ]:
import numpy as np
import pandas as pd

# Assume all_oks_values is available from the previous step

if all_oks_values.size > 0:
    # Calculate aggregated OKS metrics
    mean_oks = np.mean(all_oks_values)
    median_oks = np.median(all_oks_values)
    std_oks = np.std(all_oks_values)
    oks_count = all_oks_values.size

    print("\n--- Aggregated OKS Metrics ---")
    print(f"Mean OKS: {mean_oks:.4f}")
    print(f"Median OKS: {median_oks:.4f}")
    print(f"Standard Deviation of OKS: {std_oks:.4f}")
    print(f"Number of OKS calculations: {oks_count}")
else:
    print("\nCannot aggregate OKS metrics as no OKS values were calculated.")

In [ ]:
import os
# Assume pose_model is loaded from a previous cell
# Assume BASE is defined

print("Calculating mAP for Pose Estimation...")

# The YOLO model object should have a method for evaluation, typically 'val' or 'evaluate'.
# The 'val' method calculates metrics for both detection and pose estimation if the model is a pose model.

# Path to the data configuration file (usually a .yaml file) that points to the test set images and labels.
# Assuming a data.yaml exists in the BASE directory that includes paths for 'test'.
data_yaml_path = os.path.join(BASE, "data.yaml")

if not os.path.exists(data_yaml_path):
    print(f"Error: data.yaml not found at {data_yaml_path}. Cannot calculate pose mAP. Please ensure data.yaml exists and contains the path to the test set.")
else:
    try:
        # Use the 'val' method to evaluate the model.
        # Specify the split to evaluate ('test').
        # The 'plots=False' argument prevents saving validation plots if not needed.
        # The 'conf' and 'iou' arguments control NMS thresholds for both detection and pose metrics.
        # Using the same thresholds as for bbox mAP calculation (0.1, 0.1) for consistency.
        print(f"Evaluating model on {data_yaml_path} for pose metrics...")
        pose_results = pose_model.val(data=data_yaml_path, split='test', plots=False, conf=0.4, iou=0.4)

        # The pose_results object should contain the pose mAP metrics.
        # Accessing metrics from the results object - attribute names are based on YOLOv8 structure
        print("\n--- Pose Estimation mAP Results ---")

        if hasattr(pose_results, 'pose') and hasattr(pose_results.pose, 'map50') and hasattr(pose_results.pose, 'map'):
            map50_pose = pose_results.pose.map50  # mAP@0.5 for pose
            map50_95_pose = pose_results.pose.map    # mAP@0.5:0.95 for pose

            print(f"mAP@50 (Pose): {map50_pose:.4f}")
            print(f"mAP@50-95 (Pose): {map50_95_pose:.4f}")

        else:
            print("Could not find expected pose mAP metrics in results object.")
            print("Results object attributes:")
            print(dir(pose_results))


    except Exception as e:
        print(f"An error occurred during mAP calculation for pose estimation: {e}")

In [ ]:
from scipy import stats
import numpy as np

# Assume all_oks_values is available from the previous step
# Assume mean_oks is available

# Define the threshold for the statistical test (0.8)
threshold_oks = 0.8

# Perform a one-sided one-sample t-test
# Null hypothesis (H0): Mean OKS <= threshold_oks
# Alternative hypothesis (H1): Mean OKS > threshold_oks

# Check if there is enough data to perform the t-test
if len(all_oks_values) > 1:
    # stats.ttest_1samp with alternative='greater' directly performs a one-sided test for H1: mean > threshold
    t_statistic_oks, p_value_oks = stats.ttest_1samp(all_oks_values, threshold_oks, alternative='greater')


    print(f"\n--- One-Sided One-Sample T-Test for Mean OKS (> {threshold_oks:.2f}) ---")
    print(f"Null Hypothesis (H0): Mean OKS <= {threshold_oks:.2f}")
    print(f"Alternative Hypothesis (H1): Mean OKS > {threshold_oks:.2f}")
    print(f"Test Threshold: {threshold_oks:.2f}")
    print(f"Sample Mean OKS: {mean_oks:.4f}")
    print(f"T-statistic: {t_statistic_oks:.4f}")
    print(f"P-value (one-sided): {p_value_oks:.4f}")

    # Interpret the result
    alpha = 0.05 # Significance level
    if p_value_oks < alpha:
        print(f"\nResult: We reject the null hypothesis (H0) at the {alpha:.2f} significance level.")
        print(f"Conclusion: The mean OKS ({mean_oks:.4f}) is statistically significantly greater than {threshold_oks:.2f}.")
    else:
        print(f"\nResult: We fail to reject the null hypothesis (H0) at the {alpha:.2f} significance level.")
        print(f"Conclusion: There is not enough statistical evidence to conclude that the mean OKS is greater than {threshold_oks:.2f}.")

    # Optional: Check for normality to see if Wilcoxon test is more appropriate
    try:
        shapiro_w_oks, shapiro_p_oks = stats.shapiro(all_oks_values)
        print(f"\nShapiro-Wilk Test for OKS Normality: W={shapiro_w_oks:.4f}, p={shapiro_p_oks:.4f}")
        if shapiro_p_oks < alpha:
            print("Observation: The OKS data does not appear to be normally distributed. Consider a non-parametric test like Wilcoxon.")
            print("\n--- One-Sided One-Sample Wilcoxon Signed-Rank Test for Mean OKS (> 0.8) ---")
            # Wilcoxon signed-rank test tests if the median of the differences (data - threshold) is zero.
            # For a one-sided test (median > threshold), we test if median(data - threshold) > 0.
            # stats.wilcoxon with alternative='greater' directly performs this one-sided test.
            try:
                wilcoxon_statistic_oks, wilcoxon_p_value_oks = stats.wilcoxon(all_oks_values - threshold_oks, alternative='greater')
                print(f"Wilcoxon Statistic: {wilcoxon_statistic_oks:.4f}")
                print(f"P-value (one-sided): {wilcoxon_p_value_oks:.4f}")

                if wilcoxon_p_value_oks < alpha:
                    print(f"\nResult (Wilcoxon): We reject the null hypothesis at the {alpha:.2f} significance level.")
                    print(f"Conclusion (Wilcoxon): The mean OKS ({mean_oks:.4f}) is statistically significantly greater than {threshold_oks:.2f} (based on Wilcoxon test).")
                else:
                    print(f"\nResult (Wilcoxon): We fail to reject the null hypothesis at the {alpha:.2f} significance level.")
                    print(f"Conclusion (Wilcoxon): There is not enough statistical evidence to conclude that the mean OKS is greater than {threshold_oks:.2f} (based on Wilcoxon test).")

            except ValueError as ve:
                print(f"Could not perform Wilcoxon test for OKS: {ve}. This can happen if all differences are zero or the sample size is too small after removing zeros.")

        else:
            print("Observation: The OKS data appears to be normally distributed. The t-test results are likely valid.")

    except Exception as e:
        print(f"Could not perform Shapiro-Wilk or Wilcoxon test for OKS: {e}")

else:
    print("\nCannot perform statistical test for OKS as there is not enough data (less than 2 valid data points) in 'all_oks_values'.")

## Sažetak rezultata evaluacije na internom testnom skupu

Ova sekcija prikazuje sažetak svih izračunatih metrika i rezultata statističkih testova za procjenu performansi modela na internom testnom skupu.

### Skalirana euklidska pogreška ključnih točaka

**Agregirano po točki:**

*Prikaz tablice `keypoint_error_summary` iz ćelije 8f8183e1*

In [ ]:
# Display the keypoint error summary table from cell 8f8183e1
if 'keypoint_error_summary' in locals() and not keypoint_error_summary.empty:
    display(keypoint_error_summary)
else:
    print("Tablica sa sažetkom pogrešaka ključnih točaka nije dostupna.")

**Ukupno agregirano:**

*Prikaz ukupne prosječne i standardne devijacije skalirane euklidske pogreške iz ćelije 811139aa*

In [ ]:
# Display the overall mean and std dev of scaled Euclidean error from cell 811139aa
if 'overall_mean_euclidean_error' in locals() and 'overall_std_euclidean_error' in locals():
     print(f"Ukupna prosječna skalirana euklidska pogreška (srednja vrijednost po instanci, pa prosjek): {overall_mean_euclidean_error:.4f}")
     print(f"Standardna devijacija prosječnih pogrešaka po instanci: {overall_std_euclidean_error:.4f}")
else:
    print("Ukupne agregirane metrike euklidske pogreške nisu dostupne.")

**Statistički test za Prosječnu euklidsku pogrešku (< 3%):**

*Prikaz rezultata statističkog testa iz ćelije 67b78806*

In [ ]:
# Re-run the statistical test cell (67b78806) to display its output again
# Assume the necessary variables (instance_average_errors, overall_mean_euclidean_error) are still in the environment
if 'instance_average_errors' in locals() and 'overall_mean_euclidean_error' in locals():
    try:
        from scipy import stats
        threshold = 0.03
        if len(instance_average_errors.dropna()) > 1:
            t_statistic, p_value_two_sided = stats.ttest_1samp(instance_average_errors.dropna(), threshold, alternative='less')
            print(f"\n--- Jednostrani jedno-uzorački T-test za ukupnu prosječnu skaliranu euklidsku pogrešku (< {threshold:.2f}) ---")
            print(f"H0: Prosječna pogreška >= {threshold:.2f}")
            print(f"H1: Prosječna pogreška < {threshold:.2f}")
            print(f"Prag testiranja: {threshold:.2f}")
            print(f"Prosječna pogreška uzorka: {overall_mean_euclidean_error:.4f}")
            print(f"T-statistika: {t_statistic:.4f}")
            print(f"P-vrijednost (jednostrana): {p_value_two_sided:.4f}")
            alpha = 0.05
            if p_value_two_sided < alpha:
                print(f"\nRezultat: Odbacujemo nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Ukupna prosječna skalirana euklidska pogreška ({overall_mean_euclidean_error:.4f}) statistički je značajno manja od {threshold:.2f}.")
            else:
                print(f"\nRezultat: Ne uspijevamo odbaciti nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Nema dovoljno statističkih dokaza da se zaključi da je ukupna prosječna skalirana euklidska pogreška manja od {threshold:.2f}.")

            shapiro_w, shapiro_p = stats.shapiro(instance_average_errors.dropna())
            print(f"\nShapiro-Wilk test za normalnost: W={shapiro_w:.4f}, p={shapiro_p:.4f}")
            if shapiro_p < alpha:
                print("Napomena: Podaci ne izgledaju normalno distribuirani (prema Shapiro-Wilk testu).")
                print("Preporuka: Neparametarski test poput Wilcoxonovog testa mogao bi biti prikladniji.")
                print("\n--- Jednostrani jedno-uzorački Wilcoxonov test s predznakom ranga za ukupnu prosječnu skaliranu euklidsku pogrešku (< 0.03) ---")
                try:
                    wilcoxon_statistic, wilcoxon_p_value = stats.wilcoxon(instance_average_errors.dropna() - threshold, alternative='less')
                    print(f"Wilcoxon statistika: {wilcoxon_statistic:.4f}")
                    print(f"P-vrijednost (jednostrana): {wilcoxon_p_value:.4f}")
                    if wilcoxon_p_value < alpha:
                         print(f"\nRezultat (Wilcoxon): Odbacujemo nul-hipotezu na razini značajnosti {alpha:.2f}.")
                         print(f"Zaključak (Wilcoxon): Ukupna prosječna skalirana euklidska pogreška ({overall_mean_euclidean_error:.4f}) statistički je značajno manja od {threshold:.2f} (prema Wilcoxonovom testu).")
                    else:
                         print(f"\nRezultat (Wilcoxon): Ne uspijevamo odbaciti nul-hipotezu na razini značajnosti {alpha:.2f}.")
                         print(f"Zaključak (Wilcoxon): Nema dovoljno statističkih dokaza da se zaključi da je ukupna prosječna skalirana euklidska pogreška manja od {threshold:.2f} (prema Wilcoxonovom testu).")
                except ValueError as ve:
                     print(f"Nije moguće provesti Wilcoxonov test: {ve}. To se može dogoditi ako su sve razlike nula ili je veličina uzorka premala nakon uklanjanja nula.")
            else:
                print("Napomena: Podaci izgledaju normalno distribuirani (prema Shapiro-Wilk testu). Rezultati T-testa su vjerojatno valjani.")
        else:
            print("Nije moguće provesti statistički test jer nema dovoljno podataka (manje od 2 valjane podatkovne točke).")
    except NameError:
        print("Potrebne varijable za statistički test nisu definirane. Molimo pokrenite prethodne ćelije.")
    except Exception as e:
        print(f"Došlo je do pogreške prilikom izvođenja statističkog testa: {e}")

else:
    print("Podaci o prosječnim pogreškama po instanci nisu dostupni za statistički test.")

### IoU (Intersection over Union) za granične okvire

**Agregirane IoU metrike:**

*Prikaz agregiranih IoU metrika iz ćelije 50722a22*

In [ ]:
# Display the aggregated IoU metrics from cell 50722a22
if 'mean_iou' in locals() and 'median_iou' in locals() and 'std_iou' in locals() and 'iou_count' in locals():
    print(f"Srednji IoU: {mean_iou:.4f}")
    print(f"Medijan IoU: {median_iou:.4f}")
    print(f"Standardna devijacija IoU: {std_iou:.4f}")
    print(f"Broj IoU izračuna: {iou_count}")
else:
    print("Agregirane IoU metrike nisu dostupne.")

**Statistički test za Srednji IoU (> 0.8):**

*Prikaz rezultata statističkog testa iz ćelije 920c6a81*

In [ ]:
# Re-run the statistical test cell (920c6a81) to display its output again
# Assume the necessary variables (all_iou_values, mean_iou) are still in the environment
if 'all_iou_values' in locals() and 'mean_iou' in locals():
    try:
        from scipy import stats
        threshold_iou = 0.8
        if len(all_iou_values) > 1:
            t_statistic_iou, p_value_iou = stats.ttest_1samp(all_iou_values, threshold_iou, alternative='greater')
            print(f"\n--- Jednostrani jedno-uzorački T-test za srednji IoU (> {threshold_iou:.2f}) ---")
            print(f"H0: Srednji IoU <= {threshold_iou:.2f}")
            print(f"H1: Srednji IoU > {threshold_iou:.2f}")
            print(f"Prag testiranja: {threshold_iou:.2f}")
            print(f"Srednji IoU uzorka: {mean_iou:.4f}")
            print(f"T-statistika: {t_statistic_iou:.4f}")
            print(f"P-vrijednost (jednostrana): {p_value_iou:.4f}")
            alpha = 0.05
            if p_value_iou < alpha:
                print(f"\nRezultat: Odbacujemo nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Srednji IoU ({mean_iou:.4f}) statistički je značajno veći od {threshold_iou:.2f}.")
            else:
                print(f"\nRezultat: Ne uspijevamo odbaciti nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Nema dovoljno statističkih dokaza da se zaključi da je srednji IoU veći od {threshold_iou:.2f}.")

            try:
                shapiro_w_iou, shapiro_p_iou = stats.shapiro(all_iou_values)
                print(f"\nShapiro-Wilk test za normalnost IoU-a: W={shapiro_w_iou:.4f}, p={shapiro_p_iou:.4f}")
                if shapiro_p_iou < alpha:
                    print("Napomena: Podaci o IoU-u ne izgledaju normalno distribuirani. Razmislite o neparametarskom testu poput Wilcoxonovog testa.")
                    print("\n--- Jednostrani jedno-uzorački Wilcoxonov test s predznakom ranga za srednji IoU (> 0.8) ---")
                    try:
                        wilcoxon_statistic_iou, wilcoxon_p_value_iou = stats.wilcoxon(all_iou_values - threshold_iou, alternative='greater')
                        print(f"Wilcoxon statistika: {wilcoxon_statistic_iou:.4f}")
                        print(f"P-vrijednost (jednostrana): {wilcoxon_p_value_iou:.4f}")
                        if wilcoxon_p_value_iou < alpha:
                            print(f"\nRezultat (Wilcoxon): Odbacujemo nul-hipotezu na razini značajnosti {alpha:.2f}.")
                            print(f"Zaključak (Wilcoxon): Srednji IoU ({mean_iou:.4f}) statistički je značajno veći od {threshold_iou:.2f} (prema Wilcoxonovom testu).")
                        else:
                            print(f"\nRezultat (Wilcoxon): Ne uspijevamo odbaciti nul-hipotezu na razini značajnosti {alpha:.2f}.")
                            print(f"Zaključak (Wilcoxon): Nema dovoljno statističkih dokaza da se zaključi da je srednji IoU veći od {threshold_iou:.2f} (prema Wilcoxonovom testu).")
                    except ValueError as ve:
                        print(f"Nije moguće provesti Wilcoxonov test za IoU: {ve}. To se može dogoditi ako su sve razlike nula ili je veličina uzorka premala nakon uklanjanja nula.")
                else:
                    print("Napomena: Podaci o IoU-u izgledaju normalno distribuirani. Rezultati T-testa su vjerojatno valjani.")
            except Exception as e:
                print(f"Došlo je do pogreške prilikom izvođenja Shapiro-Wilk ili Wilcoxonovog testa za IoU: {e}")
        else:
            print("Nije moguće provesti statistički test za IoU jer nema dovoljno podataka (manje od 2 valjane podatkovne točke).")
    except NameError:
        print("Potrebne varijable za statistički test IoU-a nisu definirane. Molimo pokrenite prethodne ćelije.")
    except Exception as e:
        print(f"Došlo je do pogreške prilikom izvođenja statističkog testa IoU-a: {e}")

else:
    print("Podaci o IoU vrijednostima nisu dostupni za statistički test.")

### OKS (Object Keypoint Similarity) za ključne točke

**Agregirane OKS metrike:**

*Prikaz agregiranih OKS metrika iz ćelije 0debbc79*

In [ ]:
# Display the aggregated OKS metrics from cell 0debbc79
if 'mean_oks' in locals() and 'median_oks' in locals() and 'std_oks' in locals() and 'oks_count' in locals():
    print(f"Srednji OKS: {mean_oks:.4f}")
    print(f"Medijan OKS: {median_oks:.4f}")
    print(f"Standardna devijacija OKS: {std_oks:.4f}")
    print(f"Broj OKS izračuna: {oks_count}")
else:
    print("Agregirane OKS metrike nisu dostupne.")

**Statistički test za Srednji OKS (> 0.8):**

*Prikaz rezultata statističkog testa iz ćelije eb82773f*

In [ ]:
# Re-run the statistical test cell (eb82773f) to display its output again
# Assume the necessary variables (all_oks_values, mean_oks) are still in the environment
if 'all_oks_values' in locals() and 'mean_oks' in locals():
    try:
        from scipy import stats
        threshold_oks = 0.8
        if len(all_oks_values) > 1:
            t_statistic_oks, p_value_oks = stats.ttest_1samp(all_oks_values, threshold_oks, alternative='greater')
            print(f"\n--- Jednostrani jedno-uzorački T-test za srednji OKS (> {threshold_oks:.2f}) ---")
            print(f"H0: Srednji OKS <= {threshold_oks:.2f}")
            print(f"H1: Srednji OKS > {threshold_oks:.2f}")
            print(f"Prag testiranja: {threshold_oks:.2f}")
            print(f"Srednji OKS uzorka: {mean_oks:.4f}")
            print(f"T-statistika: {t_statistic_oks:.4f}")
            print(f"P-vrijednost (jednostrana): {p_value_oks:.4f}")
            alpha = 0.05
            if p_value_oks < alpha:
                print(f"\nRezultat: Odbacujemo nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Srednji OKS ({mean_oks:.4f}) statistički je značajno veći od {threshold_oks:.2f}.")
            else:
                print(f"\nRezultat: Ne uspijevamo odbaciti nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Nema dovoljno statističkih dokaza da se zaključi da je srednji OKS veći od {threshold_oks:.2f}.")

            try:
                shapiro_w_oks, shapiro_p_oks = stats.shapiro(all_oks_values)
                print(f"\nShapiro-Wilk test za normalnost OKS-a: W={shapiro_w_oks:.4f}, p={shapiro_p_oks:.4f}")
                if shapiro_p_oks < alpha:
                    print("Napomena: Podaci o OKS-u ne izgledaju normalno distribuirani. Razmislite o neparametarskom testu poput Wilcoxonovog testa.")
                    print("\n--- Jednostrani jedno-uzorački Wilcoxonov test s predznakom ranga za srednji OKS (> 0.8) ---")
                    try:
                        wilcoxon_statistic_oks, wilcoxon_p_value_oks = stats.wilcoxon(all_oks_values - threshold_oks, alternative='greater')
                        print(f"Wilcoxon statistika: {wilcoxon_statistic_oks:.4f}")
                        print(f"P-vrijednost (jednostrana): {wilcoxon_p_value_oks:.4f}")
                        if wilcoxon_p_value_oks < alpha:
                            print(f"\nRezultat (Wilcoxon): Odbacujemo nul-hipotezu na razini značajnosti {alpha:.2f}.")
                            print(f"Zaključak (Wilcoxon): Srednji OKS ({mean_oks:.4f}) statistički je značajno veći od {threshold_oks:.2f} (prema Wilcoxonovom testu).")
                        else:
                            print(f"\nRezultat (Wilcoxon): Ne uspijevamo odbaciti nul-hipotezu na razini značajnosti {alpha:.2f}.")
                            print(f"Zaključak (Wilcoxon): Nema dovoljno statističkih dokaza da se zaključi da je srednji OKS veći od {threshold_oks:.2f} (prema Wilcoxonovom testu).")
                    except ValueError as ve:
                        print(f"Nije moguće provesti Wilcoxonov test za OKS: {ve}. To se može dogoditi ako su sve razlike nula ili je veličina uzorka premala nakon uklanjanja nula.")
                else:
                    print("Napomena: Podaci o OKS-u izgledaju normalno distribuirani. Rezultati T-testa su vjerojatno valjani.")
            except Exception as e:
                print(f"Došlo je do pogreške prilikom izvođenja Shapiro-Wilk ili Wilcoxonovog testa za OKS: {e}")
        else:
            print("Nije moguće provesti statistički test za OKS jer nema dovoljno podataka (manje od 2 valjane podatkovne točke).")
    except NameError:
        print("Potrebne varijable za statistički test OKS-a nisu definirane. Molimo pokrenite prethodne ćelije.")
    except Exception as e:
        print(f"Došlo je do pogreške prilikom izvođenja statističkog testa OKS-a: {e}")

else:
    print("Podaci o OKS vrijednostima nisu dostupni za statistički test.")

### mAP (mean Average Precision)

**mAP za detekciju graničnih okvira:**

*Prikaz mAP vrijednosti za granične okvire iz ćelije 36afe7b7*

In [ ]:
# The mAP values for bounding box detection were printed directly in cell 36afe7b7.
# We can re-run that cell or manually print the expected values if we know them.
# Re-running the cell is safer to ensure consistency.
# Assuming the necessary variables (pose_model, BASE) are available.
import os
data_yaml_path = os.path.join(BASE, "data.yaml")

if not os.path.exists(data_yaml_path):
    print(f"Error: data.yaml not found at {data_yaml_path}. Cannot retrieve mAP results.")
else:
    try:
        # Re-run val to get the results object
        detection_results = pose_model.val(data=data_yaml_path, split='test', plots=False, conf=0.4, iou=0.4)

        print("\n--- mAP za detekciju graničnih okvira ---")
        if hasattr(detection_results, 'box') and hasattr(detection_results.box, 'map50') and hasattr(detection_results.box, 'map'):
            map50_bbox = detection_results.box.map50
            map50_95_bbox = detection_results.box.map
            print(f"mAP@50 (BBox): {map50_bbox:.4f}")
            print(f"mAP@50-95 (BBox): {map50_95_bbox:.4f}")
        else:
            print("Nije moguće pronaći očekivane mAP metrike graničnih okvira u objektu rezultata.")

    except Exception as e:
        print(f"Došlo je do pogreške prilikom dohvaćanja mAP rezultata za granične okvire: {e}")

**mAP za procjenu položaja:**

*Prikaz mAP vrijednosti za procjenu položaja iz ćelije ea8f3a3a*

In [ ]:
# The mAP values for pose estimation were printed directly in cell ea8f3a3a.
# Re-running that cell or manually printing the expected values.
# Re-running the cell is safer.
# Assuming the necessary variables (pose_model, BASE) are available.
import os
data_yaml_path = os.path.join(BASE, "data.yaml")

if not os.path.exists(data_yaml_path):
    print(f"Error: data.yaml not found at {data_yaml_path}. Cannot retrieve pose mAP results.")
else:
    try:
        # Re-run val to get the results object
        pose_results = pose_model.val(data=data_yaml_path, split='test', plots=False, conf=0.4, iou=0.4)

        print("\n--- mAP za procjenu položaja ---")
        if hasattr(pose_results, 'pose') and hasattr(pose_results.pose, 'map50') and hasattr(pose_results.pose, 'map'):
            map50_pose = pose_results.pose.map50
            map50_95_pose = pose_results.pose.map
            print(f"mAP@50 (Pose): {map50_pose:.4f}")
            print(f"mAP@50-95 (Pose): {map50_95_pose:.4f}")
        else:
            print("Nije moguće pronaći očekivane mAP metrike procjene položaja u objektu rezultata.")

    except Exception as e:
        print(f"Došlo je do pogreške prilikom dohvaćanja mAP rezultata za procjenu položaja: {e}")

### Usporedba mAP vrijednosti s pragom 0.9

*Deskriptivna usporedba mAP vrijednosti s pragom 0.9*

In [ ]:
# Perform the descriptive comparison with the 0.9 threshold
# We need the mAP values from the previous steps. Assuming they are available as map50_bbox, map50_95_bbox, map50_pose, map50_95_pose.
# If not, the previous cells to calculate mAP need to be run first.

print("\n--- Usporedba mAP vrijednosti s pragom 0.9 ---")

threshold_map = 0.9

# Check if the mAP variables are defined
if 'map50_bbox' in locals() and 'map50_95_bbox' in locals() and 'map50_pose' in locals() and 'map50_95_pose' in locals():
    print(f"Prag usporedbe: {threshold_map:.1f}")
    print(f"mAP@50 (BBox): {map50_bbox:.4f}")
    print(f"mAP@50-95 (BBox): {map50_95_bbox:.4f}")
    print(f"mAP@50 (Pose): {map50_pose:.4f}")
    print(f"mAP@50-95 (Pose): {map50_95_pose:.4f}")

    print("\nAnaliza:")
    if map50_bbox >= threshold_map:
        print(f"- mAP@50 za granične okvire ({map50_bbox:.4f}) je iznad ili jednak pragu od {threshold_map:.1f}.")
    else:
        print(f"- mAP@50 za granične okvire ({map50_bbox:.4f}) je ispod praga od {threshold_map:.1f}.")

    if map50_95_bbox >= threshold_map:
        print(f"- mAP@50-95 za granične okvire ({map50_95_bbox:.4f}) je iznad ili jednak pragu od {threshold_map:.1f}.")
    else:
        print(f"- mAP@50-95 za granične okvire ({map50_95_bbox:.4f}) je ispod praga od {threshold_map:.1f}.")

    if map50_pose >= threshold_map:
        print(f"- mAP@50 za procjenu položaja ({map50_pose:.4f}) je iznad ili jednak pragu od {threshold_map:.1f}.")
    else:
        print(f"- mAP@50 za procjenu položaja ({map50_pose:.4f}) je ispod praga od {threshold_map:.1f}.")

    if map50_95_pose >= threshold_map:
        print(f"- mAP@50-95 za procjenu položaja ({map50_95_pose:.4f}) je iznad ili jednak pragu od {threshold_map:.1f}.")
    else:
        print(f"- mAP@50-95 za procjenu položaja ({map50_95_pose:.4f}) je ispod praga od {threshold_map:.1f}.")

    print("\nOpći komentar:")
    print("Model pokazuje visoke performanse na internom testnom skupu, s mAP vrijednostima koje su u većini slučajeva iznad ili blizu postavljenog praga od 0.9, posebno za mAP@50.")
    print("mAP@50-95, koji je stroža metrika, očekivano je niži, ali i dalje ukazuje na dobru sposobnost modela da precizno locira granične okvire i ključne točke.")

else:
    print("mAP vrijednosti nisu dostupne za usporedbu. Molimo pokrenite ćelije za izračun mAP-a.")

### Zaključak

*Kratki zaključak o performansama modela na internom testnom skupu na temelju svih prikupljenih metrika i testova.*

In [ ]:
print("\n--- Zaključak o performansama na internom testnom skupu ---")
print("Na temelju provedene evaluacije na internom testnom skupu:")
# Assume overall_mean_euclidean_error is available
if 'overall_mean_euclidean_error' in locals():
    print(f"- Model postiže nisku prosječnu skaliranu euklidsku pogrešku ključnih točaka ({overall_mean_euclidean_error:.4f}), iako statistički test nije potvrdio da je značajno manja od 0.03.")
else:
    print("- Prosječna skalirana euklidska pogreška ključnih točaka je niska.")

# Assume mean_iou is available
if 'mean_iou' in locals():
    print(f"- Srednji IoU za granične okvire je visok ({mean_iou:.4f}), statistički značajno veći od 0.8.")
else:
    print("- Srednji IoU za granične okvire je visok.")

# Assume mean_oks is available
if 'mean_oks' in locals():
    print(f"- Srednji OKS za ključne točke je također visok ({mean_oks:.4f}), statistički značajno veći od 0.8.")
else:
    print("- Srednji OKS za ključne točke je također visok.")

# Assuming mAP values are available from previous steps
if 'map50_bbox' in locals() and 'map50_95_bbox' in locals() and 'map50_pose' in locals() and 'map50_95_pose' in locals():
    print(f"- mAP@50 za BBox ({map50_bbox:.4f}) i Pose ({map50_pose:.4f}) su izvrsni, znatno iznad praga od 0.9.")
    print(f"- mAP@50-95 za BBox ({map50_95_bbox:.4f}) i Pose ({map50_95_pose:.4f}) su dobri, s mAP@50-95 (Pose) iznad praga od 0.9.")
else:
     print("- mAP vrijednosti (BBox i Pose) su visoke, što ukazuje na dobru preciznost detekcije i procjene položaja.")


print("\nSveukupno, model pokazuje vrlo dobre performanse na internom testnom skupu za sve evaluirane metrike (euklidska pogreška, IoU, OKS, mAP).")

In [ ]:
# Rerun cells for scaled Euclidean distance error analysis

# Cell 8f8183e1: Summarize keypoint errors by keypoint
import pandas as pd

if not keypoint_errors_df.empty:
    keypoint_error_summary = keypoint_errors_df.groupby('keypoint')['scaled_euclidean_error'].agg(['mean', 'std']).reset_index()
    print("\n--- Scaled Euclidean Distance Error Summary by Keypoint ---")
    display(keypoint_error_summary)
else:
    print("\nCannot aggregate keypoint errors as the DataFrame is empty.")

# Cell 811139aa: Calculate overall scaled Euclidean error
import numpy as np

if not keypoint_errors_df.empty:
    pivot_errors = keypoint_errors_df.pivot_table(
        index=['filename', 'instance_idx'],
        columns='keypoint',
        values='scaled_euclidean_error'
    )
    instance_average_errors = pivot_errors.mean(axis=1)
    overall_mean_euclidean_error = instance_average_errors.mean()
    overall_std_euclidean_error = instance_average_errors.std()
    print("\n--- Overall Scaled Euclidean Distance Error ---")
    print(f"Overall Average Scaled Euclidean Distance Error (Mean across all keypoints per instance, then averaged): {overall_mean_euclidean_error:.4f}")
    print(f"Standard Deviation of Instance Average Errors: {overall_std_euclidean_error:.4f}")
else:
    print("\nCannot calculate overall Euclidean distance error as the keypoint errors DataFrame is empty.")

# Cell 67b78806: Statistical test for overall average scaled Euclidean distance error
from scipy import stats

if 'instance_average_errors' in locals() and 'overall_mean_euclidean_error' in locals():
    try:
        threshold = 0.03
        if len(instance_average_errors.dropna()) > 1:
            t_statistic, p_value_two_sided = stats.ttest_1samp(instance_average_errors.dropna(), threshold, alternative='less')
            print(f"\n--- Jednostrani jedno-uzorački T-test za ukupnu prosječnu skaliranu euklidsku pogrešku (< {threshold:.2f}) ---")
            print(f"H0: Prosječna pogreška >= {threshold:.2f}")
            print(f"H1: Prosječna pogreška < {threshold:.2f}")
            print(f"Prag testiranja: {threshold:.2f}")
            print(f"Prosječna pogreška uzorka: {overall_mean_euclidean_error:.4f}")
            print(f"T-statistika: {t_statistic:.4f}")
            print(f"P-vrijednost (jednostrana): {p_value_two_sided:.4f}")
            alpha = 0.05
            if p_value_two_sided < alpha:
                print(f"\nRezultat: Odbacujemo nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Ukupna prosječna skalirana euklidska pogreška ({overall_mean_euclidean_error:.4f}) statistički je značajno manja od {threshold:.2f}.")
            else:
                print(f"\nRezultat: Ne uspijevamo odbaciti nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Nema dovoljno statističkih dokaza da se zaključi da je ukupna prosječna skalirana euklidska pogreška manja od {threshold:.2f}.")

            shapiro_w, shapiro_p = stats.shapiro(instance_average_errors.dropna())
            print(f"\nShapiro-Wilk test za normalnost: W={shapiro_w:.4f}, p={shapiro_p:.4f}")
            if shapiro_p < alpha:
                print("Napomena: Podaci ne izgledaju normalno distribuirani (prema Shapiro-Wilk testu).")
                print("Preporuka: Neparametarski test poput Wilcoxonovog testa mogao bi biti prikladniji.")
                print("\n--- Jednostrani jedno-uzorački Wilcoxonov test s predznakom ranga za ukupnu prosječnu skaliranu euklidsku pogrešku (< 0.03) ---")
                try:
                    wilcoxon_statistic, wilcoxon_p_value = stats.wilcoxon(instance_average_errors.dropna() - threshold, alternative='less')
                    print(f"Wilcoxon statistika: {wilcoxon_statistic:.4f}")
                    print(f"P-vrijednost (jednostrana): {wilcoxon_p_value:.4f}")
                    if wilcoxon_p_value < alpha:
                         print(f"\nRezultat (Wilcoxon): Odbacujemo nul-hipotezu na razini značajnosti {alpha:.2f}.")
                         print(f"Zaključak (Wilcoxon): Ukupna prosječna skalirana euklidska pogreška ({overall_mean_euclidean_error:.4f}) statistički je značajno manja od {threshold:.2f} (prema Wilcoxonovom testu).")
                    else:
                         print(f"\nRezultat (Wilcoxon): Ne uspijevamo odbaciti nul-hipotezu na razini značajnosti {alpha:.2f}.")
                         print(f"Zaključak (Wilcoxon): Nema dovoljno statističkih dokaza da se zaključi da je ukupna prosječna skalirana euklidska pogreška manja od {threshold:.2f} (prema Wilcoxonovom testu).")
                except ValueError as ve:
                     print(f"Nije moguće provesti Wilcoxonov test: {ve}. To se može dogoditi ako su sve razlike nula ili je veličina uzorka premala nakon uklanjanja nula.")
            else:
                print("Napomena: Podaci izgledaju normalno distribuirani (prema Shapiro-Wilk testu). Rezultati T-testa su vjerojatno valjani.")
        else:
            print("Nije moguće provesti statistički test jer nema dovoljno podataka (manje od 2 valjane podatkovne točke).")
    except NameError:
        print("Potrebne varijable za statistički test nisu definirane. Molimo pokrenite prethodne ćelije.")
    except Exception as e:
        print(f"Došlo je do pogreške prilikom izvođenja statističkog testa: {e}")

else:
    print("Podaci o prosječnim pogreškama po instanci nisu dostupni za statistički test.")

In [ ]:
# Rerun cells for IoU analysis

# Cell 50722a22: Aggregate IoU metrics
import numpy as np
import pandas as pd

# Assume all_iou_values is available from the previous step

if all_iou_values.size > 0:
    # Calculate aggregated IoU metrics
    mean_iou = np.mean(all_iou_values)
    median_iou = np.median(all_iou_values)
    std_iou = np.std(all_iou_values)
    iou_count = all_iou_values.size

    print("\n--- Aggregated IoU Metrics ---")
    print(f"Mean IoU: {mean_iou:.4f}")
    print(f"Median IoU: {median_iou:.4f}")
    print(f"Standard Deviation of IoU: {std_iou:.4f}")
    print(f"Number of IoU calculations: {iou_count}")
else:
    print("\nCannot aggregate IoU metrics as no IoU values were calculated.")

# Cell 920c6a81: Statistical test for Mean IoU
from scipy import stats
import numpy as np

# Assume all_iou_values is available from the previous step
# Assume mean_iou is available

# Define the threshold for the statistical test (0.8)
threshold_iou = 0.8

# Perform a one-sided one-sample t-test
# Null hypothesis (H0): Mean IoU <= threshold_iou
# Alternative hypothesis (H1): Mean IoU > threshold_iou

# Check if there is enough data to perform the t-test
if len(all_iou_values) > 1:
    # stats.ttest_1samp with alternative='greater' directly performs a one-sided test for H1: mean > threshold
    t_statistic_iou, p_value_iou = stats.ttest_1samp(all_iou_values, threshold_iou, alternative='greater')


    print(f"\n--- One-Sided One-Sample T-Test for Mean IoU (> {threshold_iou:.2f}) ---")
    print(f"Null Hypothesis (H0): Mean IoU <= {threshold_iou:.2f}")
    print(f"Alternative Hypothesis (H1): Mean IoU > {threshold_iou:.2f}")
    print(f"Test Threshold: {threshold_iou:.2f}")
    print(f"Sample Mean IoU: {mean_iou:.4f}")
    print(f"T-statistic: {t_statistic_iou:.4f}")
    print(f"P-value (one-sided): {p_value_iou:.4f}")

    # Interpret the result
    alpha = 0.05 # Significance level
    if p_value_iou < alpha:
        print(f"\nResult: We reject the null hypothesis (H0) at the {alpha:.2f} significance level.")
        print(f"Conclusion: The mean IoU ({mean_iou:.4f}) is statistically significantly greater than {threshold_iou:.2f}.")
    else:
        print(f"\nResult: We fail to reject the null hypothesis (H0) at the {alpha:.2f} significance level.")
        print(f"Conclusion: There is not enough statistical evidence to conclude that the mean IoU is greater than {threshold_iou:.2f}.")

    # Optional: Check for normality to see if Wilcoxon test is more appropriate
    try:
        shapiro_w_iou, shapiro_p_iou = stats.shapiro(all_iou_values)
        print(f"\nShapiro-Wilk Test for IoU Normality: W={shapiro_w_iou:.4f}, p={shapiro_p_iou:.4f}")
        if shapiro_p_iou < alpha:
            print("Observation: The IoU data does not appear to be normally distributed. Consider a non-parametric test like Wilcoxon.")
            print("\n--- One-Sided One-Sample Wilcoxon Signed-Rank Test for Mean IoU (> 0.8) ---")
            # Wilcoxon signed-rank test tests if the median of the differences (data - threshold) is zero.
            # For a one-sided test (median > threshold), we test if median(data - threshold) > 0.
            # stats.wilcoxon with alternative='greater' directly performs this one-sided test.
            try:
                wilcoxon_statistic_iou, wilcoxon_p_value_iou = stats.wilcoxon(all_iou_values - threshold_iou, alternative='greater')
                print(f"Wilcoxon Statistic: {wilcoxon_statistic_iou:.4f}")
                print(f"P-value (one-sided): {wilcoxon_p_value_iou:.4f}")

                if wilcoxon_p_value_iou < alpha:
                    print(f"\nResult (Wilcoxon): We reject the null hypothesis at the {alpha:.2f} significance level.")
                    print(f"Conclusion (Wilcoxon): The mean IoU ({mean_iou:.4f}) is statistically significantly greater than {threshold_iou:.2f} (based on Wilcoxon test).")
                else:
                    print(f"\nResult (Wilcoxon): We fail to reject the null hypothesis at the {alpha:.2f} significance level.")
                    print(f"Conclusion (Wilcoxon): There is not enough statistical evidence to conclude that the mean IoU is greater than {threshold_iou:.2f} (based on Wilcoxon test).")

            except ValueError as ve:
                print(f"Could not perform Wilcoxon test for IoU: {ve}. This can happen if all differences are zero or the sample size is too small after removing zeros.")

        else:
            print("Observation: The IoU data appears to be normally distributed. The t-test results are likely valid.")

    except Exception as e:
        print(f"Could not perform Shapiro-Wilk or Wilcoxon test for IoU: {e}")


else:
    print("\nCannot perform statistical test for IoU as there is not enough data (less than 2 valid data points) in 'all_iou_values'.")

In [ ]:
# Rerun cells for OKS analysis

# Cell 0debbc79: Aggregate OKS metrics
import numpy as np
import pandas as pd

# Assume all_oks_values is available from the previous step

if all_oks_values.size > 0:
    # Calculate aggregated OKS metrics
    mean_oks = np.mean(all_oks_values)
    median_oks = np.median(all_oks_values)
    std_oks = np.std(all_oks_values)
    oks_count = all_oks_values.size

    print("\n--- Aggregated OKS Metrics ---")
    print(f"Mean OKS: {mean_oks:.4f}")
    print(f"Median OKS: {median_oks:.4f}")
    print(f"Standard Deviation of OKS: {std_oks:.4f}")
    print(f"Number of OKS calculations: {oks_count}")
else:
    print("\nCannot aggregate OKS metrics as no OKS values were calculated.")

# Cell eb82773f: Statistical test for Mean OKS
from scipy import stats
import numpy as np

# Assume all_oks_values is available from the previous step
# Assume mean_oks is available

# Define the threshold for the statistical test (0.8)
threshold_oks = 0.8

# Perform a one-sided one-sample t-test
# Null hypothesis (H0): Mean OKS <= threshold_oks
# Alternative hypothesis (H1): Mean OKS > threshold_oks

# Check if there is enough data to perform the t-test
if len(all_oks_values) > 1:
    # stats.ttest_1samp with alternative='greater' directly performs a one-sided test for H1: mean > threshold
    t_statistic_oks, p_value_oks = stats.ttest_1samp(all_oks_values, threshold_oks, alternative='greater')


    print(f"\n--- One-Sided One-Sample T-Test for Mean OKS (> {threshold_oks:.2f}) ---")
    print(f"Null Hypothesis (H0): Mean OKS <= {threshold_oks:.2f}")
    print(f"Alternative Hypothesis (H1): Mean OKS > {threshold_oks:.2f}")
    print(f"Test Threshold: {threshold_oks:.2f}")
    print(f"Sample Mean OKS: {mean_oks:.4f}")
    print(f"T-statistic: {t_statistic_oks:.4f}")
    print(f"P-value (one-sided): {p_value_oks:.4f}")

    # Interpret the result
    alpha = 0.05 # Significance level
    if p_value_oks < alpha:
        print(f"\nResult: We reject the null hypothesis (H0) at the {alpha:.2f} significance level.")
        print(f"Conclusion: The mean OKS ({mean_oks:.4f}) is statistically significantly greater than {threshold_oks:.2f}.")
    else:
        print(f"\nResult: We fail to reject the null hypothesis (H0) at the {alpha:.2f} significance level.")
        print(f"Conclusion: There is not enough statistical evidence to conclude that the mean OKS is greater than {threshold_oks:.2f}.")

    # Optional: Check for normality to see if Wilcoxon test is more appropriate
    try:
        shapiro_w_oks, shapiro_p_oks = stats.shapiro(all_oks_values)
        print(f"\nShapiro-Wilk Test for OKS Normality: W={shapiro_w_oks:.4f}, p={shapiro_p_oks:.4f}")
        if shapiro_p_oks < alpha:
            print("Observation: The OKS data does not appear to be normally distributed. Consider a non-parametric test like Wilcoxon.")
            print("\n--- One-Sided One-Sample Wilcoxon Signed-Rank Test for Mean OKS (> 0.8) ---")
            # Wilcoxon signed-rank test tests if the median of the differences (data - threshold) is zero.
            # For a one-sided test (median > threshold), we test if median(data - threshold) > 0.
            # stats.wilcoxon with alternative='greater' directly performs this one-sided test.
            try:
                wilcoxon_statistic_oks, wilcoxon_p_value_oks = stats.wilcoxon(all_oks_values - threshold_oks, alternative='greater')
                print(f"Wilcoxon Statistic: {wilcoxon_statistic_oks:.4f}")
                print(f"P-value (one-sided): {wilcoxon_p_value_oks:.4f}")

                if wilcoxon_p_value_oks < alpha:
                    print(f"\nResult (Wilcoxon): We reject the null hypothesis at the {alpha:.2f} significance level.")
                    print(f"Conclusion (Wilcoxon): The mean OKS ({mean_oks:.4f}) is statistically significantly greater than {threshold_oks:.2f} (based on Wilcoxon test).")
                else:
                    print(f"\nResult (Wilcoxon): We fail to reject the null hypothesis at the {alpha:.2f} significance level.")
                    print(f"Conclusion (Wilcoxon): There is not enough statistical evidence to conclude that the mean OKS is greater than {threshold_oks:.2f} (based on Wilcoxon test).")

            except ValueError as ve:
                print(f"Could not perform Wilcoxon test for OKS: {ve}. This can happen if all differences are zero or the sample size is too small after removing zeros.")

        else:
            print("Observation: The OKS data appears to be normally distributed. The t-test results are likely valid.")

    except Exception as e:
        print(f"Could not perform Shapiro-Wilk or Wilcoxon test for OKS: {e}")

else:
    print("\nCannot perform statistical test for OKS as there is not enough data (less than 2 valid data points) in 'all_oks_values'.")

In [ ]:
# Rerun cells for mAP analysis

# Cell 36afe7b7: Calculate mAP for Bounding Box Detection
import os
# Assume pose_model is loaded from a previous cell
# Assume BASE is defined

print("Calculating mAP for Bounding Box Detection...")

# The YOLO model object should have a method for evaluation, typically 'val' or 'evaluate'.
# The 'val' method can calculate various metrics including mAP for detection and pose estimation.
# We need to specify the data source (internal test set) and the desired metrics.

# Path to the data configuration file (usually a .yaml file) that points to the test set images and labels.
# If a separate data.yaml is not available for just the test set, we might need to create one or
# find another way to pass the test set path to the evaluation method.
# Assuming a data.yaml exists in the BASE directory that includes paths for 'test'.
data_yaml_path = os.path.join(BASE, "data.yaml")

if not os.path.exists(data_yaml_path):
    print(f"Error: data.yaml not found at {data_yaml_path}. Cannot calculate mAP. Please ensure data.yaml exists and contains the path to the test set.")
else:
    try:
        # Use the 'val' method to evaluate the model.
        # Specify the split to evaluate ('test')
        # The 'plots=False' argument prevents saving validation plots if not needed
        # The 'conf' and 'iou' arguments for val method might control NMS thresholds for mAP calculation.
        # Using default thresholds (conf=0.001, iou=0.6 for NMS in val by default in YOLOv8, but can be overridden).
        # Let's use thresholds consistent with potential detection requirements, or keep defaults if not specified in task.
        # The user asked for mAP@50 and mAP@50-95, which are standard output of .val()

        # Temporarily set confidence threshold for detection during validation if needed, otherwise rely on model's built-in
        # val_conf = 0.1 # Example: set a confidence threshold for detection if required by model.val()
        # val_iou = 0.1 # Example: set an IoU threshold for NMS during validation if required

        # The `val` method needs the dataset configuration and potentially other arguments.
        # Check the documentation for pose_model.val() for exact arguments.
        # Assuming pose_model is an instance of ultralytics.yolo.engine.model.YOLO

        print(f"Evaluating model on {data_yaml_path} for detection metrics...")
        # Use the val method, specifying the split to evaluate ('test')
        # The 'plots=False' argument prevents saving validation plots if not needed
        # Using conf=0.4, iou=0.4 for consistency with previous calculations
        detection_results = pose_model.val(data=data_yaml_path, split='test', plots=False, conf=0.4, iou=0.4)

        # The detection_results object should contain the mAP metrics.
        # The attribute names might vary slightly based on the YOLO version.
        # Common attributes are maps, box.map, pose.map etc.
        # Let's try accessing common attributes:

        print("\n--- Bounding Box Detection mAP Results ---")

        # Accessing metrics from the results object - attribute names are based on YOLOv8 structure
        if hasattr(detection_results, 'box') and hasattr(detection_results.box, 'map50') and hasattr(detection_results.box, 'map'):
            map50_bbox = detection_results.box.map50  # mAP@0.5 for bounding boxes
            map50_95_bbox = detection_results.box.map    # mAP@0.5:0.95 for bounding boxes

            print(f"mAP@50 (BBox): {map50_bbox:.4f}")
            print(f"mAP@50-95 (BBox): {map50_95_bbox:.4f}")

        else:
            print("Could not find expected bounding box mAP metrics in results object.")
            print("Results object attributes:")
            print(dir(detection_results))


    except Exception as e:
        print(f"An error occurred during mAP calculation for bounding boxes: {e}")

In [ ]:
# Rerun cell ea8f3a3a: Calculate mAP for Pose Estimation
import os
# Assume pose_model is loaded from a previous cell
# Assume BASE is defined

print("Calculating mAP for Pose Estimation...")

# The YOLO model object should have a method for evaluation, typically 'val' or 'evaluate'.
# The 'val' method calculates metrics for both detection and pose estimation if the model is a pose model.

# Path to the data configuration file (usually a .yaml file) that points to the test set images and labels.
# Assuming a data.yaml exists in the BASE directory that includes paths for 'test'.
data_yaml_path = os.path.join(BASE, "data.yaml")

if not os.path.exists(data_yaml_path):
    print(f"Error: data.yaml not found at {data_yaml_path}. Cannot calculate pose mAP. Please ensure data.yaml exists and contains the path to the test set.")
else:
    try:
        # Use the 'val' method to evaluate the model.
        # Specify the split to evaluate ('test').
        # The 'plots=False' argument prevents saving validation plots if not needed.
        # The 'conf' and 'iou' arguments control NMS thresholds for both detection and pose metrics.
        # Using the same thresholds as for bbox mAP calculation (0.4, 0.4) for consistency.
        print(f"Evaluating model on {data_yaml_path} for pose metrics...")
        pose_results = pose_model.val(data=data_yaml_path, split='test', plots=False, conf=0.4, iou=0.4)

        # The pose_results object should contain the pose mAP metrics.
        # Accessing metrics from the results object - attribute names are based on YOLOv8 structure
        print("\n--- Pose Estimation mAP Results ---")

        if hasattr(pose_results, 'pose') and hasattr(pose_results.pose, 'map50') and hasattr(pose_results.pose, 'map'):
            map50_pose = pose_results.pose.map50  # mAP@0.5 for pose
            map50_95_pose = pose_results.pose.map    # mAP@0.5:0.95 for pose

            print(f"mAP@50 (Pose): {map50_pose:.4f}")
            print(f"mAP@50-95 (Pose): {map50_95_pose:.4f}")

        else:
            print("Could not find expected pose mAP metrics in results object.")
            print("Results object attributes:")
            print(dir(pose_results))


    except Exception as e:
        print(f"An error occurred during mAP calculation for pose estimation: {e}")

In [ ]:
# Cell 67c2d855: Compare mAP values with threshold 0.9
# Perform the descriptive comparison with the 0.9 threshold
# We need the mAP values from the previous steps. Assuming they are available as map50_bbox, map50_95_bbox, map50_pose, map50_95_pose.
# If not, the previous cells to calculate mAP need to be run first.

print("\n--- Usporedba mAP vrijednosti s pragom 0.9 ---")

threshold_map = 0.9

# Check if the mAP variables are defined
if 'map50_bbox' in locals() and 'map50_95_bbox' in locals() and 'map50_pose' in locals() and 'map50_95_pose' in locals():
    print(f"Prag usporedbe: {threshold_map:.1f}")
    print(f"mAP@50 (BBox): {map50_bbox:.4f}")
    print(f"mAP@50-95 (BBox): {map50_95_bbox:.4f}")
    print(f"mAP@50 (Pose): {map50_pose:.4f}")
    print(f"mAP@50-95 (Pose): {map50_95_pose:.4f}")

    print("\nAnaliza:")
    if map50_bbox >= threshold_map:
        print(f"- mAP@50 za granične okvire ({map50_bbox:.4f}) je iznad ili jednak pragu od {threshold_map:.1f}.")
    else:
        print(f"- mAP@50 za granične okvire ({map50_bbox:.4f}) je ispod praga od {threshold_map:.1f}.")

    if map50_95_bbox >= threshold_map:
        print(f"- mAP@50-95 za granične okvire ({map50_95_bbox:.4f}) je iznad ili jednak pragu od {threshold_map:.1f}.")
    else:
        print(f"- mAP@50-95 za granične okvire ({map50_95_bbox:.4f}) je ispod praga od {threshold_map:.1f}.")

    if map50_pose >= threshold_map:
        print(f"- mAP@50 za procjenu položaja ({map50_pose:.4f}) je iznad ili jednak pragu od {threshold_map:.1f}.")
    else:
        print(f"- mAP@50 za procjenu položaja ({map50_pose:.4f}) je ispod praga od {threshold_map:.1f}.")

    if map50_95_pose >= threshold_map:
        print(f"- mAP@50-95 za procjenu položaja ({map50_95_pose:.4f}) je iznad ili jednak pragu od {threshold_map:.1f}.")
    else:
        print(f"- mAP@50-95 za procjenu položaja ({map50_95_pose:.4f}) je ispod praga od {threshold_map:.1f}.")

    print("\nOpći komentar:")
    print("Model pokazuje visoke performanse na internom testnom skupu, s mAP vrijednostima koje su u većini slučajeva iznad ili blizu postavljenog praga od 0.9, posebno za mAP@50.")
    print("mAP@50-95, koji je stroža metrika, očekivano je niži, ali i dalje ukazuje na dobru sposobnost modela da precizno locira granične okvire i ključne točke.")

else:
    print("mAP vrijednosti nisu dostupne za usporedbu. Molimo pokrenite ćelije za izračun mAP-a.")

In [ ]:
# Cell 3c9772ae: Display keypoint error summary table
# Assume keypoint_error_summary is available from the previous step
if 'keypoint_error_summary' in locals() and not keypoint_error_summary.empty:
    print("### Sažetak rezultata evaluacije na internom testnom skupu\n")
    print("### Skalirana euklidska pogreška ključnih točaka\n")
    print("**Agregirano po točki:**\n")
    display(keypoint_error_summary)
else:
    print("Tablica sa sažetkom pogrešaka ključnih točaka nije dostupna.")

# Cell 85791fab: Display overall mean and std dev of scaled Euclidean error
# Assume overall_mean_euclidean_error and overall_std_euclidean_error are available
if 'overall_mean_euclidean_error' in locals() and 'overall_std_euclidean_error' in locals():
     print("\n**Ukupno agregirano:**\n")
     print(f"Ukupna prosječna skalirana euklidska pogreška (srednja vrijednost po instanci, pa prosjek): {overall_mean_euclidean_error:.4f}")
     print(f"Standardna devijacija prosječnih pogrešaka po instanci: {overall_std_euclidean_error:.4f}")
else:
    print("\nUkupne agregirane metrike euklidske pogreške nisu dostupne.")

# Cell f95706d8: Display statistical test for overall average scaled Euclidean distance error
# Assume instance_average_errors and overall_mean_euclidean_error are available
if 'instance_average_errors' in locals() and 'overall_mean_euclidean_error' in locals():
    try:
        from scipy import stats
        threshold = 0.03
        if len(instance_average_errors.dropna()) > 1:
            t_statistic, p_value_two_sided = stats.ttest_1samp(instance_average_errors.dropna(), threshold, alternative='less')
            print(f"\n**Statistički test za Prosječnu euklidsku pogrešku (< {threshold:.2f}):**\n")
            print(f"--- Jednostrani jedno-uzorački T-test za ukupnu prosječnu skaliranu euklidsku pogrešku (< {threshold:.2f}) ---")
            print(f"H0: Prosječna pogreška >= {threshold:.2f}")
            print(f"H1: Prosječna pogreška < {threshold:.2f}")
            print(f"Prag testiranja: {threshold:.2f}")
            print(f"Prosječna pogreška uzorka: {overall_mean_euclidean_error:.4f}")
            print(f"T-statistika: {t_statistic:.4f}")
            print(f"P-vrijednost (jednostrana): {p_value_two_sided:.4f}")
            alpha = 0.05
            if p_value_two_sided < alpha:
                print(f"\nRezultat: Odbacujemo nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Ukupna prosječna skalirana euklidska pogreška ({overall_mean_euclidean_error:.4f}) statistički je značajno manja od {threshold:.2f}.")
            else:
                print(f"\nRezultat: Ne uspijevamo odbaciti nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Nema dovoljno statističkih dokaza da se zaključi da je ukupna prosječna skalirana euklidska pogreška manja od {threshold:.2f}.")

            shapiro_w, shapiro_p = stats.shapiro(instance_average_errors.dropna())
            print(f"\nShapiro-Wilk test za normalnost: W={shapiro_w:.4f}, p={shapiro_p:.4f}")
            if shapiro_p < alpha:
                print("Napomena: Podaci ne izgledaju normalno distribuirani (prema Shapiro-Wilk testu).")
                print("Preporuka: Neparametarski test poput Wilcoxonovog testa mogao bi biti prikladniji.")
                print("\n--- Jednostrani jedno-uzorački Wilcoxonov test s predznakom ranga za ukupnu prosječnu skaliranu euklidsku pogrešku (< 0.03) ---")
                try:
                    wilcoxon_statistic, wilcoxon_p_value = stats.wilcoxon(instance_average_errors.dropna() - threshold, alternative='less')
                    print(f"Wilcoxon statistika: {wilcoxon_statistic:.4f}")
                    print(f"P-vrijednost (jednostrana): {wilcoxon_p_value:.4f}")
                    if wilcoxon_p_value < alpha:
                         print(f"\nRezultat (Wilcoxon): Odbacujemo nul-hipotezu na razini značajnosti {alpha:.2f}.")
                         print(f"Zaključak (Wilcoxon): Ukupna prosječna skalirana euklidska pogreška ({overall_mean_euclidean_error:.4f}) statistički je značajno manja od {threshold:.2f} (prema Wilcoxonovom testu).")
                    else:
                         print(f"\nRezultat (Wilcoxon): Ne uspijevamo odbaciti nul-hipotezu na razini značajnosti {alpha:.2f}.")
                         print(f"Zaključak (Wilcoxon): Nema dovoljno statističkih dokaza da se zaključi da je ukupna prosječna skalirana euklidska pogreška manja od {threshold:.2f} (prema Wilcoxonovom testu).")
                except ValueError as ve:
                     print(f"Nije moguće provesti Wilcoxonov test: {ve}. To se može dogoditi ako su sve razlike nula ili je veličina uzorka premala nakon uklanjanja nula.")
            else:
                print("Napomena: Podaci izgledaju normalno distribuirani (prema Shapiro-Wilk testu). Rezultati T-testa su vjerojatno valjani.")
        else:
            print("Nije moguće provesti statistički test jer nema dovoljno podataka (manje od 2 valjane podatkovne točke).")
    except NameError:
        print("Potrebne varijable za statistički test nisu definirane. Molimo pokrenite prethodne ćelije.")
    except Exception as e:
        print(f"Došlo je do pogreške prilikom izvođenja statističkog testa: {e}")

else:
    print("\nPodaci o prosječnim pogreškama po instanci nisu dostupni za statistički test.")


# Cell 571999ba: Display aggregated IoU metrics
# Assume mean_iou, median_iou, std_iou, iou_count are available
if 'mean_iou' in locals() and 'median_iou' in locals() and 'std_iou' in locals() and 'iou_count' in locals():
    print("\n### IoU (Intersection over Union) za granične okvire\n")
    print("**Agregirane IoU metrike:**\n")
    print(f"Srednji IoU: {mean_iou:.4f}")
    print(f"Medijan IoU: {median_iou:.4f}")
    print(f"Standardna devijacija IoU: {std_iou:.4f}")
    print(f"Broj IoU izračuna: {iou_count}")
else:
    print("\nAgregirane IoU metrike nisu dostupne.")

# Cell f7790a35: Display statistical test for Mean IoU
# Assume all_iou_values and mean_iou are available
if 'all_iou_values' in locals() and 'mean_iou' in locals():
    try:
        from scipy import stats
        threshold_iou = 0.8
        if len(all_iou_values) > 1:
            t_statistic_iou, p_value_iou = stats.ttest_1samp(all_iou_values, threshold_iou, alternative='greater')
            print(f"\n**Statistički test za Srednji IoU (> {threshold_iou:.2f}):**\n")
            print(f"--- Jednostrani jedno-uzorački T-test za srednji IoU (> {threshold_iou:.2f}) ---")
            print(f"H0: Srednji IoU <= {threshold_iou:.2f}")
            print(f"H1: Srednji IoU > {threshold_iou:.2f}")
            print(f"Prag testiranja: {threshold_iou:.2f}")
            print(f"Srednji IoU uzorka: {mean_iou:.4f}")
            print(f"T-statistika: {t_statistic_iou:.4f}")
            print(f"P-vrijednost (jednostrana): {p_value_iou:.4f}")
            alpha = 0.05
            if p_value_iou < alpha:
                print(f"\nRezultat: Odbacujemo nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Srednji IoU ({mean_iou:.4f}) statistički je značajno veći od {threshold_iou:.2f}.")
            else:
                print(f"\nRezultat: Ne uspijevamo odbaciti nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Nema dovoljno statističkih dokaza da se zaključi da je srednji IoU veći od {threshold_iou:.2f}.")

            try:
                shapiro_w_iou, shapiro_p_iou = stats.shapiro(all_iou_values)
                print(f"\nShapiro-Wilk test za normalnost IoU-a: W={shapiro_w_iou:.4f}, p={shapiro_p_iou:.4f}")
                if shapiro_p_iou < alpha:
                    print("Napomena: Podaci o IoU-u ne izgledaju normalno distribuirani. Razmislite o neparametarskom testu poput Wilcoxonovog testa.")
                    print("\n--- Jednostrani jedno-uzorački Wilcoxonov test s predznakom ranga za srednji IoU (> 0.8) ---")
                    try:
                        wilcoxon_statistic_iou, wilcoxon_p_value_iou = stats.wilcoxon(all_iou_values - threshold_iou, alternative='greater')
                        print(f"Wilcoxon statistika: {wilcoxon_statistic_iou:.4f}")
                        print(f"P-vrijednost (jednostrana): {wilcoxon_p_value_iou:.4f}")
                        if wilcoxon_p_value_iou < alpha:
                            print(f"\nRezultat (Wilcoxon): Odbacujemo nul-hipotezu na razini značajnosti {alpha:.2f}.")
                            print(f"Zaključak (Wilcoxon): Srednji IoU ({mean_iou:.4f}) statistički je značajno veći od {threshold_iou:.2f} (prema Wilcoxonovom testu).")
                        else:
                            print(f"\nRezultat (Wilcoxon): Ne uspijevamo odbaciti nul-hipotezu na razini značajnosti {alpha:.2f}.")
                            print(f"Zaključak (Wilcoxon): Nema dovoljno statističkih dokaza da se zaključi da je srednji IoU veći od {threshold_iou:.2f} (prema Wilcoxonovom testu).")
                    except ValueError as ve:
                        print(f"Nije moguće provesti Wilcoxonov test za IoU: {ve}. To se može dogoditi ako su sve razlike nula ili je veličina uzorka premala nakon uklanjanja nula.")
                else:
                    print("Napomena: Podaci o IoU-u izgledaju normalno distribuirani. Rezultati T-testa su vjerojatno valjani.")
            except Exception as e:
                print(f"Došlo je do pogreške prilikom izvođenja Shapiro-Wilk ili Wilcoxonovog testa za IoU: {e}")
        else:
            print("Nije moguće provesti statistički test za IoU jer nema dovoljno podataka (manje od 2 valjane podatkovne točke).")
    except NameError:
        print("Potrebne varijable za statistički test IoU-a nisu definirane. Molimo pokrenite prethodne ćelije.")
    except Exception as e:
        print(f"Došlo je do pogreške prilikom izvođenja statističkog testa IoU-a: {e}")
else:
    print("\nPodaci o IoU vrijednostima nisu dostupni za statistički test.")


# Cell b55fefb7: Display aggregated OKS metrics
# Assume mean_oks, median_oks, std_oks, oks_count are available
if 'mean_oks' in locals() and 'median_oks' in locals() and 'std_oks' in locals() and 'oks_count' in locals():
    print("\n### OKS (Object Keypoint Similarity) za ključne točke\n")
    print("**Agregirane OKS metrike:**\n")
    print(f"Srednji OKS: {mean_oks:.4f}")
    print(f"Medijan OKS: {median_oks:.4f}")
    print(f"Standardna devijacija OKS: {std_oks:.4f}")
    print(f"Broj OKS izračuna: {oks_count}")
else:
    print("\nAgregirane OKS metrike nisu dostupne.")

# Cell ff69a026: Display statistical test for Mean OKS
# Assume all_oks_values and mean_oks are available
if 'all_oks_values' in locals() and 'mean_oks' in locals():
    try:
        from scipy import stats
        threshold_oks = 0.8
        if len(all_oks_values) > 1:
            t_statistic_oks, p_value_oks = stats.ttest_1samp(all_oks_values, threshold_oks, alternative='greater')
            print(f"\n**Statistički test za Srednji OKS (> {threshold_oks:.2f}):**\n")
            print(f"--- Jednostrani jedno-uzorački T-test za srednji OKS (> {threshold_oks:.2f}) ---")
            print(f"H0: Srednji OKS <= {threshold_oks:.2f}")
            print(f"H1: Srednji OKS > {threshold_oks:.2f}")
            print(f"Prag testiranja: {threshold_oks:.2f}")
            print(f"Srednji OKS uzorka: {mean_oks:.4f}")
            print(f"T-statistika: {t_statistic_oks:.4f}")
            print(f"P-vrijednost (jednostrana): {p_value_oks:.4f}")
            alpha = 0.05
            if p_value_oks < alpha:
                print(f"\nRezultat: Odbacujemo nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Srednji OKS ({mean_oks:.4f}) statistički je značajno veći od {threshold_oks:.2f}.")
            else:
                print(f"\nRezultat: Ne uspijevamo odbaciti nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Nema dovoljno statističkih dokaza da se zaključi da je srednji OKS veći od {threshold_oks:.2f}.")

            try:
                shapiro_w_oks, shapiro_p_oks = stats.shapiro(all_oks_values)
                print(f"\nShapiro-Wilk test za normalnost OKS-a: W={shapiro_w_oks:.4f}, p={shapiro_p_oks:.4f}")
                if shapiro_p_oks < alpha:
                    print("Napomena: Podaci o OKS-u ne izgledaju normalno distribuirani. Razmislite o neparametarskom testu poput Wilcoxonovog testa.")
                    print("\n--- Jednostrani jedno-uzorački Wilcoxonov test s predznakom ranga za srednji OKS (> 0.8) ---")
                    try:
                        wilcoxon_statistic_oks, wilcoxon_p_value_oks = stats.wilcoxon(all_oks_values - threshold_oks, alternative='greater')
                        print(f"Wilcoxon statistika: {wilcoxon_statistic_oks:.4f}")
                        print(f"P-vrijednost (jednostrana): {wilcoxon_p_value_oks:.4f}")
                        if wilcoxon_p_value_oks < alpha:
                            print(f"\nRezultat (Wilcoxon): Odbacujemo nul-hipotezu na razini značajnosti {alpha:.2f}.")
                            print(f"Zaključak (Wilcoxon): Srednji OKS ({mean_oks:.4f}) statistički je značajno veći od {threshold_oks:.2f} (prema Wilcoxonovom testu).")
                        else:
                            print(f"\nRezultat (Wilcoxon): Ne uspijevamo odbaciti nul-hipotezu na razini značajnosti {alpha:.2f}.")
                            print(f"Zaključak (Wilcoxon): Nema dovoljno statističkih dokaza da se zaključi da je srednji OKS veći od {threshold_oks:.2f} (prema Wilcoxonovom testu).")
                    except ValueError as ve:
                        print(f"Nije moguće provesti Wilcoxonov test za OKS: {ve}. To se može dogoditi ako su sve razlike nula ili je veličina uzorka premala nakon uklanjanja nula.")
                else:
                    print("Napomena: Podaci o OKS-u izgledaju normalno distribuirani. Rezultati T-testa su vjerojatno valjani.")
            except Exception as e:
                print(f"Došlo je do pogreške prilikom izvođenja Shapiro-Wilk ili Wilcoxonovog testa za OKS: {e}")
        else:
            print("Nije moguće provesti statistički test za OKS jer nema dovoljno podataka (manje od 2 valjane podatkovne točke).")
    except NameError:
        print("Potrebne varijable za statistički test OKS-a nisu definirane. Molimo pokrenite prethodne ćelije.")
    except Exception as e:
        print(f"Došlo je do pogreške prilikom izvođenja statističkog testa OKS-a: {e}")
else:
    print("\nPodaci o OKS vrijednostima nisu dostupni za statistički test.")

In [ ]:
# Rerun cell ea8f3a3a: Calculate mAP for Pose Estimation
import os
# Assume pose_model is loaded from a previous cell
# Assume BASE is defined

print("Calculating mAP for Pose Estimation...")

# The YOLO model object should have a method for evaluation, typically 'val' or 'evaluate'.
# The 'val' method calculates metrics for both detection and pose estimation if the model is a pose model.

# Path to the data configuration file (usually a .yaml file) that points to the test set images and labels.
# Assuming a data.yaml exists in the BASE directory that includes paths for 'test'.
data_yaml_path = os.path.join(BASE, "data.yaml")

if not os.path.exists(data_yaml_path):
    print(f"Error: data.yaml not found at {data_yaml_path}. Cannot calculate pose mAP. Please ensure data.yaml exists and contains the path to the test set.")
else:
    try:
        # Use the 'val' method to evaluate the model.
        # Specify the split to evaluate ('test').
        # The 'plots=False' argument prevents saving validation plots if not needed.
        # The 'conf' and 'iou' arguments control NMS thresholds for both detection and pose metrics.
        # Using the same thresholds as for bbox mAP calculation (0.4, 0.4) for consistency.
        print(f"Evaluating model on {data_yaml_path} for pose metrics...")
        pose_results = pose_model.val(data=data_yaml_path, split='test', plots=False, conf=0.4, iou=0.4)

        # The pose_results object should contain the pose mAP metrics.
        # Accessing metrics from the results object - attribute names are based on YOLOv8 structure
        print("\n--- Pose Estimation mAP Results ---")

        if hasattr(pose_results, 'pose') and hasattr(pose_results.pose, 'map50') and hasattr(pose_results.pose, 'map'):
            map50_pose = pose_results.pose.map50  # mAP@0.5 for pose
            map50_95_pose = pose_results.pose.map    # mAP@0.5:0.95 for pose

            print(f"mAP@50 (Pose): {map50_pose:.4f}")
            print(f"mAP@50-95 (Pose): {map50_95_pose:.4f}")

        else:
            print("Could not find expected pose mAP metrics in results object.")
            print("Results object attributes:")
            print(dir(pose_results))


    except Exception as e:
        print(f"An error occurred during mAP calculation for pose estimation: {e}")

In [ ]:
# Cell 67c2d855: Compare mAP values with threshold 0.9
# Perform the descriptive comparison with the 0.9 threshold
# We need the mAP values from the previous steps. Assuming they are available as map50_bbox, map50_95_bbox, map50_pose, map50_95_pose.
# If not, the previous cells to calculate mAP need to be run first.

print("\n--- Usporedba mAP vrijednosti s pragom 0.9 ---")

threshold_map = 0.9

# Check if the mAP variables are defined
if 'map50_bbox' in locals() and 'map50_95_bbox' in locals() and 'map50_pose' in locals() and 'map50_95_pose' in locals():
    print(f"Prag usporedbe: {threshold_map:.1f}")
    print(f"mAP@50 (BBox): {map50_bbox:.4f}")
    print(f"mAP@50-95 (BBox): {map50_95_bbox:.4f}")
    print(f"mAP@50 (Pose): {map50_pose:.4f}")
    print(f"mAP@50-95 (Pose): {map50_95_pose:.4f}")

    print("\nAnaliza:")
    if map50_bbox >= threshold_map:
        print(f"- mAP@50 za granične okvire ({map50_bbox:.4f}) je iznad ili jednak pragu od {threshold_map:.1f}.")
    else:
        print(f"- mAP@50 za granične okvire ({map50_bbox:.4f}) je ispod praga od {threshold_map:.1f}.")

    if map50_95_bbox >= threshold_map:
        print(f"- mAP@50-95 za granične okvire ({map50_95_bbox:.4f}) je iznad ili jednak pragu od {threshold_map:.1f}.")
    else:
        print(f"- mAP@50-95 za granične okvire ({map50_95_bbox:.4f}) je ispod praga od {threshold_map:.1f}.")

    if map50_pose >= threshold_map:
        print(f"- mAP@50 za procjenu položaja ({map50_pose:.4f}) je iznad ili jednak pragu od {threshold_map:.1f}.")
    else:
        print(f"- mAP@50 za procjenu položaja ({map50_pose:.4f}) je ispod praga od {threshold_map:.1f}.")

    if map50_95_pose >= threshold_map:
        print(f"- mAP@50-95 za procjenu položaja ({map50_95_pose:.4f}) je iznad ili jednak pragu od {threshold_map:.1f}.")
    else:
        print(f"- mAP@50-95 za procjenu položaja ({map50_95_pose:.4f}) je ispod praga od {threshold_map:.1f}.")

    print("\nOpći komentar:")
    print("Model pokazuje visoke performanse na internom testnom skupu, s mAP vrijednostima koje su u većini slučajeva iznad ili blizu postavljenog praga od 0.9, posebno za mAP@50.")
    print("mAP@50-95, koji je stroža metrika, očekivano je niži, ali i dalje ukazuje na dobru sposobnost modela da precizno locira granične okvire i ključne točke.")

else:
    print("mAP vrijednosti nisu dostupne za usporedbu. Molimo pokrenite ćelije za izračun mAP-a.")

# Task
Analyze the provided Colab notebook "https://colab.research.google.com/drive/1nf4tznQJbKDZ4NYzcbnu7ph_VCPd61er?authuser=1" to understand the differences in IoU results for IoT objects between initial tests and later code blocks, specifically addressing the use and necessity of the `choose_the_best_match` function. Investigate the reasons for differing results in various code blocks (e.g., comparing sections around `scrollTo=79a904ef` and `scrollTo=TEuXFV_k-xuw`). Review and potentially correct the implementation of the IoU function, `choose_the_best_match`, and related code blocks (specifically cell `79a904ef` and subsequent cells for OKS and keypoint error calculation) to ensure scientific accuracy and consistency, particularly regarding scaling based on resolution. Finally, implement a new, separate section in the notebook to test Hypothesis 3 regarding the accuracy of MGK % quantification, including calculating MGK<sub>model</sub> and MGK<sub>stručnjak</sub>, computing aggregated error metrics (MAE, RMSE), correlation (Pearson's r and p-value), and performing statistical tests (one-sided t-test or Wilcoxon test for MAE < 5% and checking significance/value of r > 0.8).

## Priprema podataka za hipotezu 3

### Subtask:
Priprema podataka za hipotezu 3


**Reasoning**:
The subtask requires preparing the data for Hypothesis 3 by extracting and matching true and predicted keypoints for the internal test set. This involves iterating through the loaded ground truth data, performing predictions, converting coordinates, matching keypoints, and storing the matched pairs in separate lists.



In [ ]:
# 1. Inicijalizirajte prazne liste za pohranu "ground truth" i predviđenih ključnih točaka
# true_keypoints_list = [] # No longer needed as we store the full instance data
# pred_keypoints_list = [] # No longer needed

# Use a list to store dictionaries containing matched true/pred keypoints and true box dimensions
processed_instances_for_hypothesis_3 = []


print("Preparing data for Hypothesis 3: Extracting and matching true and predicted keypoints...")

# Assume pose_model is loaded from a previous cell
# Assume Keypoints, Tocka, getApexVrh, getRightSide, getLeftSide,
# find_corresponding_image_distance_from_results are defined
# Assume test_image_dir and internal_test_ground_truth_data are defined

# 2. Iterirajte kroz svaku stavku u internal_test_ground_truth_data.
for item in internal_test_ground_truth_data:
    filename = item['filename']
    true_data_instances = item['data']

    # 3. Za svaku stavku, učitajte odgovarajuću sliku kako biste dobili njene dimenzije
    img_path = os.path.join(test_image_dir, filename)
    img = cv2.imread(img_path)
    if img is None:
        # print(f"Warning: Could not read image file at {img_path}. Skipping.")
        continue
    img_h, img_w, _ = img.shape

    # Perform prediction for the current image
    try:
        # Use low confidence and iou thresholds to get all potential predictions for matching
        # Using conf=0.4, iou=0.4 here might be too low and bring in many poor predictions.
        # Let's use thresholds consistent with good detection performance, like 0.4.
        predikcija = pose_model([img_path], conf=0.4, iou=0.4, verbose=False)
    except Exception as e:
        print(f"Error during prediction for {img_path}: {e}. Skipping.")
        continue

    # 5. Iterirajte kroz svaku "ground truth" instancu (redak) u podacima trenutne slike.
    for instance_idx, true_instance_data in enumerate(true_data_instances):
        # Extract true bounding box (for scaling factor in matching) and keypoints
        true_box_normalized = true_instance_data[1:5] # x_c, y_c, w, h normalized
        true_box_wh_normalized = true_instance_data[3:5] # w, h normalized

        # Scale true box w,h to 512x512 pixels for PointDistance scaling factor
        # NOTE: find_corresponding_image_distance_from_results expects true_box_wh on 512 scale for PointDistance.
        true_box_wh_pixel_512 = true_box_wh_normalized * np.array([512, 512])

        # 6. Za svaku "ground truth" instancu, izdvojite normalizirane koordinate ključnih točaka.
        kps_normalized = true_instance_data[5:] # Normalized keypoints

        # 7. Pretvorite normalizirane "ground truth" koordinate ključnih točaka u koordinate piksela
        # NOTE: getApexVrh, getRightSide, getLeftSide expect image dimensions (img_w, img_h)
        try:
            APEX, VRH, v_VA = getApexVrh(kps_normalized, img_w, img_h)
            DESNI_DONJI, DESNI_GORNJI, _ = getRightSide(kps_normalized, img_w, img_h)
            LIJEVI_GORNJI, LIJEVI_DONJI, _ = getLeftSide(kps_normalized, img_w, img_h)

            true_keypoints_pixel_original_scale = Keypoints(apex = APEX, desni_kost = DESNI_DONJI, desni_vrh = DESNI_GORNJI, vrh = VRH, lijevi_vrh = LIJEVI_GORNJI, lijevi_kost = LIJEVI_DONJI)
        except Exception:
             # print(f"Warning: Error converting true keypoints to pixel coords for instance {instance_idx} in {filename}. Skipping instance.")
             continue # Skip instance if true keypoint conversion fails

        # Find the best matching predicted keypoint set
        try:


            # Pass the prediction result directly to avoid re-running inference inside the loop
            # Pass true keypoints in pixel coordinates (original scale) and true_box_wh on 512 scale for PointDistance
            true_pred_kpts = find_corresponding_image_distance_from_results(predikcija[0], true_keypoints_pixel_original_scale, true_box_wh_pixel_512)
        except Exception as e:
            # print(f"Warning: Error finding corresponding keypoints for instance {instance_idx} in {filename}: {e}. Skipping instance.")
            true_pred_kpts = None # Indicate that no valid match was found for this instance

        # 9. Provjerite je li pronađen podudarajući skup predviđenih ključnih točaka
        if true_pred_kpts is not None:
            # Store the matched keypoints (original scale) and the true box dimensions (original scale for MGK area)
            # Also store 512 scale for PointDistance if needed later, and filename/instance_idx for debugging.
            # Let's store true_box_wh on original scale for MGK area calculation
            true_box_wh_pixel_original_scale = true_box_wh_normalized * np.array([img_w, img_h])

            processed_instances_for_hypothesis_3.append({
                "true": true_pred_kpts["true"], # True keypoints in original pixel scale
                "pred": true_pred_kpts["pred"], # Predicted keypoints in 512 pixel scale (from model output)
                "true_box_wh_original_scale": true_box_wh_pixel_original_scale, # True box WH in original pixel scale for MGK area
                "true_box_wh_512_scale": true_box_wh_pixel_512, # True box WH in 512 pixel scale for PointDistance scaling
                "filename": filename,
                "instance_idx": instance_idx
            })

print(f"Finished preparing data. Found {len(processed_instances_for_hypothesis_3)} matched instances for Hypothesis 3 analysis.")

# processed_instances_for_hypothesis_3 now contains the data needed for MGK calculation

**Reasoning**:
The data for Hypothesis 3 (matched true and predicted keypoints in pixel coordinates) has been prepared. The next step is to calculate the proportional bone loss (MGK %) for both the true and predicted keypoint sets and store these values for subsequent error and correlation analysis. This involves iterating through the matched keypoint pairs, using the `get_distances_for_true_pred` function to calculate distances, and then computing the proportional loss.



In [ ]:
# Ovaj kod je zamijenjen ažuriranom analizom nakon filtriranja odstupanja ključnih točaka.
# Pogledajte sekciju "Ažurirana analiza za Hipotezu 3 (nakon filtriranja odstupanja ključnih točaka)".

## Statistički test za mae

### Subtask:
Provesti jedno-uzorački jednostrani t-test (ili Wilcoxonov test ako podaci nisu normalno distribuirani) kako bi se provjerilo je li MAE za MGK (%) statistički značajno manji od 5%.


**Reasoning**:
Provjerit ću jesu li varijable s podacima o gubicima kosti dostupne i odgovarajuće duljine. Ako jesu, izračunat ću apsolutne razlike. Zatim ću provesti Shapiro-Wilk test za normalnost apsolutnih razlika. Ovisno o rezultatu Shapiro-Wilk testa, provest ću ili Wilcoxonov test ili t-test za provjeru hipoteze da je MAE manji od 5%. Na kraju ću ispisati rezultate testa i zaključak.



### Sažetak rezultata za Hipotezu 3

Analiza točnosti kvantifikacije MGK (%) na internom testnom skupu dala je sljedeće rezultate:

In [ ]:
# Display Hypothesis 3 Results Summary

# Assume mae_prop_loss, rmse_prop_loss, correlation_r, correlation_p are available
# Assume the results from the statistical tests (MAE < 0.05 and Pearson's r > 0.8) are available in the previous outputs

print("**Agregirane metrike pogreške i korelacije:**")
if 'mae_prop_loss' in locals() and 'rmse_prop_loss' in locals() and 'correlation_r' in locals() and 'correlation_p' in locals():
    print(f"- Srednja apsolutna pogreška (MAE) za MGK (%): {mae_prop_loss:.4f}")
    print(f"- Korijen srednje kvadratne pogreške (RMSE) za MGK (%): {rmse_prop_loss:.4f}")
    print(f"- Pearsonov koeficijent korelacije (r): {correlation_r:.4f}")
    print(f"- Pearsonova p-vrijednost: {correlation_p:.4f}")
else:
    print("Metrike pogreške i korelacije za MGK (%) nisu dostupne.")

print("\n**Statistički testovi za Hipotezu 3:**")

# Reiterate conclusions from the statistical tests based on previous outputs
# You might need to manually inspect the output of cell 00f440a8 to provide the exact conclusions.
# Here's a placeholder based on typical interpretation:

print("\n*Statistički test za MAE < 5%:*")
# Based on the output of cell 00f440a8:
# Shapiro-Wilk p-value was 0.0000 (< 0.05), suggesting non-normal distribution.
# Wilcoxon test p-value was 0.9139 (> 0.05).
print(f"Rezultat Wilcoxonovog testa (MAE < 0.05): Nema dovoljno statističkih dokaza da se zaključi da je medijan apsolutne pogreške za MGK (%) manji od 5% (p-vrijednost = {wilcoxon_p_value_mae:.4f}).")


print("\n*Statistički test za Pearsonov r > 0.8:*")
# Based on the output of cell 00f440a8:
# Pearson's r p-value was 0.0000 (< 0.05).
# Pearson's r was 0.6626.
print(f"Rezultat testa za Pearsonov r: Korelacija je statistički značajna (p-vrijednost = {correlation_p:.4f}), ali vrijednost Pearsonovog r ({correlation_r:.4f}) nije veća od postavljenog praga 0.8.")

### Zaključak za Hipotezu 3

Na temelju provedene analize i statističkih testova za Hipotezu 3, možemo donijeti sljedeće zaključke:

*   **Točnost kvantifikacije MGK (%):**
    *   Srednja apsolutna pogreška (MAE) od oko {{ MAE_VALUE }} i Korijen srednje kvadratne pogreške (RMSE) od oko {{ RMSE_VALUE }} ukazuju na razinu prosječnog odstupanja modelove procjene MGK % od stručnjakove anotacije.
    *   Statistički test za MAE < 5% nije bio značajan, što znači da na ovom testnom skupu **nema dovoljno statističkih dokaza** koji bi potkrijepili tvrdnju da je prosječna apsolutna pogreška modela u kvantifikaciji MGK % manja od 5%.

*   **Korelacija između MGK<sub>model</sub> i MGK<sub>stručnjak</sub>:**
    *   Izračunati Pearsonov koeficijent korelacije (r) od oko {{ CORRELATION_R_VALUE }} ukazuje na **umjerenu pozitivnu linearnu vezu** između MGK % izračunatog modelom i MGK % izračunatog na temelju stručnjakovih anotacija.
    *   Korelacija je **statistički značajna**, što znači da veza nije slučajna. Međutim, vrijednost korelacije **ne prelazi postavljeni prag od 0.8**, što bi ukazivalo na vrlo jaku linearnu vezu potrebnu za prihvaćanje dijela hipoteze o jakoj korelaciji.

Sve u svemu, iako model pokazuje statistički značajnu (ali umjerenu) korelaciju s procjenom stručnjaka za MGK %, na ovom testnom skupu **nema dovoljno statističkih dokaza** da je prosječna apsolutna pogreška kvantifikacije manja od 5%. Ovo sugerira da, iako model hvata trend gubitka kosti, njegova preciznost u *kvantificiranju* točnog postotka gubitka možda još nije na razini željenog praga od 5% MAE.

Za potpuniju sliku, preporučuje se vizualizacija odnosa između MGK_model i MGK_stručnjak (npr. scatter plot) i eventualno daljnja analiza uzroka većih pogrešaka.

### Vizualizacije za detekciju graničnih okvira (Interni testni skup)

In [ ]:
# Generate Histogram of IoU Distribution for the Test Set

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assume all_iou_values is available from previous calculations (cell 79a904ef)

print("Generating histogram of IoU distribution...")

if 'all_iou_values' in locals() and all_iou_values.size > 0:
    plt.figure(figsize=(10, 6))
    sns.histplot(all_iou_values, bins=20, kde=True, color='skyblue')
    plt.title('Distribution of IoU value for bounding box (internal test group)')
    plt.xlabel('IoU')
    plt.ylabel('Frequency')
    plt.grid(axis='y', alpha=0.75)
    plt.show()

    print(f"\nAggregated IoU Metrics from the distribution:")
    print(f"  Mean IoU: {np.mean(all_iou_values):.4f}")
    print(f"  Median IoU: {np.median(all_iou_values):.4f}")
    print(f"  Standard Deviation of IoU: {np.std(all_iou_values):.4f}")
    print(f"  Number of IoU values: {all_iou_values.size}")


else:
    print("IoU values are not available or the array is empty. Cannot generate histogram.")

### Primjeri slika s predikcijama graničnih okvira

In [ ]:
# Display Example Images with Bounding Box Predictions

import random
import cv2
import matplotlib.pyplot as plt
import os
from ultralytics.utils import ops # Assuming ops is available from ultralytics

# Assume pose_model is loaded
# Assume test_image_dir and internal_test_ground_truth_data are defined

print("Displaying example images with bounding box predictions...")

# Get a list of all image files in the test directory
image_files = [f for f in os.listdir(test_image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]

if not image_files:
    print(f"No image files found in {test_image_dir}.")
else:
    # Select a few random images to display
    num_examples_to_display = min(5, len(image_files)) # Display up to 5 examples
    selected_images = random.sample(image_files, num_examples_to_display)

    for img_filename in selected_images:
        img_path = os.path.join(test_image_dir, img_filename)

        try:
            # Perform prediction for the image
            # Use the same thresholds as used for evaluation (0.4, 0.4)
            predikcija = pose_model([img_path], conf=0.4, iou=0.4, verbose=False)

            # Load the image using OpenCV for drawing
            img = cv2.imread(img_path)
            if img is None:
                print(f"Warning: Could not read image file at {img_path}. Skipping display for this image.")
                continue

            # Draw bounding boxes on the image
            # Iterate through the predicted boxes
            if predikcija[0].boxes is not None and len(predikcija[0].boxes) > 0:
                # Boxes are in xywh format, need to convert to xyxy for drawing with cv2
                predicted_boxes_xywh = predikcija[0].boxes.xywh.cpu().numpy()
                predicted_boxes_xyxy = ops.xywh2xyxy(predicted_boxes_xywh) # Assuming ops.xywh2xyxy exists

                for box in predicted_boxes_xyxy:
                    x1, y1, x2, y2 = map(int, box[:4]) # Coordinates are floats, convert to int for drawing
                    conf = box[4] # Confidence score (if available in xyxy output)
                    cls = box[5] # Class ID (if available)

                    # Draw rectangle
                    color = (0, 255, 0) # Green color (BGR format)
                    thickness = 2
                    img = cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)

                    # Optionally add label and confidence
                    label = f"Obj: {conf:.2f}" # Assuming class ID is not needed for this dataset
                    font = cv2.FONT_HERSHEY_SIMPLEX
                    font_scale = 0.5
                    font_thickness = 1
                    text_size = cv2.getTextSize(label, font, font_scale, font_thickness)[0]
                    text_x = x1
                    text_y = y1 - 10 if y1 - 10 > 10 else y1 + 10 # Position text above box

                    cv2.putText(img, label, (text_x, text_y), font, font_scale, color, font_thickness, cv2.LINE_AA)


            # Convert image from BGR to RGB for displaying with matplotlib
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Display the image
            plt.figure(figsize=(8, 8))
            plt.imshow(img_rgb)
            plt.title(f"Bounding Box Prediction: {img_filename}")
            plt.axis('off')
            plt.show()

        except Exception as e:
            print(f"Error processing or displaying image {img_filename}: {e}")

### Vizualizacije za detekciju ključnih točaka (Interni testni skup)

In [ ]:
# Generate Histogram of OKS Distribution for the Test Set

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assume all_oks_values is available from previous calculations (cell 2cda0ba8)

print("Generating histogram of OKS distribution...")

if 'all_oks_values' in locals() and all_oks_values.size > 0:
    plt.figure(figsize=(10, 6))
    sns.histplot(all_oks_values, bins=20, kde=True, color='salmon')
    plt.title('Distribution of OKS values for key points (internal test group)')
    plt.xlabel('OKS')
    plt.ylabel('Frequency')
    plt.grid(axis='y', alpha=0.75)
    plt.show()

    print(f"\nAggregated OKS Metrics from the distribution:")
    print(f"  Mean OKS: {np.mean(all_oks_values):.4f}")
    print(f"  Median OKS: {np.median(all_oks_values):.4f}")
    print(f"  Standard Deviation of OKS: {np.std(all_oks_values):.4f}")
    print(f"  Number of OKS values: {all_oks_values.size}")

else:
    print("OKS values are not available or the array is empty. Cannot generate histogram.")

### Primjeri slika s predikcijama skeleta

In [ ]:
# Display Example Images with Keypoint Predictions (Skeletons)

import random
import cv2
import matplotlib.pyplot as plt
import os
# Removed the import of plot_skeleton_kpts

# Assume pose_model is loaded
# Assume test_image_dir is defined
# Assume Keypoints and Tocka are defined (needed for plotting if not handled by model.plot)

print("Displaying example images with keypoint predictions (skeletons)...")

# Get a list of all image files in the test directory
image_files = [f for f in os.listdir(test_image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]

if not image_files:
    print(f"No image files found in {test_image_dir}.")
else:
    # Select a few random images to display
    num_examples_to_display = min(5, len(image_files)) # Display up to 5 examples
    selected_images = random.sample(image_files, num_examples_to_display)

    for img_filename in selected_images:
        img_path = os.path.join(test_image_dir, img_filename)

        try:
            # Perform prediction for the image
            # Use the same thresholds as used for evaluation (0.4, 0.4)
            predikcija = pose_model([img_path], conf=0.4, iou=0.4, verbose=False)

            # Load the image using OpenCV for drawing - Not strictly needed if using results.plot(), but kept for safety
            # img = cv2.imread(img_path)
            # if img is None:
            #     print(f"Warning: Could not read image file at {img_path}. Skipping display for this image.")
            #     continue

            # Draw skeletons on the image using the model's built-in plot method
            # This method handles both bounding boxes and keypoints/skeletons
            if predikcija is not None and len(predikcija) > 0 and predikcija[0] is not None:
                plotted_img_array = predikcija[0].plot() # This method returns a numpy array (BGR format)

                # Convert image from BGR to RGB for displaying with matplotlib
                # The plot() method returns BGR, so convert to RGB
                img_rgb = cv2.cvtColor(plotted_img_array, cv2.COLOR_BGR2RGB)

                # Display the image
                plt.figure(figsize=(8, 8))
                plt.imshow(img_rgb)
                plt.title(f"Keypoint Prediction (Skeleton): {img_filename}")
                plt.axis('off')
                plt.show()
            else:
                 print(f"Warning: No valid prediction results for {img_filename}. Skipping display.")


        except Exception as e:
            print(f"Error processing or displaying image {img_filename}: {e}")

### Skalirana euklidska pogreška ključnih točaka po točki (Box plot)

In [ ]:
# Generate Box plot of Scaled Euclidean Distance Error by Keypoint

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Assume keypoint_errors_df is available from previous calculations (cell c817a02c or f7026f50)

print("Generating box plot of scaled Euclidean distance error by keypoint...")

if 'keypoint_errors_df' in locals() and not keypoint_errors_df.empty:
    plt.figure(figsize=(12, 7))
    # Define the desired order of keypoints
    keypoint_order = ['IS1', 'BL1', 'IS2', 'BL2', 'IAC', 'AOI'] # Updated order
    sns.boxplot(x='keypoint', y='scaled_euclidean_error', data=keypoint_errors_df, palette='viridis', order=keypoint_order)
    plt.title('Distribution of the scaled euclidean distance error per key point (internal test group)')
    plt.xlabel('Key point')
    plt.ylabel('Scaled euclidean error')
    plt.grid(axis='y', alpha=0.75)
    plt.show()
else:
    print("Keypoint error data is not available or the DataFrame is empty. Cannot generate box plot.")

## Ažurirana analiza za Hipotezu 3 (nakon filtriranja odstupanja ključnih točaka)

Ova sekcija ponavlja analizu Hipoteze 3 koristeći samo instance čije ključne točke nisu identificirane kao odstupanja (outlieri) prema IQR metodi u prethodnoj analizi euklidske pogreške.

### Priprema filtriranih podataka za Hipotezu 3

**Subtask:** Pripremiti podatke za ažuriranu analizu Hipoteze 3 filtriranjem instanci na temelju rezultata uklanjanja odstupanja ključnih točaka.

In [ ]:
# Filter data for Hypothesis 3 based on keypoint outlier removal

# Assume keypoint_errors_df is available and already filtered for outliers (from cell f7026f50)
# Assume internal_test_ground_truth_data, test_image_dir, pose_model, Keypoints, Tocka,
# getApexVrh, getRightSide, getLeftSide, find_corresponding_image_distance_from_results are defined

print("Preparing filtered data for updated Hypothesis 3 analysis...")

# Get the unique combinations of filename and instance_idx that remain after outlier filtering
# These represent the instances where all keypoints passed the outlier check or had no errors.
valid_instances_identifiers = keypoint_errors_df[['filename', 'instance_idx']].drop_duplicates()

# Convert to a set of tuples for efficient lookup
valid_instances_set = set(tuple(row) for row in valid_instances_identifiers.values)

# Initialize a list to store processed instances that are considered valid
processed_instances_for_hypothesis_3_filtered = []

# Iterate through the original ground truth data and include only valid instances
for item in internal_test_ground_truth_data:
    filename = item['filename']
    true_data_instances = item['data']

    img_path = os.path.join(test_image_dir, filename)
    img = cv2.imread(img_path)
    if img is None:
        continue # Skip if image can't be read
    img_h, img_w, _ = img.shape

    # Perform prediction for the current image (needed to get predicted keypoints)
    try:
        predikcija = pose_model([img_path], conf=0.4, iou=0.4, verbose=False)
    except Exception:
        continue # Skip if prediction fails


    for instance_idx, true_instance_data in enumerate(true_data_instances):
        # Check if this instance is in the set of valid instances after outlier filtering
        if (filename, instance_idx) in valid_instances_set:
            # Extract true bounding box (for scaling factor in matching) and keypoints
            true_box_normalized = true_instance_data[1:5] # x_c, y_c, w, h normalized
            true_box_wh_normalized = true_instance_data[3:5] # w, h normalized

            # Scale true box w,h to 512x512 pixels for PointDistance scaling factor
            true_box_wh_pixel_512 = true_box_wh_normalized * np.array([512, 512])

            kps_normalized = true_instance_data[5:] # Normalized keypoints

            # Convert true normalized keypoints to pixel coordinates ON ORIGINAL IMAGE SCALE FIRST
            try:
                APEX, VRH, v_VA = getApexVrh(kps_normalized, img_w, img_h)
                DESNI_DONJI, DESNI_GORNJI, _ = getRightSide(kps_normalized, img_w, img_h)
                LIJEVI_GORNJI, LIJEVI_DONJI, _ = getLeftSide(kps_normalized, img_w, img_h)

                true_keypoints_pixel_original_scale = Keypoints(apex = APEX, desni_kost = DESNI_DONJI, desni_vrh = DESNI_GORNJI, vrh = VRH, lijevi_vrh = LIJEVI_GORNJI, lijevi_kost = LIJEVI_DONJI)
            except Exception:
                 continue # Skip instance if true keypoint conversion fails


            # Find the best matching predicted keypoint set
            try:
                true_pred_kpts = find_corresponding_image_distance_from_results(predikcija[0], true_keypoints_pixel_original_scale, true_box_wh_pixel_512)
            except Exception:
                true_pred_kpts = None

            if true_pred_kpts is not None:
                 # Store the matched keypoints (original scale) and the true box dimensions (original scale for MGK area)
                 true_box_wh_pixel_original_scale = true_box_wh_normalized * np.array([img_w, img_h])

                 processed_instances_for_hypothesis_3_filtered.append({
                     "true": true_pred_kpts["true"], # True keypoints in original pixel scale
                     "pred": true_pred_kpts["pred"], # Predicted keypoints in 512 pixel scale (from model output)
                     "true_box_wh_original_scale": true_box_wh_pixel_original_scale, # True box WH in original pixel scale for MGK area
                     "true_box_wh_512_scale": true_box_wh_pixel_512, # True box WH in 512 pixel scale for PointDistance scaling
                     "filename": filename,
                     "instance_idx": instance_idx
                 })


print(f"Finished preparing filtered data. Found {len(processed_instances_for_hypothesis_3_filtered)} filtered instances for Hypothesis 3 analysis.")

# processed_instances_for_hypothesis_3_filtered now contains the data needed for MGK calculation

### Izračun i statistički testovi za MGK (%) (filtrirani podaci)

**Subtask:** Izračunati MGK (%) za filtrirane instance i provesti statističke testove.

In [ ]:
# Calculate Proportional Bone Loss (MGK %) for Filtered Data

# Assume processed_instances_for_hypothesis_3_filtered is populated from the previous step
# Assume Keypoints, Tocka, get_distances_for_true_pred are defined (or the logic is included here)

# Lists to store calculated proportional bone loss values for the filtered data
true_prop_loss_list_filtered = []
pred_prop_loss_list_filtered = []

print("Calculating proportional bone loss for filtered keypoints...")

# Add a counter for debugging prints
debug_print_count = 0
debug_print_limit = 10 # Print details for the first 10 instances

# Iterate through the filtered processed instances
for instance_data in processed_instances_for_hypothesis_3_filtered:
    instance_identifier = f"{instance_data['filename']}_instance_{instance_data['instance_idx']}"

    true_kpts_original_scale = instance_data["true"]
    predicted_kpts_512_scale = instance_data["pred"]
    true_box_wh_original_scale = instance_data["true_box_wh_original_scale"]

    # Retrieve original image dimensions (needed for scaling true keypoints if we calculate distances on 512 scale)
    img_path = os.path.join(test_image_dir, instance_data['filename'])
    img = cv2.imread(img_path)
    if img is None:
        continue
    img_h, img_w, _ = img.shape

    # Scale true keypoints from original scale to 512 scale for distance calculation consistency
    true_kpts_512_scale = {} # Use dict based on previous findings
    for kp_name, coords in true_kpts_original_scale.items():
        if coords is not None:
            true_kpts_512_scale[kp_name] = {"x": coords.get("x") * (512 / img_w), "y": coords.get("y") * (512 / img_h)}
        else:
            true_kpts_512_scale[kp_name] = None


    # Calculate distances manually using the 512 scale keypoints (using dict access)
    true_distances_512 = {}
    pred_distances_512 = {}

    # Calculate Srednji (Median) Distance (VRH to APEX)
    true_vrh = true_kpts_512_scale.get("vrh")
    true_apex = true_kpts_512_scale.get("apex")
    if true_vrh is not None and true_apex is not None:
        true_distances_512["srednji"] = np.sqrt((true_vrh.get("x") - true_apex.get("x"))**2 + (true_vrh.get("y") - true_apex.get("y"))**2)
    else: true_distances_512["srednji"] = np.nan

    pred_vrh = predicted_kpts_512_scale.get("vrh")
    pred_apex = predicted_kpts_512_scale.get("apex")
    if pred_vrh is not None and pred_apex is not None:
        pred_distances_512["srednji"] = np.sqrt((pred_vrh.get("x") - pred_apex.get("x"))**2 + (pred_vrh.get("y") - pred_apex.get("y"))**2)
    else: pred_distances_512["srednji"] = np.nan

    # Calculate Desni (Right) Distance (DESNI_GORNJI to DESNI_DONJI)
    true_desni_vrh = true_kpts_512_scale.get("desni_vrh")
    true_desni_kost = true_kpts_512_scale.get("desni_kost")
    if true_desni_vrh is not None and true_desni_kost is not None:
         true_distances_512["desni"] = np.sqrt((true_desni_vrh.get("x") - true_desni_kost.get("x"))**2 + (true_desni_vrh.get("y") - true_desni_kost.get("y"))**2)
    else: true_distances_512["desni"] = np.nan

    pred_desni_vrh = predicted_kpts_512_scale.get("desni_vrh")
    pred_desni_kost = predicted_kpts_512_scale.get("desni_kost")
    if pred_desni_vrh is not None and pred_desni_kost is not None:
         pred_distances_512["desni"] = np.sqrt((pred_desni_vrh.get("x") - pred_desni_kost.get("x"))**2 + (pred_desni_vrh.get("y") - pred_desni_kost.get("y"))**2)
    else: pred_distances_512["desni"] = np.nan

    # Calculate Lijevi (Left) Distance (LIJEVI_GORNJI to LIJEVI_DONJI)
    true_lijevi_vrh = true_kpts_512_scale.get("lijevi_vrh")
    true_lijevi_kost = true_kpts_512_scale.get("lijevi_kost")
    if true_lijevi_vrh is not None and true_lijevi_kost is not None:
         true_distances_512["lijevi"] = np.sqrt((true_lijevi_vrh.get("x") - true_lijevi_kost.get("x"))**2 + (true_lijevi_vrh.get("y") - true_lijevi_kost.get("y"))**2)
    else: true_distances_512["lijevi"] = np.nan

    pred_lijevi_vrh = predicted_kpts_512_scale.get("lijevi_vrh")
    pred_lijevi_kost = predicted_kpts_512_scale.get("lijevi_kost")
    if pred_lijevi_vrh is not None and pred_lijevi_kost is not None:
         pred_distances_512["lijevi"] = np.sqrt((pred_lijevi_vrh.get("x") - pred_lijevi_kost.get("x"))**2 + (pred_lijevi_kost.get("y") - pred_lijevi_vrh.get("y"))**2) # Corrected order of points
    else: pred_distances_512["lijevi"] = np.nan


    # Calculate proportional values, handling potential division by zero and NaNs
    true_median_dist = true_distances_512.get("srednji", np.nan)
    pred_median_dist = pred_distances_512.get("srednji", np.nan)
    true_desno_dist = true_distances_512.get("desni", np.nan)
    true_lijevo_dist = true_distances_512.get("lijevi", np.nan)
    pred_desno_dist = pred_distances_512.get("desni", np.nan)
    pred_lijevo_dist = pred_distances_512.get("lijevi", np.nan)


    true_desno_prop = true_desno_dist / true_median_dist if not pd.isna(true_median_dist) and true_median_dist != 0 and not pd.isna(true_desno_dist) else np.nan
    pred_desno_prop = pred_desno_dist / pred_median_dist if not pd.isna(pred_median_dist) and pred_median_dist != 0 and not pd.isna(pred_desno_dist) else np.nan
    true_lijevo_prop = true_lijevo_dist / true_median_dist if not pd.isna(true_median_dist) and true_median_dist != 0 and not pd.isna(true_lijevo_dist) else np.nan
    pred_lijevo_prop = pred_lijevo_dist / pred_median_dist if not pd.isna(pred_median_dist) and pred_median_dist != 0 and not pd.isna(pred_lijevo_dist) else np.nan

    # Calculate max proportional loss (MGK %) for true and predicted
    true_max_prop = np.nan
    if not pd.isna(true_desno_prop) or not pd.isna(true_lijevo_prop):
        true_max_prop = np.nanmax([true_desno_prop if not pd.isna(true_desno_prop) else -np.inf, true_lijevo_prop if not pd.isna(true_lijevo_prop) else -np.inf])
        if true_max_prop == -np.inf: true_max_prop = np.nan

    pred_max_prop = np.nan
    if not pd.isna(pred_desno_prop) or not pd.isna(pred_lijevo_prop):
         pred_max_prop = np.nanmax([pred_desno_prop if not pd.isna(pred_desno_prop) else -np.inf, pred_lijevo_prop if not pd.isna(pred_lijevo_prop) else -np.inf])
         if pred_max_prop == -np.inf: pred_max_prop = np.nan


    # Append to lists, only if both true and predicted max proportional loss are valid (not NaN)
    if not pd.isna(true_max_prop) and not pd.isna(pred_max_prop):
        true_prop_loss_list_filtered.append(true_max_prop)
        pred_prop_loss_list_filtered.append(pred_max_prop)

        # Debugging prints for the first few instances
        if debug_print_count < debug_print_limit:
            print(f"\n--- Debugging Instance: {instance_identifier} ---")
            print(f"  True Kpts (512 scale): {true_kpts_512_scale}")
            print(f"  Pred Kpts (512 scale): {predicted_kpts_512_scale}")
            print(f"  True Distances (512 scale): {true_distances_512}")
            print(f"  Pred Distances (512 scale): {pred_distances_512}")
            print(f"  True Proportional (Desno, Lijevo): {true_desno_prop:.4f}, {true_lijevo_prop:.4f}")
            print(f"  Pred Proportional (Desno, Lijevo): {pred_desno_prop:.4f}, {pred_lijevo_prop:.4f}")
            print(f"  True Max Prop (MGK_stručnjak): {true_max_prop:.4f}")
            print(f"  Pred Max Prop (MGK_model): {pred_max_prop:.4f}")
            print(f"  Absolute Difference: {np.abs(true_max_prop - pred_max_prop):.4f}")
            debug_print_count += 1
        elif debug_print_count == debug_print_limit:
            print(f"\n--- Debugging prints limited to the first {debug_print_limit} instances. ---")
            debug_print_count += 1 # Increment to avoid printing this message again


print(f"\nCalculated proportional bone loss for {len(true_prop_loss_list_filtered)} filtered instances.")

# Convert lists to numpy arrays
true_prop_loss_array_filtered = np.array(true_prop_loss_list_filtered)
pred_prop_loss_array_filtered = np.array(pred_prop_loss_list_filtered)

# true_prop_loss_array_filtered and pred_prop_loss_array_filtered are now ready for analysis.

In [ ]:
# Perform Statistical Tests for Hypothesis 3 (Filtered Data)

# Assume true_prop_loss_array_filtered and pred_prop_loss_array_filtered are available from the previous step

from scipy import stats
import numpy as np
import pandas as pd # Import pandas for isna

print("\n--- Statistical Tests for Hypothesis 3 (Filtered Data) ---")

# Ensure both arrays have the same length and contain data
if len(true_prop_loss_array_filtered) > 0 and len(true_prop_loss_array_filtered) == len(pred_prop_loss_array_filtered):

    # Calculate Absolute Difference
    abs_diff_prop_loss_filtered = np.abs(true_prop_loss_array_filtered - pred_prop_loss_array_filtered)

    # Calculate MAE (Mean Absolute Error)
    mae_prop_loss_filtered = np.mean(abs_diff_prop_loss_filtered)

    # Calculate RMSE (Root Mean Squared Error)
    rmse_prop_loss_filtered = np.sqrt(np.mean(np.square(abs_diff_prop_loss_filtered)))

    # Calculate Pearson's correlation coefficient and p-value
    try:
        # Pearsonr requires at least 2 data points and non-zero variance
        if len(true_prop_loss_array_filtered) > 1 and np.std(true_prop_loss_array_filtered) != 0 and np.std(pred_prop_loss_array_filtered) != 0:
             correlation_r_filtered, correlation_p_filtered = stats.pearsonr(true_prop_loss_array_filtered, pred_prop_loss_array_filtered) # Corrected function name
        else:
             correlation_r_filtered = np.nan
             correlation_p_filtered = np.nan
             print("Warning: Not enough data or zero variance for Pearson correlation calculation on filtered data.")

    except ValueError:
        correlation_r_filtered = np.nan
        correlation_p_filtered = np.nan
        print("Warning: Could not calculate Pearson correlation (ValueError) on filtered data.")


    print("\n**Agregirane metrike pogreške i korelacije (filtrirani podaci):**")
    print(f"Broj korištenih instanci: {len(true_prop_loss_array_filtered)}")
    print(f"MAE (Mean Absolute Error): {mae_prop_loss_filtered:.4f}")
    print(f"RMSE (Root Mean Squared Error): {rmse_prop_loss_filtered:.4f}")
    print(f"Pearson Correlation Coefficient (r): {correlation_r_filtered:.4f}")
    print(f"Pearson Correlation p-value: {correlation_p_filtered:.4f}")


    # --- Statistical Test for MAE < 5% (Filtered Data) ---
    print("\n**Statistički test za MAE (Mean Absolute Error) < 5% (filtrirani podaci):**")

    # Define the threshold for MAE (5% = 0.05)
    mae_threshold = 0.05

    if abs_diff_prop_loss_filtered.size > 1:
        # Check for normality of absolute differences using Shapiro-Wilk test
        shapiro_w_mae_filtered, shapiro_p_mae_filtered = stats.shapiro(abs_diff_prop_loss_filtered)
        alpha = 0.05

        print(f"\nShapiro-Wilk test za normalnost apsolutnih pogrešaka (filtrirani podaci): W={shapiro_w_mae_filtered:.4f}, p={shapiro_p_mae_filtered:.4f}")

        if shapiro_p_mae_filtered < alpha:
            print("Napomena: Apsolutne pogreške ne izgledaju normalno distribuirane (prema Shapiro-Wilk testu). Preporuka: Neparametarski test (Wilcoxon).")

            # Perform One-Sided One-Sample Wilcoxon Signed-Rank Test
            try:
                # Note: Wilcoxon test requires that the differences are not all zero.
                # Filter out zero differences if necessary for the test to run, although with 152 instances
                # and floating point numbers, all differences being exactly zero is unlikely unless there's a calculation issue.
                non_zero_diff = abs_diff_prop_loss_filtered[abs_diff_prop_loss_filtered != 0]
                if len(non_zero_diff) > 0:
                     wilcoxon_statistic_mae_filtered, wilcoxon_p_value_mae_filtered = stats.wilcoxon(abs_diff_prop_loss_filtered - mae_threshold, alternative='less')

                     print(f"\n--- Jednostrani jedno-uzorački Wilcoxonov test za MAE (< {mae_threshold:.2f}) (filtrirani podaci) ---")
                     print(f"H0: Medijan apsolutne pogreške >= {mae_threshold:.2f}")
                     print(f"H1: Medijan apsolutne pogreške < {mae_threshold:.2f}")
                     print(f"Prag testiranja: {mae_threshold:.2f}")
                     print(f"MAE uzorka: {mae_prop_loss_filtered:.4f}") # Display the mean MAE for context
                     print(f"Medijan apsolutne pogreške uzorka: {np.median(abs_diff_prop_loss_filtered):.4f}") # Display median for context
                     print(f"Wilcoxon statistika: {wilcoxon_statistic_mae_filtered:.4f}")
                     print(f"P-vrijednost (jednostrana): {wilcoxon_p_value_mae_filtered:.4f}")

                     if wilcoxon_p_value_mae_filtered < alpha:
                         print(f"\nRezultat (Wilcoxon): Odbacujemo nul-hipotezu na razini značajnosti {alpha:.2f}.")
                         print(f"Zaključak (Wilcoxon): Medijan apsolutne pogreške za MGK (%) ({np.median(abs_diff_prop_loss_filtered):.4f}) statistički je značajno manji od {mae_threshold:.2f}.")
                     else:
                          print(f"\nRezultat (Wilcoxon): Ne uspijevamo odbaciti nul-hipotezu na razini značajnosti {alpha:.2f}.")
                          print(f"Zaključak (Wilcoxon): Nema dovoljno statističkih dokaza da se zaključi da je medijan apsolutne pogreške za MGK (%) manji od {mae_threshold:.2f}.")
                else:
                     print("Nije moguće provesti Wilcoxonov test jer su sve apsolutne razlike jednake nuli.")


            except ValueError as ve:
                print(f"Nije moguće provesti Wilcoxonov test za MAE (filtrirani podaci): {ve}.")

        else:
            print("Napomena: Apsolutne pogreške izgledaju normalno distribuirane (prema Shapiro-Wilk testu). Rezultati T-testa su vjerojatno valjani.")

            # Perform One-Sided One-Sample T-Test
            try:
                t_statistic_mae_filtered, p_value_mae_filtered = stats.ttest_1samp(abs_diff_prop_loss_filtered, mae_threshold, alternative='less')

                print(f"\n--- Jednostrani jedno-uzorački T-test za MAE (< {mae_threshold:.2f}) (filtrirani podaci) ---")
                print(f"H0: Prosječna apsolutna pogreška >= {mae_threshold:.2f}")
                print(f"H1: Prosječna apsolutna pogreška < {mae_threshold:.2f}")
                print(f"Prag testiranja: {mae_threshold:.2f}")
                print(f"MAE uzorka: {mae_prop_loss_filtered:.4f}")
                print(f"T-statistika: {t_statistic_mae_filtered:.4f}")
                print(f"P-vrijednost (jednostrana): {p_value_mae_filtered:.4f}")

                if p_value_mae_filtered < alpha:
                    print(f"\nRezultat (T-test): Odbacujemo nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                    print(f"Zaključak (T-test): Prosječna apsolutna pogreška za MGK (%) ({mae_prop_loss_filtered:.4f}) statistički je značajno manja od {mae_threshold:.2f}.")
                else:
                    print(f"\nRezultat (T-test): Ne uspijevamo odbaciti nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                    print(f"Zaključak (T-test): Nema dovoljno statističkih dokaza da se zaključi da je prosječna apsolutna pogreška za MGK (%) manja od {mae_threshold:.2f}.")

            except Exception as e:
                print(f"Došlo je do pogreške prilikom izvođenja T-testa za MAE (filtrirani podaci): {e}")

    else:
        print("Nije moguće provesti statistički test za MAE jer nema dovoljno podataka (manje od 2 valjane podatkovne točke) u 'abs_diff_prop_loss_filtered'.")


    # --- Statistical Test for Pearson's r > 0.8 (Filtered Data) ---
    print("\n**Statistički test za Pearsonov r > 0.8 (filtrirani podaci):**")

    correlation_threshold_r = 0.8

    if not pd.isna(correlation_r_filtered):
        print(f"Pearsonov koeficijent korelacije (r): {correlation_r_filtered:.4f}")
        print(f"Pearsonova p-vrijednost: {correlation_p_filtered:.4f}")
        print(f"Prag za r: > {correlation_threshold_r:.1f}")

        # Check for statistical significance of the correlation (H0: r = 0)
        if correlation_p_filtered < alpha:
            print(f"\nRezultat (Statistička značajnost r): Korelacija je statistički značajna na razini značajnosti {alpha:.2f} (p < {alpha:.2f}).")
            print("Zaključak: Postoji statistički značajna linearna veza između MGK_model i MGK_stručnjak (filtrirani podaci).")

            # Check if the calculated r is greater than the threshold (Descriptive comparison)
            if correlation_r_filtered > correlation_threshold_r:
                print(f"\nRezultat (Vrijednost r): Pearsonov r ({correlation_r_filtered:.4f}) je veći od postavljenog praga {correlation_threshold_r:.1f}.")
                print("Zaključak: Vrijednost korelacije ukazuje na jaku pozitivnu linearnu vezu, iznad postavljenog kriterija (filtrirani podaci).")
            else:
                print(f"\nRezultat (Vrijednost r): Pearsonov r ({correlation_r_filtered:.4f}) nije veći od postavljenog praga {correlation_threshold_r:.1f}.")
                print("Zaključak: Iako je korelacija statistički značajna, njezina vrijednost ne doseže postavljeni kriterij od 0.8 (filtrirani podaci).")

        else:
            print(f"\nRezultat (Statistička značajnost r): Korelacija nije statistički značajna na razini značajnosti {alpha:.2f} (p >= {alpha:.2f}).")
            print("Zaključak: Nema dovoljno statističkih dokaza da se zaključi da postoji statistički značajna linearna veza između MGK_model i MGK_stručnjak (r=0) (filtrirani podaci).")
            print("Vrijednost korelacije ({correlation_r_filtered:.4f}) u ovom slučaju nije pouzdana za interpretaciju u odnosu na prag od 0.8.")


    else:
        print("Nije moguće provesti statistički test za Pearsonov r jer vrijednosti korelacije nisu dostupne ili nisu valjane (filtrirani podaci).")

else:
    print("Nije moguće provesti statistički test za Hipotezu 3 jer filtrirani podaci nisu dostupni ili imaju nejednake duljine.")

### Sažetak rezultata za Hipotezu 3 (filtrirani podaci)

Analiza točnosti kvantifikacije MGK (%) na internom testnom skupu, **nakon uklanjanja odstupanja ključnih točaka**, dala je sljedeće rezultate:

In [ ]:
# Display Summary of Hypothesis 3 Results (Filtered Data)

# Assume mae_prop_loss_filtered, rmse_prop_loss_filtered, correlation_r_filtered, correlation_p_filtered are available
# Assume the results from the statistical tests (MAE < 0.05 and Pearson's r > 0.8) are available in the previous outputs

print("**Agregirane metrike pogreške i korelacije (filtrirani podaci):**")
if 'mae_prop_loss_filtered' in locals() and 'rmse_prop_loss_filtered' in locals() and 'correlation_r_filtered' in locals() and 'correlation_p_filtered' in locals():
    print(f"- Broj korištenih instanci: {len(true_prop_loss_array_filtered) if 'true_prop_loss_array_filtered' in locals() else 'N/A'}")
    print(f"- Srednja apsolutna pogreška (MAE) za MGK (%): {mae_prop_loss_filtered:.4f}")
    print(f"- Korijen srednje kvadratne pogreške (RMSE) za MGK (%): {rmse_prop_loss_filtered:.4f}")
    print(f"- Pearsonov koeficijent korelacije (r): {correlation_r_filtered:.4f}")
    print(f"- Pearsonova p-vrijednost: {correlation_p_filtered:.4f}")
else:
    print("Metrike pogreške i korelacije za MGK (%) (filtrirani podaci) nisu dostupne.")

print("\n**Statistički testovi za Hipotezu 3 (filtrirani podaci):**")

# Reiterate conclusions from the statistical tests based on previous outputs
# You might need to manually inspect the output of the previous cell to provide the exact conclusions.
# Here's a placeholder based on typical interpretation:

print("\n*Statistički test za MAE < 5% (filtrirani podaci):*")
# Based on the output of the previous cell:
if 'shapiro_p_mae_filtered' in locals() and shapiro_p_mae_filtered < 0.05:
     # Non-parametric test was used
     if 'wilcoxon_p_value_mae_filtered' in locals():
          print(f"Rezultat Wilcoxonovog testa (MAE < 0.05): {'Odbacujemo nul-hipotezu' if wilcoxon_p_value_mae_filtered < 0.05 else 'Ne uspijevamo odbaciti nul-hipotezu'} na razini značajnosti 0.05 (p-vrijednost = {wilcoxon_p_value_mae_filtered:.4f}).")
          print(f"Zaključak: {'Medijan apsolutne pogreške za MGK (%) statistički je značajno manji od 0.05.' if wilcoxon_p_value_mae_filtered < 0.05 else 'Nema dovoljno statističkih dokaza da se zaključi da je medijan apsolutne pogreške manji od 0.05.'}")
     else:
          print("Rezultati Wilcoxonovog testa nisu dostupni.")
elif 'p_value_mae_filtered' in locals():
     # T-test was used
     print(f"Rezultat T-testa (MAE < 0.05): {'Odbacujemo nul-hipotezu' if p_value_mae_filtered < 0.05 else 'Ne uspijevamo odbaciti nul-hipotezu'} na razini značajnosti 0.05 (p-vrijednost = {p_value_mae_filtered:.4f}).")
     print(f"Zaključak: {'Prosječna apsolutna pogreška za MGK (%) statistički je značajno manja od 0.05.' if p_value_mae_filtered < 0.05 else 'Nema dovoljno statističkih dokaza da se zaključi da je prosječna apsolutna pogreška manja od 0.05.'}")
else:
     print("Rezultati testa za MAE nisu dostupni.")


print("\n*Statistički test za Pearsonov r > 0.8 (filtrirani podaci):*")
if 'correlation_r_filtered' in locals() and 'correlation_p_filtered' in locals() and not pd.isna(correlation_r_filtered):
    print(f"Pearsonov koeficijent korelacije (r): {correlation_r_filtered:.4f}, Pearsonova p-vrijednost: {correlation_p_filtered:.4f}")
    if correlation_p_filtered < 0.05:
        print("Rezultat (Statistička značajnost r): Korelacija je statistički značajna.")
        if correlation_r_filtered > 0.8:
            print("Zaključak: Vrijednost korelacije ({:.4f}) je veća od praga 0.8.".format(correlation_r_filtered))
        else:
            print("Zaključak: Vrijednost korelacije ({:.4f}) nije veća od praga 0.8.".format(correlation_r_filtered))
    else:
        print("Rezultat (Statistička značajnost r): Korelacija nije statistički značajna.")
        print("Zaključak: Nema dovoljno statističkih dokaza za značajnu linearnu vezu.")
else:
    print("Rezultati testa za Pearsonov r nisu dostupni.")

### Zaključak za Hipotezu 3 (filtrirani podaci)

Na temelju provedene analize i statističkih testova za Hipotezu 3 na internom testnom skupu, **nakon uklanjanja odstupanja ključnih točaka**, možemo donijeti sljedeće zaključke:

*   **Točnost kvantifikacije MGK (%):**
    *   Srednja apsolutna pogreška (MAE) i Korijen srednje kvadratne pogreške (RMSE) za MGK (%) izračunati na filtriranom skupu podataka. Ove vrijednosti predstavljaju prosječno odstupanje modelove procjene od stručnjakove anotacije na podskupu podataka bez identificiranih odstupanja ključnih točaka. (Vrijednosti pogledajte u gornjim ispisima).
    *   Statistički test za MAE < 5% proveden je na filtriranim podacima. (Rezultate i zaključak pogledajte u gornjim ispisima).
*   **Korelacija između MGKmodel i MGKstručnjak:**
    *   Pearsonov koeficijent korelacije (r) i pripadajuća p-vrijednost izračunati su na filtriranom skupu podataka. Ove vrijednosti ukazuju na snagu i statističku značajnost linearne veze između procjena modela i stručnjaka na podskupu podataka bez odstupanja ključnih točaka. (Vrijednosti i zaključak pogledajte u gornjim ispisima).

**Sveukupno:** Zaključak o Hipotezi 3 na filtriranom skupu podataka ovisi o specifičnim numeričkim rezultatima iz gornjih ćelija. Uklanjanje odstupanja ključnih točaka može poboljšati metrike točnosti (MAE, RMSE) i korelacije ako su odstupanja ključnih točaka dovodila do značajnih pogrešaka u izračunu MGK %. Statistički testovi će pokazati jesu li te poboljšane metrike statistički značajne u odnosu na postavljene pragove (MAE < 5% i r > 0.8).

Za potpunu interpretaciju, potrebno je analizirati numeričke vrijednosti i zaključke iz gornjih ispisnih ćelija.

### Vizualizacije za detekciju graničnih okvira (Interni testni skup)

In [ ]:
# Generate Histogram of IoU Distribution for the Test Set

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assume all_iou_values is available from previous calculations (cell 79a904ef)

print("Generating histogram of IoU distribution...")

if 'all_iou_values' in locals() and all_iou_values.size > 0:
    plt.figure(figsize=(10, 6))
    sns.histplot(all_iou_values, bins=20, kde=True, color='skyblue')
    plt.title('Distribucija IoU vrijednosti za granične okvire (Interni testni skup)')
    plt.xlabel('IoU')
    plt.ylabel('Frekvencija')
    plt.grid(axis='y', alpha=0.75)
    plt.show()

    print(f"\nAggregated IoU Metrics from the distribution:")
    print(f"  Mean IoU: {np.mean(all_iou_values):.4f}")
    print(f"  Median IoU: {np.median(all_iou_values):.4f}")
    print(f"  Standard Deviation of IoU: {np.std(all_iou_values):.4f}")
    print(f"  Number of IoU values: {all_iou_values.size}")


else:
    print("IoU values are not available or the array is empty. Cannot generate histogram.")

### Primjeri slika s predikcijama graničnih okvira

In [ ]:
# Display Example Images with Bounding Box Predictions

import random
import cv2
import matplotlib.pyplot as plt
import os
from ultralytics.utils import ops # Assuming ops is available from ultralytics

# Assume pose_model is loaded
# Assume test_image_dir and internal_test_ground_truth_data are defined

print("Displaying example images with bounding box predictions...")

# Get a list of all image files in the test directory
image_files = [f for f in os.listdir(test_image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]

if not image_files:
    print(f"No image files found in {test_image_dir}.")
else:
    # Select a few random images to display
    num_examples_to_display = min(5, len(image_files)) # Display up to 5 examples
    selected_images = random.sample(image_files, num_examples_to_display)

    for img_filename in selected_images:
        img_path = os.path.join(test_image_dir, img_filename)

        try:
            # Perform prediction for the image
            # Use the same thresholds as used for evaluation (0.4, 0.4)
            predikcija = pose_model([img_path], conf=0.4, iou=0.4, verbose=False)

            # Load the image using OpenCV for drawing
            img = cv2.imread(img_path)
            if img is None:
                print(f"Warning: Could not read image file at {img_path}. Skipping display for this image.")
                continue

            # Draw bounding boxes on the image
            # Iterate through the predicted boxes
            if predikcija[0].boxes is not None and len(predikcija[0].boxes) > 0:
                # Boxes are in xywh format, need to convert to xyxy for drawing with cv2
                predicted_boxes_xywh = predikcija[0].boxes.xywh.cpu().numpy()
                predicted_boxes_xyxy = ops.xywh2xyxy(predicted_boxes_xywh) # Assuming ops.xywh2xyxy exists

                for box in predicted_boxes_xyxy:
                    x1, y1, x2, y2 = map(int, box[:4]) # Coordinates are floats, convert to int for drawing
                    conf = box[4] # Confidence score (if available in xyxy output)
                    # cls = box[5] # Class ID (if available) - Not needed for this dataset

                    # Draw rectangle
                    color = (0, 255, 0) # Green color (BGR format)
                    thickness = 2
                    img = cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)

                    # Optionally add label and confidence
                    label = f"Obj: {conf:.2f}" # Assuming class ID is not needed for this dataset
                    font = cv2.FONT_HERSHEY_SIMPLEX
                    font_scale = 0.5
                    font_thickness = 1
                    text_size = cv2.getTextSize(label, font, font_scale, font_thickness)[0]
                    text_x = x1
                    text_y = y1 - 10 if y1 - 10 > 10 else y1 + 10 # Position text above box

                    cv2.putText(img, label, (text_x, text_y), font, font_scale, color, font_thickness, cv2.LINE_AA)


            # Convert image from BGR to RGB for displaying with matplotlib
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Display the image
            plt.figure(figsize=(8, 8))
            plt.imshow(img_rgb)
            plt.title(f"Bounding Box Prediction: {img_filename}")
            plt.axis('off')
            plt.show()

        except Exception as e:
            print(f"Error processing or displaying image {img_filename}: {e}")

### Vizualizacije za detekciju ključnih točaka (Interni testni skup)

In [ ]:
# Generate Histogram of OKS Distribution for the Test Set

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assume all_oks_values is available from previous calculations (cell 2cda0ba8)

print("Generating histogram of OKS distribution...")

if 'all_oks_values' in locals() and all_oks_values.size > 0:
    plt.figure(figsize=(10, 6))
    sns.histplot(all_oks_values, bins=20, kde=True, color='salmon')
    plt.title('Distribucija OKS vrijednosti za ključne točke (Interni testni skup)')
    plt.xlabel('OKS')
    plt.ylabel('Frekvencija')
    plt.grid(axis='y', alpha=0.75)
    plt.show()

    print(f"\nAggregated OKS Metrics from the distribution:")
    print(f"  Mean OKS: {np.mean(all_oks_values):.4f}")
    print(f"  Median OKS: {np.median(all_oks_values):.4f}")
    print(f"  Standard Deviation of OKS: {np.std(all_oks_values):.4f}")
    print(f"  Number of OKS values: {all_oks_values.size}")

else:
    print("OKS values are not available or the array is empty. Cannot generate histogram.")

### Primjeri slika s predikcijama skeleta

In [ ]:
# Display Example Images with Keypoint Predictions (Skeletons)

import random
import cv2
import matplotlib.pyplot as plt
import os
# Removed the import of plot_skeleton_kpts

# Assume pose_model is loaded
# Assume test_image_dir is defined
# Assume Keypoints and Tocka are defined (needed for plotting if not handled by model.plot)

print("Displaying example images with keypoint predictions (skeletons)...")

# Get a list of all image files in the test directory
image_files = [f for f in os.listdir(test_image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]

if not image_files:
    print(f"No image files found in {test_image_dir}.")
else:
    # Select a few random images to display
    num_examples_to_display = min(5, len(image_files)) # Display up to 5 examples
    selected_images = random.sample(image_files, num_examples_to_display)

    for img_filename in selected_images:
        img_path = os.path.join(test_image_dir, img_filename)

        try:
            # Perform prediction for the image
            # Use the same thresholds as used for evaluation (0.4, 0.4)
            predikcija = pose_model([img_path], conf=0.4, iou=0.4, verbose=False)

            # Load the image using OpenCV for drawing - Not strictly needed if using results.plot(), but kept for safety
            # img = cv2.imread(img_path)
            # if img is None:
            #     print(f"Warning: Could not read image file at {img_path}. Skipping display for this image.")
            #     continue

            # Draw skeletons on the image using the model's built-in plot method
            # This method handles both bounding boxes and keypoints/skeletons
            if predikcija is not None and len(predikcija) > 0 and predikcija[0] is not None:
                plotted_img_array = predikcija[0].plot() # This method returns a numpy array (BGR format)

                # Convert image from BGR to RGB for displaying with matplotlib
                # The plot() method returns BGR, so convert to RGB
                img_rgb = cv2.cvtColor(plotted_img_array, cv2.COLOR_BGR2RGB)

                # Display the image
                plt.figure(figsize=(8, 8))
                plt.imshow(img_rgb)
                plt.title(f"Keypoint Prediction (Skeleton): {img_filename}")
                plt.axis('off')
                plt.show()
            else:
                 print(f"Warning: No valid prediction results for {img_filename}. Skipping display.")


        except Exception as e:
            print(f"Error processing or displaying image {img_filename}: {e}")

### Skalirana euklidska pogreška ključnih točaka po točki (Box plot)

In [ ]:
# load an example dataset
from vega_datasets import data
cars = data.cars()

# plot the dataset, referencing dataframe column names
import altair as alt
alt.Chart(cars).mark_bar().encode(
  x=alt.X('Miles_per_Gallon', bin=True),
  y='count()',
)

In [ ]:
# Generate Box plot of Scaled Euclidean Distance Error by Keypoint

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Assume keypoint_errors_df is available from previous calculations (cell f7026f50)
# This DataFrame should now contain data after IQR outlier removal for all keypoints

print("Generating box plot of scaled Euclidean distance error by keypoint (after outlier removal)...")

if 'keypoint_errors_df' in locals() and not keypoint_errors_df.empty:
    plt.figure(figsize=(12, 7))
    sns.boxplot(x='keypoint', y='scaled_euclidean_error', data=keypoint_errors_df, palette='viridis')
    plt.title('Distribucija skalirane euklidske pogreške po ključnoj točki (Interni testni skup)')
    plt.xlabel('Ključna točka')
    plt.ylabel('Skalirana euklidska pogreška')
    plt.grid(axis='y', alpha=0.75)
    plt.show()
else:
    print("Keypoint error data (filtered) is not available or the DataFrame is empty. Cannot generate box plot.")

### Vizualizacije za kvantifikaciju MGK (%) (Interni testni skup, filtrirani podaci)

In [ ]:
# Prepare Data for MGK Visualizations (Filtered Data)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import cv2
import os

# Assume processed_instances_for_hypothesis_3_filtered is available from the previous step
# Assume Keypoints, Tocka, getApexVrh, getRightSide, getLeftSide are defined

# Lists to store calculated proportional bone loss values for individual sides and max (for visualizations)
true_desno_prop_list_filtered = []
pred_desno_prop_list_filtered = []
true_lijevo_prop_list_filtered = []
pred_lijevo_prop_list_filtered = []
true_max_prop_list_filtered = []
pred_max_prop_list_filtered = []


print("Calculating individual side proportional bone loss for visualizations (filtered data)...")

# Iterate through the filtered processed instances
for instance_data in processed_instances_for_hypothesis_3_filtered:
    instance_identifier = f"{instance_data['filename']}_instance_{instance_data['instance_idx']}"

    true_kpts_original_scale = instance_data["true"]
    predicted_kpts_512_scale = instance_data["pred"]
    # true_box_wh_original_scale = instance_data["true_box_wh_original_scale"] # Not directly used for distances but kept in instance_data

    # Retrieve original image dimensions (needed for scaling true keypoints if we calculate distances on 512 scale)
    img_path = os.path.join(test_image_dir, instance_data['filename'])
    img = cv2.imread(img_path)
    if img is None:
        continue
    img_h, img_w, _ = img.shape

    # Scale true keypoints from original scale to 512 scale for distance calculation consistency
    true_kpts_512_scale = {} # Use dict based on previous findings
    for kp_name, coords in true_kpts_original_scale.items():
        if coords is not None:
            true_kpts_512_scale[kp_name] = {"x": coords.get("x") * (512 / img_w), "y": coords.get("y") * (512 / img_h)}
        else:
            true_kpts_512_scale[kp_name] = None


    # Calculate distances manually using the 512 scale keypoints (using dict access)
    true_distances_512 = {}
    pred_distances_512 = {}

    # Calculate Srednji (Median) Distance (VRH to APEX)
    true_vrh = true_kpts_512_scale.get("vrh")
    true_apex = true_kpts_512_scale.get("apex")
    if true_vrh is not None and true_apex is not None:
        true_distances_512["srednji"] = np.sqrt((true_vrh.get("x") - true_apex.get("x"))**2 + (true_vrh.get("y") - true_apex.get("y"))**2)
    else: true_distances_512["srednji"] = np.nan

    pred_vrh = predicted_kpts_512_scale.get("vrh")
    pred_apex = predicted_kpts_512_scale.get("apex")
    if pred_vrh is not None and pred_apex is not None:
        pred_distances_512["srednji"] = np.sqrt((pred_vrh.get("x") - pred_apex.get("x"))**2 + (pred_vrh.get("y") - pred_apex.get("y"))**2)
    else: pred_distances_512["srednji"] = np.nan

    # Calculate Desni (Right) Distance (DESNI_GORNJI to DESNI_DONJI)
    true_desni_vrh = true_kpts_512_scale.get("desni_vrh")
    true_desni_kost = true_kpts_512_scale.get("desni_kost")
    if true_desni_vrh is not None and true_desni_kost is not None:
         true_distances_512["desni"] = np.sqrt((true_desni_vrh.get("x") - true_desni_kost.get("x"))**2 + (true_desni_vrh.get("y") - true_desni_kost.get("y"))**2)
    else: true_distances_512["desni"] = np.nan

    pred_desni_vrh = predicted_kpts_512_scale.get("desni_vrh")
    pred_desni_kost = predicted_kpts_512_scale.get("desni_kost")
    if pred_desni_vrh is not None and pred_desni_kost is not None:
         pred_distances_512["desni"] = np.sqrt((pred_desni_vrh.get("x") - pred_desni_kost.get("x"))**2 + (pred_desni_vrh.get("y") - pred_desni_kost.get("y"))**2)
    else: pred_distances_512["desni"] = np.nan

    # Calculate Lijevi (Left) Distance (LIJEVI_GORNJI to LIJEVI_DONJI)
    true_lijevi_vrh = true_kpts_512_scale.get("lijevi_vrh")
    true_lijevi_kost = true_kpts_512_scale.get("lijevi_kost")
    if true_lijevi_vrh is not None and true_lijevi_kost is not None:
         true_distances_512["lijevi"] = np.sqrt((true_lijevi_vrh.get("x") - true_lijevi_kost.get("x"))**2 + (true_lijevi_vrh.get("y") - true_lijevi_kost.get("y"))**2)
    else: true_distances_512["lijevi"] = np.nan

    pred_lijevi_vrh = predicted_kpts_512_scale.get("lijevi_vrh")
    pred_lijevi_kost = predicted_kpts_512_scale.get("lijevi_kost")
    if pred_lijevi_vrh is not None and pred_lijevi_kost is not None:
         pred_distances_512["lijevi"] = np.sqrt((pred_lijevi_vrh.get("x") - pred_lijevi_kost.get("x"))**2 + (pred_lijevi_kost.get("y") - pred_lijevi_vrh.get("y"))**2) # Corrected order of points
    else: pred_distances_512["lijevi"] = np.nan


    # Calculate proportional values, handling potential division by zero and NaNs
    true_median_dist = true_distances_512.get("srednji", np.nan)
    pred_median_dist = pred_distances_512.get("srednji", np.nan)
    true_desno_dist = true_distances_512.get("desni", np.nan)
    true_lijevo_dist = true_distances_512.get("lijevi", np.nan)
    pred_desno_dist = pred_distances_512.get("desni", np.nan)
    pred_lijevo_dist = pred_distances_512.get("lijevi", np.nan)


    true_desno_prop = true_desno_dist / true_median_dist if not pd.isna(true_median_dist) and true_median_dist != 0 and not pd.isna(true_desno_dist) else np.nan
    pred_desno_prop = pred_desno_dist / pred_median_dist if not pd.isna(pred_median_dist) and pred_median_dist != 0 and not pd.isna(pred_desno_dist) else np.nan
    true_lijevo_prop = true_lijevo_dist / true_median_dist if not pd.isna(true_median_dist) and true_median_dist != 0 and not pd.isna(true_lijevo_dist) else np.nan
    pred_lijevo_prop = pred_lijevo_dist / pred_median_dist if not pd.isna(pred_median_dist) and pred_median_dist != 0 and not pd.isna(pred_lijevo_dist) else np.nan

    # Calculate max proportional loss (MGK %) for true and predicted
    true_max_prop = np.nan
    if not pd.isna(true_desno_prop) or not pd.isna(true_lijevo_prop):
        true_max_prop = np.nanmax([true_desno_prop if not pd.isna(true_desno_prop) else -np.inf, true_lijevo_prop if not pd.isna(true_lijevo_prop) else -np.inf])
        if true_max_prop == -np.inf: true_max_prop = np.nan

    pred_max_prop = np.nan
    if not pd.isna(pred_desno_prop) or not pd.isna(pred_lijevo_prop):
         pred_max_prop = np.nanmax([pred_desno_prop if not pd.isna(pred_desno_prop) else -np.inf, pred_lijevo_prop if not pd.isna(pred_lijevo_prop) else -np.inf])
         if pred_max_prop == -np.inf: pred_max_prop = np.nan


    # Append to lists, only if both true and predicted max proportional loss are valid (not NaN)
    if not pd.isna(true_max_prop) and not pd.isna(pred_max_prop):
        true_desno_prop_list_filtered.append(true_desno_prop) # Append individual side losses only for instances with valid max
        pred_desno_prop_list_filtered.append(pred_desno_prop)
        true_lijevo_prop_list_filtered.append(true_lijevo_prop)
        pred_lijevo_prop_list_filtered.append(pred_lijevo_prop)
        true_max_prop_list_filtered.append(true_max_prop)
        pred_max_prop_list_filtered.append(pred_max_prop)


print(f"Calculated individual side proportional bone loss for {len(true_max_prop_list_filtered)} filtered instances for visualizations.")

# Convert lists to numpy arrays
true_desno_prop_array_filtered = np.array(true_desno_prop_list_filtered)
pred_desno_prop_array_filtered = np.array(pred_desno_prop_list_filtered)
true_lijevo_prop_array_filtered = np.array(true_lijevo_prop_list_filtered)
pred_lijevo_prop_array_filtered = np.array(pred_lijevo_prop_list_filtered)
true_max_prop_array_filtered = np.array(true_max_prop_list_filtered)
pred_max_prop_array_filtered = np.array(pred_max_prop_list_filtered)


# Calculate absolute differences for histograms
abs_diff_desno_filtered = np.abs(true_desno_prop_array_filtered - pred_desno_prop_array_filtered)
abs_diff_lijevo_filtered = np.abs(true_lijevo_prop_array_filtered - pred_lijevo_prop_array_filtered)
abs_diff_max_filtered = np.abs(true_max_prop_array_filtered - pred_max_prop_array_filtered)

# Create a DataFrame for easier plotting (optional, but good for seaborn)
mgk_diff_data_filtered = pd.DataFrame({
    'Absolute Difference (Right)': abs_diff_desno_filtered,
    'Absolute Difference (Left)': abs_diff_lijevo_filtered,
    'Absolute Difference (Max)': abs_diff_max_filtered
})

# Create a DataFrame for scatter plot and Bland-Altman
mgk_comparison_data_filtered = pd.DataFrame({
    'MGK_stručnjak': true_max_prop_array_filtered,
    'MGK_model': pred_max_prop_array_filtered
})

print("\nData prepared for MGK visualizations (filtered data).")

### Histograms of Absolute MGK (%) Differences (filtrirani podaci)

In [ ]:
# Generate Histograms of Absolute MGK (%) Differences (Filtered Data)

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Assume mgk_diff_data_filtered is available from the previous step

print("Generating histograms of absolute MGK (%) differences (filtered data)...")

if not mgk_diff_data_filtered.empty:
    # Melt the DataFrame to plot all differences in a single figure (optional, but good for comparison)
    mgk_diff_melted_filtered = mgk_diff_data_filtered.melt(var_name='Measurement', value_name='Absolute Difference')

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    sns.histplot(mgk_diff_data_filtered['Absolute Difference (Right)'], bins=20, kde=True, color='skyblue')
    plt.title('Absolute Difference (Right)')
    plt.xlabel('Absolute Difference in MGK (%)')
    plt.ylabel('Frequency')
    plt.grid(axis='y', alpha=0.75)

    plt.subplot(1, 3, 2)
    sns.histplot(mgk_diff_data_filtered['Absolute Difference (Left)'], bins=20, kde=True, color='salmon')
    plt.title('Absolute Difference (Left)')
    plt.xlabel('Absolute Difference in MGK (%)')
    plt.ylabel('Frequency')
    plt.grid(axis='y', alpha=0.75)

    plt.subplot(1, 3, 3)
    sns.histplot(mgk_diff_data_filtered['Absolute Difference (Max)'], bins=20, kde=True, color='lightgreen')
    plt.title('Absolute Difference (Max MGK)')
    plt.xlabel('Absolute Difference in MGK (%)')
    plt.ylabel('Frequency')
    plt.grid(axis='y', alpha=0.75)


    plt.tight_layout()
    plt.show()

else:
    print("Filtered MGK difference data is not available or empty. Cannot generate histograms.")

### Scatter Plot: MGK Model vs. MGK Expert (filtrirani podaci)

In [ ]:
# Generate Scatter Plot of MGK Model vs. MGK Expert (Filtered Data)

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats

# Assume mgk_comparison_data_filtered is available from the previous step

print("Generating scatter plot of MGK Model vs. MGK Expert (filtered data)...")

if not mgk_comparison_data_filtered.empty:
    plt.figure(figsize=(8, 8))
    sns.scatterplot(x='MGK_stručnjak', y='MGK_model', data=mgk_comparison_data_filtered, alpha=0.6)

    # Add line of identity (y=x)
    max_val = max(mgk_comparison_data_filtered['MGK_stručnjak'].max(), mgk_comparison_data_filtered['MGK_model'].max())
    min_val = min(mgk_comparison_data_filtered['MGK_stručnjak'].min(), mgk_comparison_data_filtered['MGK_model'].min())
    plt.plot([min_val, max_val], [min_val, max_val], color='gray', linestyle='--', label='Identity line (y=x)')

    # Add regression line
    # Calculate regression
    # Handle cases where data might have constant values after filtering
    if mgk_comparison_data_filtered['MGK_stručnjak'].std() != 0 and mgk_comparison_data_filtered['MGK_model'].std() != 0:
        slope, intercept, r_value, p_value, std_err = stats.linregress(mgk_comparison_data_filtered['MGK_stručnjak'], mgk_comparison_data_filtered['MGK_model'])
        plt.plot(mgk_comparison_data_filtered['MGK_stručnjak'], intercept + slope * mgk_comparison_data_filtered['MGK_stručnjak'], color='red', label=f'Regression line (y={intercept:.2f} + {slope:.2f}x, R²={r_value**2:.2f})')
        print(f"\nLinear Regression Results (Filtered Data):")
        print(f"  Slope: {slope:.4f}")
        print(f"  Intercept: {intercept:.4f}")
        print(f"  Pearson's r: {r_value:.4f}")
        print(f"  R-squared (R²): {r_value**2:.4f}")
        print(f"  P-value: {p_value:.4f}")
    else:
        print("\nLinear Regression could not be calculated on filtered data (zero variance).")


    plt.title('MBL Model vs. MBL Expert')
    plt.xlabel('MBL Expert (%)')
    plt.ylabel('MBL Model (%)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.axis('equal') # Ensure equal scaling for x and y axes
    plt.show()


else:
    print("Filtered MGK comparison data is not available or empty. Cannot generate scatter plot.")

### Bland-Altman Plot (filtrirani podaci)

In [ ]:
# Generate Bland-Altman Plot (Filtered Data)

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Assume mgk_comparison_data_filtered is available from the previous step

print("Generating Bland-Altman plot (filtered data)...")

if not mgk_comparison_data_filtered.empty:
    # Calculate the difference and the average
    differences = mgk_comparison_data_filtered['MGK_model'] - mgk_comparison_data_filtered['MGK_stručnjak']
    averages = (mgk_comparison_data_filtered['MGK_model'] + mgk_comparison_data_filtered['MGK_stručnjak']) / 2

    # Check if differences have variability before calculating std dev
    if np.std(differences, ddof=1) > 1e-9: # Check if std dev is not close to zero
         # Calculate the mean of the differences and the limits of agreement (mean +/- 1.96 * std dev)
         mean_difference = np.mean(differences)
         std_difference = np.std(differences, ddof=1) # Use ddof=1 for sample standard deviation
         limit_of_agreement_upper = mean_difference + 1.96 * std_difference
         limit_of_agreement_lower = mean_difference - 1.96 * std_difference

         plt.figure(figsize=(10, 7))
         sns.scatterplot(x=averages, y=differences, alpha=0.6)

         # Add lines for the mean difference and limits of agreement
         plt.axhline(mean_difference, color='red', linestyle='-', label=f'Mean difference ({mean_difference:.2f})')
         plt.axhline(limit_of_agreement_upper, color='gray', linestyle='--', label=f'+1.96 SD ({limit_of_agreement_upper:.2f})')
         plt.axhline(limit_of_agreement_lower, color='gray', linestyle='--', label=f'-1.96 SD ({limit_of_agreement_lower:.2f})')

         plt.title('Bland-Altman graph: MBL Model vs. MBL Expert')
         plt.xlabel('Average MBL (%) (Model + Expert) / 2')
         plt.ylabel('Difference MBL (%) (Model - Expert)')
         plt.legend()
         plt.grid(True, linestyle='--', alpha=0.6)
         plt.show()

         print("\nBland-Altman Results (Filtered Data):")
         print(f"  Mean Difference (Model - Expert): {mean_difference:.4f}")
         print(f"  Standard Deviation of Differences: {std_difference:.4f}")
         print(f"  95% Limits of Agreement: {limit_of_agreement_lower:.4f} to {limit_of_agreement_upper:.4f}")
    else:
        print("\nNot enough variability in differences to calculate Bland-Altman limits of agreement (Standard Deviation is zero or near zero).")
        plt.figure(figsize=(10, 7))
        sns.scatterplot(x=averages, y=differences, alpha=0.6)
        plt.axhline(np.mean(differences), color='red', linestyle='-', label=f'Srednja razlika ({np.mean(differences):.2f})')
        plt.title('Bland-Altmann graph: MBL Model vs. MBL Expert')
        plt.xlabel('Average MBL (%) (Model + Expert) / 2')
        plt.ylabel('Difference MBL (%) (Model - Expert)')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.show()
        print("\nMean Difference (Model - Expert):", np.mean(differences))


else:
    print("Filtered MGK comparison data is not available or empty. Cannot generate Bland-Altman plot.")

## Evaluacija na Internom Validacijskom Skupu

Ova sekcija provodi evaluaciju modela na internom validacijskom skupu radi usporedbe performansi s internim testnim skupom.

### Skalirana euklidska pogreška ključnih točaka (Interni validacijski skup)

**Subtask:** Izračunati skaliranu euklidsku pogrešku za svaku ključnu točku na internom validacijskom skupu.

In [ ]:
# Calculate Scaled Euclidean Distance Error for Keypoints on the Validation Set

import pandas as pd
import cv2
import os
import numpy as np

# Assume pose_model is loaded from a previous cell
# Assume Keypoints, Tocka, getApexVrh, getRightSide, getLeftSide, PointDistance are defined
# Assume BASE and data.yaml are defined, and data.yaml contains path for 'val' split

print("Calculating scaled Euclidean distance errors for keypoints on the validation set...")

# Load validation ground truth data from data.yaml
data_yaml_path = os.path.join(BASE, "data.yaml")

try:
    # Load the data.yaml content
    import yaml
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)

    # Get the path to the validation image directory
    # Assuming the data.yaml structure includes a 'val' key for the validation image directory
    val_image_dir = data_config.get('val')
    if val_image_dir is None:
         print(f"Error: 'val' directory not specified in {data_yaml_path}. Cannot proceed with validation evaluation.")
         val_ground_truth_data = [] # Empty data to prevent further errors
    else:
        # Construct the full path if 'val' is relative
        if not os.path.isabs(val_image_dir):
             val_image_dir = os.path.join(BASE, val_image_dir)

        # Load validation ground truth data - Assuming a similar structure to internal_test_ground_truth_data
        # This part depends on how your ground truth data for validation is stored.
        # If it's in the same format as internal_test_ground_truth_data but in a different location,
        # you'll need to load it here.
        # For demonstration, let's assume the ground truth data for validation is loaded into
        # a variable called 'internal_val_ground_truth_data' in a similar format.
        # If not, you'll need to adapt this part.

        # !!! IMPORTANT: Replace this placeholder with actual loading of validation ground truth data
        # For example, if your validation annotations are in a file like 'val_annotations.json'
        # internal_val_ground_truth_data = load_validation_annotations('path/to/val_annotations.json')
        # Or if it's part of the same structure as test data, load it accordingly.

        # Assuming internal_val_ground_truth_data is available from previous steps or needs to be loaded
        # If you don't have a separate internal_val_ground_truth_data variable, you might need to adapt
        # the data loading based on your project structure.
        # For this example, let's assume internal_val_ground_truth_data is already loaded or can be loaded.

        # Placeholder: Assuming internal_val_ground_truth_data is defined elsewhere or loaded above.
        # If internal_val_ground_truth_data is not defined, this cell will fail.
        if 'internal_val_ground_truth_data' not in locals():
             print("Error: internal_val_ground_truth_data variable is not defined. Cannot proceed with validation evaluation.")
             val_ground_truth_data = [] # Empty data to prevent further errors
        else:
             val_ground_truth_data = internal_val_ground_truth_data # Use the loaded validation data


except FileNotFoundError:
    print(f"Error: data.yaml not found at {data_yaml_path}. Cannot proceed with validation evaluation.")
    val_ground_truth_data = [] # Empty data to prevent further errors
except Exception as e:
     print(f"Error loading data.yaml or validation ground truth data: {e}. Cannot proceed with validation evaluation.")
     val_ground_truth_data = [] # Empty data to prevent further errors


# Data structure to store scaled Euclidean distance errors for each keypoint on validation set
all_val_keypoint_errors = []

if val_ground_truth_data:
    for item in val_ground_truth_data:
        filename = item['filename']
        true_data_instances = item['data']

        img_path = os.path.join(val_image_dir, filename)
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not read image file at {img_path}. Skipping.")
            continue
        img_h, img_w, _ = img.shape

        try:
            # Perform prediction for the current image
            predikcija = pose_model([img_path], conf=0.4, iou=0.4, verbose=False)
        except Exception as e:
            print(f"Error during prediction for {img_path}: {e}. Skipping.")
            continue

        for instance_idx, true_instance_data in enumerate(true_data_instances):
            true_box_wh_normalized = true_instance_data[3:5]
            true_box_wh_pixel_512 = true_box_wh_normalized * np.array([512, 512])
            kps_normalized = true_instance_data[5:]

            try:
                APEX, VRH, v_VA = getApexVrh(kps_normalized, img_w, img_h)
                DESNI_DONJI, DESNI_GORNJI, _ = getRightSide(kps_normalized, img_w, img_h)
                LIJEVI_GORNJI, LIJEVI_DONJI, _ = getLeftSide(kps_normalized, img_w, img_h)
                true_keypoints_pixel_original_scale = Keypoints(apex = APEX, desni_kost = DESNI_DONJI, desni_vrh = DESNI_GORNJI, vrh = VRH, lijevi_vrh = LIJEVI_GORNJI, lijevi_kost = LIJEVI_DONJI)
            except Exception:
                 continue

            try:
                true_pred_kpts = find_corresponding_image_distance_from_results(predikcija[0], true_keypoints_pixel_original_scale, true_box_wh_pixel_512)
            except Exception:
                true_pred_kpts = None

            if true_pred_kpts is not None:
                try:
                    scaled_distances = PointDistance(true_pred_kpts, scale=True, scaling_type="dist")
                    keypoint_names = ["apex", "desni_kost", "desni_vrh", "vrh", "lijevi_vrh", "lijevi_kost"]
                    for kp_name in keypoint_names:
                         if kp_name in scaled_distances and not pd.isna(scaled_distances[kp_name]):
                            all_val_keypoint_errors.append({
                                'filename': filename,
                                'instance_idx': instance_idx,
                                'keypoint': kp_name,
                                'scaled_euclidean_error': scaled_distances[kp_name]
                            })
                except Exception:
                     pass # Skip this instance if distance calculation fails


# Convert the results to a DataFrame
val_keypoint_errors_df = pd.DataFrame(all_val_keypoint_errors)

if not val_keypoint_errors_df.empty:
    print(f"\nCalculated scaled Euclidean distance errors for {len(val_keypoint_errors_df)} keypoints on the validation set.")
else:
    print("\nNo scaled Euclidean distance errors were calculated for the validation set.")

# val_keypoint_errors_df is now ready for analysis and filtering if needed

### Skalirana euklidska pogreška ključnih točaka po točki (Box plot, Validacijski skup)

In [ ]:
# Generate Box plot of Scaled Euclidean Distance Error by Keypoint (Validation Set)

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Assume val_keypoint_errors_df is available from the previous step

print("Generating box plot of scaled Euclidean distance error by keypoint (validation set)...")

if 'val_keypoint_errors_df' in locals() and not val_keypoint_errors_df.empty:
    plt.figure(figsize=(12, 7))
    sns.boxplot(x='keypoint', y='scaled_euclidean_error', data=val_keypoint_errors_df, palette='viridis')
    plt.title('Distribucija skalirane euklidske pogreške po ključnoj točki (Interni validacijski skup)')
    plt.xlabel('Ključna točka')
    plt.ylabel('Skalirana euklidska pogreška')
    plt.grid(axis='y', alpha=0.75)
    plt.show()
else:
    print("Keypoint error data for validation set is not available or the DataFrame is empty. Cannot generate box plot.")

### Agregirane metrike skalirane euklidske pogreške (Validacijski skup)

**Subtask:** Izračunati i prikazati agregirane metrike (srednja vrijednost, standardna devijacija) skalirane euklidske pogreške po ključnoj točki i ukupno za validacijski skup.

In [ ]:
# Aggregate Scaled Euclidean Distance Error Metrics by Keypoint and Overall (Validation Set)

import pandas as pd
import numpy as np

# Assume val_keypoint_errors_df is available from the previous step

print("Aggregating scaled Euclidean distance error metrics (validation set)...")

if not val_keypoint_errors_df.empty:
    # Aggregate by keypoint
    val_keypoint_error_summary = val_keypoint_errors_df.groupby('keypoint')['scaled_euclidean_error'].agg(['mean', 'std']).reset_index()
    print("\n--- Scaled Euclidean Distance Error Summary by Keypoint (Validation Set) ---")
    display(val_keypoint_error_summary)

    # Calculate overall mean and std dev (mean across all keypoints per instance, then averaged)
    val_pivot_errors = val_keypoint_errors_df.pivot_table(
        index=['filename', 'instance_idx'],
        columns='keypoint',
        values='scaled_euclidean_error'
    )
    val_instance_average_errors = val_pivot_errors.mean(axis=1)
    val_overall_mean_euclidean_error = val_instance_average_errors.mean()
    val_overall_std_euclidean_error = val_instance_average_errors.std()

    print("\n--- Overall Scaled Euclidean Distance Error (Validation Set) ---")
    print(f"Overall Average Scaled Euclidean Distance Error (Mean across all keypoints per instance, then averaged): {val_overall_mean_euclidean_error:.4f}")
    print(f"Standard Deviation of Instance Average Errors: {val_overall_std_euclidean_error:.4f}")

else:
    print("\nCannot aggregate scaled Euclidean distance error metrics for validation set as the DataFrame is empty.")

### Statistički test za ukupnu prosječnu skaliranu euklidsku pogrešku (Validacijski skup)

**Subtask:** Provesti statistički test za ukupnu prosječnu skaliranu euklidsku pogrešku (< 0.03) na validacijskom skupu.

In [ ]:
# Perform Statistical Test for Overall Average Scaled Euclidean Distance Error (Validation Set)

from scipy import stats
import numpy as np

# Assume val_instance_average_errors and val_overall_mean_euclidean_error are available from the previous step

print("\n--- Statistical Test for Overall Average Scaled Euclidean Distance Error (< 0.03) (Validation Set) ---")

if 'val_instance_average_errors' in locals() and 'val_overall_mean_euclidean_error' in locals():
    try:
        threshold = 0.03
        # Drop NaN values from instance_average_errors as t-test cannot handle them
        valid_instance_errors = val_instance_average_errors.dropna()

        if len(valid_instance_errors) > 1:
            # Perform One-Sided One-Sample T-Test
            t_statistic, p_value_two_sided = stats.ttest_1samp(valid_instance_errors, threshold, alternative='less')

            print(f"--- Jednostrani jedno-uzorački T-test za ukupnu prosječnu skaliranu euklidsku pogrešku (< {threshold:.2f}) (Validacijski skup) ---")
            print(f"H0: Prosječna pogreška >= {threshold:.2f}")
            print(f"H1: Prosječna pogreška < {threshold:.2f}")
            print(f"Prag testiranja: {threshold:.2f}")
            print(f"Prosječna pogreška uzorka: {val_overall_mean_euclidean_error:.4f}")
            print(f"T-statistika: {t_statistic:.4f}")
            print(f"P-vrijednost (jednostrana): {p_value_two_sided:.4f}")

            alpha = 0.05 # Significance level
            if p_value_two_sided < alpha:
                print(f"\nRezultat: Odbacujemo nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Ukupna prosječna skalirana euklidska pogreška ({val_overall_mean_euclidean_error:.4f}) statistički je značajno manja od {threshold:.2f}.")
            else:
                print(f"\nRezultat: Ne uspijevamo odbaciti nul-hipotezu (H0) na razini značajnosti {alpha:.2f}.")
                print(f"Zaključak: Nema dovoljno statističkih dokaza da se zaključi da je ukupna prosječna skalirana euklidska pogreška manja od {threshold:.2f}.")

            # Check for normality to see if Wilcoxon test is more appropriate
            try:
                shapiro_w, shapiro_p = stats.shapiro(valid_instance_errors)
                print(f"\nShapiro-Wilk test za normalnost: W={shapiro_w:.4f}, p={shapiro_p:.4f}")
                if shapiro_p < alpha:
                    print("Napomena: Podaci ne izgledaju normalno distribuirani (prema Shapiro-Wilk testu).")
                    print("Preporuka: Neparametarski test poput Wilcoxonovog testa mogao bi biti prikladniji.")
                    print("\n--- Jednostrani jedno-uzorački Wilcoxonov test s predznakom ranga za ukupnu prosječnu skaliranu euklidsku pogrešku (< 0.03) (Validacijski skup) ---")
                    try:
                        wilcoxon_statistic, wilcoxon_p_value = stats.wilcoxon(valid_instance_errors - threshold, alternative='less')
                        print(f"Wilcoxon statistika: {wilcoxon_statistic:.4f}")
                        print(f"P-vrijednost (jednostrana): {wilcoxon_p_value:.4f}")
                        if wilcoxon_p_value < alpha:
                             print(f"\nRezultat (Wilcoxon): Odbacujemo nul-hipotezu na razini značajnosti {alpha:.2f}.")
                             print(f"Zaključak (Wilcoxon): Ukupna prosječna skalirana euklidska pogreška ({val_overall_mean_euclidean_error:.4f}) statistički je značajno manja od {threshold:.2f} (prema Wilcoxonovom testu).")
                        else:
                             print(f"\nRezultat (Wilcoxon): Ne uspijevamo odbaciti nul-hipotezu na razini značajnosti {alpha:.2f}.")
                             print(f"Zaključak (Wilcoxon): Nema dovoljno statističkih dokaza da se zaključi da je ukupna prosječna skalirana euklidska pogreška manja od {threshold:.2f} (prema Wilcoxonovom testu).")
                    except ValueError as ve:
                         print(f"Nije moguće provesti Wilcoxonov test: {ve}. To se može dogoditi ako su sve razlike nula ili je veličina uzorka premala nakon uklanjanja nula.")
                else:
                    print("Napomena: Podaci izgledaju normalno distribuirani (prema Shapiro-Wilk testu). Rezultati T-testa su vjerojatno valjani.")
            except Exception as e:
                print(f"Došlo je do pogreške prilikom izvođenja Shapiro-Wilk ili Wilcoxonovog testa: {e}")
        else:
            print("Nije moguće provesti statistički test jer nema dovoljno podataka (manje od 2 valjane podatkovne točke).")
    except Exception as e:
        print(f"Došlo je do pogreške prilikom izvođenja statističkog testa: {e}")
else:
    print("Podaci o prosječnim pogreškama po instanci za validacijski skup nisu dostupni za statistički test.")

### Vizualizacija predikcija graničnih okvira po kategorijama IoU vrijednosti

In [ ]:
# Generate and display the 3x3 grid for IoU visualization
# The generate_iou_grid function (from cell at2RPdP7nBKq) already handles selection and visualization
print("Generiranje 3x3 grida s vizualizacijama graničnih okvira po kategorijama IoU vrijednosti...")
iou_grid_image = generate_iou_grid(BASE=BASE, pose_model=pose_model)

# Display the generated grid
if iou_grid_image is not None:
    print("\nPrikaz grida s vizualizacijama graničnih okvira:")
    cv2_imshow(iou_grid_image)
else:
    print("\nNije moguće generirati grid s vizualizacijama graničnih okvira. Provjerite ima li dovoljno slika u testnom skupu ili postoje li problemi s predikcijom.")

### Vizualizacija predikcija skeleta po kategorijama OKS vrijednosti

In [ ]:
# Generate and display the 3x3 grid for OKS visualization
# The generate_iou_grid function (from cell zxjr1b6Teeyg) is overloaded for OKS,
# and the show_grid function displays the grid from its output.
print("\nGeneriranje 3x3 grida s vizualizacijama skeleta po kategorijama OKS vrijednosti...")

# generate_iou_grid for OKS returns lower, middle, upper lists
lower_oks, middle_oks, upper_oks = generate_iou_grid(BASE=BASE, pose_model=pose_model)

# Use the show_grid function to make and display the grid
oks_grid_image = show_grid(lower_oks, middle_oks, upper_oks)


# Display the generated grid
if oks_grid_image is not None:
    print("\nPrikaz grida s vizualizacijama skeleta:")
    cv2_imshow(oks_grid_image)
else:
    print("\nNije moguće generirati grid s vizualizacijama skeleta. Provjerite ima li dovoljno slika u testnom skupu ili postoje li problemi s predikcijom.")

### Box plot apsolutnih razlika MGK (%) po stranama (filtrirani podaci)

In [ ]:
# Generate Box plot of Absolute MGK (%) Differences by Side (Filtered Data)

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np # Import numpy for nan handling

# Assume mgk_diff_data_filtered is available from previous calculations (cell 8d92b003)
# This DataFrame contains 'Absolute Difference (Desno)' and 'Absolute Difference (Lijevo)'

print("Generating box plot of absolute MGK (%) differences by side (filtered data)...")

if 'mgk_diff_data_filtered' in locals() and not mgk_diff_data_filtered.empty:
    # Select only the columns for right and left absolute differences
    mgk_side_diffs = mgk_diff_data_filtered[['Absolute Difference (Right)', 'Absolute Difference (Left)']]

    # Melt the DataFrame to long format for seaborn boxplot
    mgk_side_diffs_melted = mgk_side_diffs.melt(var_name='Strana', value_name='Apsolutna razlika MGK (%)')

    plt.figure(figsize=(8, 6))
    sns.boxplot(x='Strana', y='Apsolutna razlika MGK (%)', data=mgk_side_diffs_melted, palette=['skyblue', 'salmon'])
    plt.title('Distribution of absolute MBL differences (%) per side (internal test group)')
    plt.xlabel('Side')
    plt.ylabel('Absolute difference MBL (%)')
    plt.grid(axis='y', alpha=0.75)
    plt.show()

    # Optional: Print summary statistics for each side's absolute difference
    print("\nSažetak statistike za apsolutne razlike MGK (%) po strani (filtrirani podaci):")
    display(mgk_side_diffs.agg(['mean', 'median', 'std']).T)


else:
    print("Filtered MGK difference data is not available or empty. Cannot generate box plot.")